# 01 — Data layer exploration (Step 1: what does each source actually contain?)

Built part by part: each part asks one question of one source, the output is read with the
owner, a note is written, then the next part is added. Runs on Kaggle with internet on.
Versions pinned to `uv.lock` (CLAUDE.md §8).

In [1]:
from pathlib import Path

if Path("/kaggle/working").exists():  # Kaggle only; locally uv.lock provides these
    get_ipython().run_line_magic("pip", "install -q pandas==3.0.5 duckdb==1.5.5")

In [2]:
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

ON_KAGGLE = Path("/kaggle/working").exists()
REPO = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "pyproject.toml").exists())  # kernel may start in notebooks/
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO / "data" / "cache"  # local: reuse the project caches
TM_URL = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb"
TM_DB = WORK / "transfermarkt.duckdb" if ON_KAGGLE else REPO / "data" / "raw" / "transfermarkt.duckdb"
if not TM_DB.exists():  # 211 MB weekly snapshot of dcaribou/transfermarkt-datasets
    subprocess.run(["curl", "-sL", "-o", str(TM_DB), TM_URL], check=True)
con = duckdb.connect(str(TM_DB), read_only=True)
print("snapshot:", con.execute("SELECT * FROM version").fetchall())

snapshot: [('154367dfa6d6eb0b86332e332f9df0a080c7ddce',)]


## Part 1 — Transfermarkt `player_valuations`: how are valuations spaced, are there duplicate dates?

Feeds the `value_at` rule (which valuation counts as "the value at date X").

In [3]:
con.execute("""
SELECT player_id, COUNT(*) AS n, COUNT(DISTINCT date) AS distinct_dates, MIN(date) AS first, MAX(date) AS last
FROM player_valuations GROUP BY 1 ORDER BY n DESC LIMIT 5""").df()

,player_id,n,distinct_dates,first,last
0,39333,57,57,2006-05-21,2025-12-29
1,44162,55,55,2007-09-20,2025-10-21
2,51894,53,53,2007-08-23,2026-05-21
3,35047,53,53,2006-09-24,2026-06-05
4,39153,52,52,2006-10-26,2022-11-08


In [4]:
busiest_player_id = con.execute(
    "SELECT player_id FROM player_valuations GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 1"
).fetchone()[0]
valuations = con.execute(
    f"SELECT * FROM player_valuations WHERE player_id = {busiest_player_id} ORDER BY date"
).df()
valuations["gap_days"] = valuations.date.diff().dt.days
valuations.tail(12)

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,gap_days
45,39333,2023-03-14,475000,Basaksehir FK,6890,TR1,138.0
46,39333,2023-06-08,375000,Basaksehir FK,6890,TR1,86.0
47,39333,2023-10-20,375000,Eyüpspor,7160,TR1,134.0
48,39333,2023-12-29,325000,Eyüpspor,7160,TR1,70.0
49,39333,2024-03-26,300000,Eyüpspor,7160,TR1,88.0
50,39333,2024-06-13,225000,Eyüpspor,7160,TR1,79.0
51,39333,2024-10-04,175000,Eyüpspor,7160,TR1,113.0
52,39333,2024-12-23,175000,Eyüpspor,7160,TR1,80.0
53,39333,2025-03-20,150000,Eyüpspor,7160,TR1,87.0
54,39333,2025-06-18,125000,Eyüpspor,7160,TR1,90.0


In [5]:
con.execute("""
SELECT quantile_cont(gap, [0.1, 0.5, 0.9]) AS gap_days_p10_p50_p90, COUNT(*) AS gaps
FROM (SELECT date_diff('day', LAG(date) OVER (PARTITION BY player_id ORDER BY date), date) AS gap
      FROM player_valuations) WHERE gap IS NOT NULL""").df()

,gap_days_p10_p50_p90,gaps
0,"[81.0, 168.0, 268.0]",614773


**Note (Part 1, read 2026-08-27).** Kaggle image pins pandas 2.2.2; the 3.0.5 pin installs with
conflict warnings but the notebook runs on 3.0.5 (pandas imported after the install), so parity
with `uv.lock` holds. Snapshot `154367d`. No duplicate valuation dates. Each valuation row carries
the club at that date (`current_club_id`) — a second source of club-at-time. Revaluations follow a
March/June/October/December rhythm: gaps p10 = 81, median = 168, p90 = 268 days (614,773 gaps).
A June update exists for most players, so "last valuation on or before 1 July" is the candidate
rule — but p90 = 268 days means it can be nine months stale for some. Decision deferred to
Part 1b: staleness at 1 July, per Big-5 season.

## Part 1b — How stale is "last valuation on or before 1 July" for Big-5 players?

Candidates for `value_at`: (a) last valuation on or before the date; (b) nearest valuation within
±30 days, else NaN; (c) interpolate between the two surrounding valuations.
Criterion: (a) is acceptable if the median staleness at 1 July is under ~45 days and the share
older than 180 days is small; otherwise (b) or (c).

In [6]:
BIG5 = ("GB1", "ES1", "IT1", "L1", "FR1")
big5_sql = ",".join(f"'{c}'" for c in BIG5)
con.execute(f"""
WITH big5_players AS (
  SELECT DISTINCT a.player_id, CAST(g.season AS INTEGER) AS season
  FROM appearances a JOIN games g ON a.game_id = g.game_id
  WHERE g.competition_id IN ({big5_sql}) AND CAST(g.season AS INTEGER) BETWEEN 2014 AND 2025
),
at_july AS (
  SELECT p.player_id, p.season, MAKE_DATE(p.season, 7, 1) AS july1,
         MAX(v.date) FILTER (WHERE v.date <= MAKE_DATE(p.season, 7, 1)) AS last_before,
         MIN(v.date) FILTER (WHERE v.date >  MAKE_DATE(p.season, 7, 1)) AS first_after
  FROM big5_players p LEFT JOIN player_valuations v ON v.player_id = p.player_id
  GROUP BY 1, 2
)
SELECT season, COUNT(*) AS player_seasons,
       SUM(last_before IS NULL) AS no_valuation_before,
       quantile_cont(date_diff('day', last_before, july1), 0.5) AS stale_days_median,
       quantile_cont(date_diff('day', last_before, july1), 0.9) AS stale_days_p90,
       AVG(CAST(date_diff('day', last_before, july1) > 180 AS INTEGER)) AS share_older_than_180d,
       AVG(CAST(date_diff('day', last_before, july1) <= 30 OR date_diff('day', july1, first_after) <= 30 AS INTEGER)) AS share_within_30d
FROM at_july GROUP BY 1 ORDER BY 1""").df()

,season,player_seasons,no_valuation_before,stale_days_median,stale_days_p90,share_older_than_180d,share_within_30d
0,2014,2476,152.0,159.0,238.0,0.105422,0.544244
1,2015,2639,170.0,0.0,25.0,0.017416,0.951367
2,2016,2613,170.0,137.0,179.0,0.026607,0.818256
3,2017,2566,127.0,23.0,30.0,0.019270,0.951020
4,2018,2519,128.0,27.0,39.0,0.017148,0.583437
5,2019,2609,163.0,25.0,28.0,0.011447,0.967585
6,2020,2685,200.0,84.0,84.0,0.035815,0.152427
7,2021,2502,62.0,23.0,35.0,0.011475,0.836885
8,2022,2721,144.0,24.0,32.0,0.007761,0.765776
9,2023,2711,147.0,15.0,22.0,0.014431,0.979337


**Note (Part 1b, read 2026-08-27).** Staleness of "last valuation on or before 1 July" for Big-5
player-seasons: median 15–28 days and p90 22–39 in nine of twelve seasons (the June revaluation
lands before the cut); share older than 180 days 0.8–3.6%. Three seasons are systematically
staler — 2014-15 (median 159 d, 10.5% > 180 d), 2016-17 (median 137 d), 2020-21 (median 84 d,
the COVID revaluation pause). `share_within_30d` swings 15–98% only because the June update date
moves relative to 1 July. 2–8% of player-seasons (62–213) have no valuation before 1 July at all
— debutants — and get `NaN`.

**DECIDED — `value_at` = last valuation on or before the date**, stored with `value_age_days`
so staleness is visible; 2014-15, 2016-17 and 2020-21 flagged as staler in the writeup.
Rejected: nearest-within-±30-days (blanks 15–85% of players in swing seasons for a cadence
artefact); interpolation (invents a number between two market decisions).

## Part 2 — Transfermarkt `transfers`: what do a fee of 0 and a NULL fee mean, and why does one player-season have several rows?

Feeds the fee rule in `market` (spec §4.I: cost = fee, or market value when the fee is
undisclosed or the move is free) and how loans are treated.
Criterion: the rule must be readable off the rows — if 0 and NULL each map to one situation
(free / loan / undisclosed) consistently, encode that; if they are mixed, the fee is unusable
below a floor and market value is the cost.

In [7]:
con.execute("DESCRIBE transfers").df()

,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,transfer_date,DATE,YES,None,None,None
2,transfer_season,VARCHAR,YES,None,None,None
3,from_club_id,INTEGER,YES,None,None,None
4,to_club_id,INTEGER,YES,None,None,None
5,from_club_name,VARCHAR,YES,None,None,None
6,to_club_name,VARCHAR,YES,None,None,None
7,transfer_fee,"DECIMAL(18,3)",YES,None,None,None
8,market_value_in_eur,"DECIMAL(18,3)",YES,None,None,None
9,player_name,VARCHAR,YES,None,None,None


In [8]:
con.execute("""
SELECT CASE WHEN transfer_fee IS NULL THEN 'NULL' WHEN transfer_fee = 0 THEN '0' ELSE '>0' END AS fee_class,
       COUNT(*) AS rows_, COUNT(DISTINCT player_id) AS players,
       SUM(CAST(from_club_name = 'Without Club' AS INTEGER)) AS from_without_club,
       SUM(CAST(to_club_name = 'Without Club' AS INTEGER)) AS to_without_club,
       SUM(CAST(from_club_name = 'Retired' OR to_club_name = 'Retired' AS INTEGER)) AS retired
FROM transfers WHERE TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
GROUP BY 1 ORDER BY 1""").df()

,fee_class,rows_,players,from_without_club,to_without_club,retired
0,0,85521,19032,72.0,15.0,0.0
1,>0,16105,9644,0.0,0.0,0.0
2,NULL,49893,20217,2336.0,2967.0,130.0


In [9]:
con.execute("""SELECT player_name, transfer_season, transfer_date, from_club_name, to_club_name, transfer_fee, market_value_in_eur
FROM transfers WHERE transfer_fee = 0 AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
USING SAMPLE 15 ROWS (reservoir, 1)""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee,market_value_in_eur
0,Amir Syafiz,25/26,2026-06-30,Young Lions,Hougang Utd.,0.0,50000.0
1,Ismaël Koné,25/26,2026-02-01,Sassuolo,Marseille,0.0,14000000.0
2,Kiril Popov,24/25,2024-08-03,Kolos Kovalivka,Chornomorets,0.0,150000.0
3,Sito,24/25,2024-07-04,Asteras Aktor,Intercity,0.0,350000.0
4,Killian Phillips,23/24,2023-08-11,Palace U21,Wycombe,0.0,150000.0
5,Richie Laryea,22/23,2023-06-30,Toronto,Nott'm Forest,0.0,2500000.0
6,Jansen Miller,22/23,2023-05-01,Indiana Hoosier,Long Island RR,0.0,NaN
7,Kanu,22/23,2023-01-01,Botafogo,EC Bahia,0.0,3500000.0
8,Iván López,22/23,2022-12-31,Sacachispas,Ferro,0.0,25000.0
9,Thomas Sabitzer,22/23,2022-08-30,LASK,WSG Tirol,0.0,700000.0


In [10]:
con.execute("""SELECT player_name, transfer_season, transfer_date, from_club_name, to_club_name, transfer_fee, market_value_in_eur
FROM transfers WHERE transfer_fee IS NULL AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
USING SAMPLE 15 ROWS (reservoir, 1)""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee,market_value_in_eur
0,Adrián Cova,24/25,2025-01-07,Without Club,Krumovgrad,NaN,NaN
1,Al Bahlul Bousahmin,23/24,2024-02-01,Asaria,Al-Ahly,NaN,NaN
2,Mamadou Ousmane Diagne,23/24,2024-01-01,Malmö FF U19,Malmö,NaN,NaN
3,Iñigo Arguibide,23/24,2023-07-01,Osasuna U19,Osasuna B,NaN,NaN


In [11]:
con.execute("""
SELECT rows_per_player_season, COUNT(*) AS player_seasons
FROM (SELECT player_id, transfer_season, COUNT(*) AS rows_per_player_season FROM transfers GROUP BY 1, 2)
GROUP BY 1 ORDER BY 1""").df()

,rows_per_player_season,player_seasons
0,1,96124
1,2,24929
2,3,6643
3,4,1769
4,5,336
5,6,50
6,7,22
7,8,2
8,9,2
9,10,1


In [12]:
con.execute("""
WITH multi AS (
  SELECT player_id, transfer_season FROM transfers GROUP BY 1, 2 HAVING COUNT(*) >= 3
  ORDER BY player_id LIMIT 3)
SELECT t.player_name, t.transfer_season, t.transfer_date, t.from_club_name, t.to_club_name, t.transfer_fee
FROM transfers t JOIN multi USING (player_id, transfer_season)
ORDER BY t.player_name, t.transfer_date""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee
0,Luiz Gustavo,05/06,2005-08-31,Ipanema-AL,CRB U20,0.0
1,Luiz Gustavo,05/06,2006-02-06,CRB U20,Ipanema-AL,0.0
2,Luiz Gustavo,05/06,2006-05-31,Ipanema-AL,CRB U20,0.0
3,Luiz Gustavo,06/07,2006-07-04,CRB U20,Coruripe,0.0
4,Luiz Gustavo,06/07,2006-09-04,Coruripe,CRB U20,0.0
5,Luiz Gustavo,06/07,2006-09-05,CRB U20,Miguelense FC,NaN
6,Luiz Gustavo,06/07,2006-11-28,Miguelense FC,Corinthians-AL,0.0
7,Luiz Gustavo,06/07,2007-04-30,Corinthians-AL,Miguelense FC,0.0
8,Luiz Gustavo,06/07,2007-05-07,Miguelense FC,Corinthians-AL,NaN
9,Luiz Gustavo,06/07,2007-05-08,Corinthians-AL,CRB,0.0


In [13]:
# A known paid transfer and a known loan, to read the encoding directly
con.execute("""SELECT player_name, transfer_season, transfer_date, from_club_name, to_club_name, transfer_fee, market_value_in_eur
FROM transfers WHERE player_name IN ('Declan Rice', 'João Félix', 'Kylian Mbappé') ORDER BY player_name, transfer_date""").df()

,player_name,transfer_season,transfer_date,from_club_name,to_club_name,transfer_fee,market_value_in_eur
0,Declan Rice,13/14,2013-07-01,Chelsea Youth,West Ham Yth.,0.0,NaN
1,Declan Rice,15/16,2015-07-01,West Ham Yth.,West Ham U18,NaN,NaN
2,Declan Rice,16/17,2016-07-01,West Ham U18,West Ham U23,NaN,NaN
3,Declan Rice,17/18,2017-07-01,West Ham U23,West Ham,NaN,NaN
4,Declan Rice,23/24,2023-07-15,West Ham,Arsenal,116600000.0,90000000.0
5,João Félix,08/09,2008-07-01,Pestinhas Form.,FC Porto Youth,NaN,NaN
6,João Félix,11/12,2011-07-01,FC Porto Youth,Dragon Force F.,NaN,NaN
7,João Félix,12/13,2012-07-01,Dragon Force F.,DragonForce U15,NaN,NaN
8,João Félix,13/14,2013-07-01,DragonForce U15,FC Porto U15,NaN,NaN
9,João Félix,14/15,2014-07-01,FC Porto U15,Padroense U17,NaN,NaN


**Note (Part 2, read 2026-08-27).** `>0` = disclosed fee (Rice 116.6M, Félix 127.2M, Mbappé 180M).
`0` = loan out, loan return (mirror row dated 30 June) *or* free transfer (Mbappé → Real 2024) —
indistinguishable from the fee alone. `NULL` = youth/reserve promotion, "Without Club", retirement:
not a market transaction. 26% of player-seasons have 2+ rows, overwhelmingly loan + return.
The transfer row carries `market_value_in_eur` at transfer time. Decision deferred to Part 2b:
can loans be separated from free transfers structurally (a later mirror move B→A)?

## Part 2b — Can a 0-fee move be classified as a loan by its mirror move?

Rule under test: a move A→B is a loan if the same player later moves B→A. Criterion: the rule is
usable if fee-0 moves mirror often and fee>0 moves almost never do (a paid transfer that is
"mirrored" would be a re-purchase, which is rare).

In [14]:
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee,
         CASE WHEN transfer_fee IS NULL THEN 'NULL' WHEN transfer_fee = 0 THEN '0' ELSE '>0' END AS fee_class
  FROM transfers WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
),
mirrored AS (
  SELECT m.*, MIN(r.transfer_date) AS return_date
  FROM moves m LEFT JOIN moves r
    ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
   AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
  GROUP BY ALL
)
SELECT fee_class, COUNT(*) AS moves,
       AVG(CAST(return_date IS NOT NULL AS INTEGER)) AS share_with_mirror_within_24m,
       quantile_cont(date_diff('day', transfer_date, return_date), 0.5) AS mirror_gap_days_median
FROM mirrored GROUP BY 1 ORDER BY 1""").df()

,fee_class,moves,share_with_mirror_within_24m,mirror_gap_days_median
0,0,85521,0.367886,181.0
1,>0,16105,0.034586,189.0
2,NULL,49893,0.021406,263.5


In [15]:
# The mirror rows themselves: are returns dated on season ends (30 June), and do they carry fee 0?
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
)
SELECT strftime(r.transfer_date, '%m-%d') AS return_day, COUNT(*) AS returns,
       AVG(CAST(r.transfer_fee = 0 AS INTEGER)) AS share_fee_0
FROM moves m JOIN moves r
  ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
 AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
WHERE m.transfer_fee = 0
GROUP BY 1 ORDER BY 2 DESC LIMIT 8""").df()

,return_day,returns,share_fee_0
0,06-30,14183,1.000000
1,12-31,3921,1.000000
2,07-01,1954,0.319136
3,05-31,1612,1.000000
4,11-30,770,1.000000
5,01-01,694,0.630088
6,01-31,620,0.960331
7,08-01,350,0.948949


In [16]:
# Fee-0 moves with NO mirror: free transfers? Spot-check against contract logic — 1 July dates dominate?
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
),
no_mirror AS (
  SELECT m.* FROM moves m LEFT JOIN moves r
    ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
   AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
  WHERE m.transfer_fee = 0 AND r.player_id IS NULL
)
SELECT player_name, transfer_date, from_club_name, to_club_name, market_value_in_eur
FROM no_mirror WHERE market_value_in_eur >= 5000000 ORDER BY hash(player_id) LIMIT 12  -- deterministic sample after the filter""").df()

,player_name,transfer_date,from_club_name,to_club_name,market_value_in_eur
0,Donny van de Beek,2024-06-30,Frankfurt,Man Utd,5000000.0
1,Donny van de Beek,2022-05-31,Everton,Man Utd,25000000.0
2,Houssem Aouar,2023-07-01,Lyon,Roma,13000000.0
3,Christos Mandas,2026-06-30,Bournemouth,Lazio,8000000.0
4,Henry Onyekuru,2021-06-30,Galatasaray,Monaco,7000000.0
5,Ebenezer Akinsanmiro,2026-06-30,Pisa,Inter,7000000.0
6,Donyell Malen,2026-06-30,Roma,Aston Villa,45000000.0
7,Boubacar Kamara,2022-07-01,Marseille,Aston Villa,25000000.0
8,Hans Nicolussi Caviglia,2026-06-30,Parma,Venezia,6000000.0
9,Hans Nicolussi Caviglia,2026-01-28,Fiorentina,Venezia,7000000.0


**Note (Part 2b, read 2026-08-27).** Mirror rule: 36.8% of fee-0 moves have a B→A mirror within
24 months (median gap 181 d); fee>0 3.5% (paid loans); NULL 2.1%. Returns cluster on window ends —
30 Jun (14,183, 100% fee 0), 31 Dec (3,921), 31 May, 30 Nov, 31 Jan; only 1 Jul / 1 Jan are mixed.
DuckDB's trailing `USING SAMPLE` samples *before* the WHERE (two earlier sample cells came back
short/empty); replaced by `ORDER BY hash(player_id) LIMIT n`. Two cases still to classify: the
return rows themselves, and loan-with-option (0-fee A→B followed by a paid A→B).

## Part 2c — Full structural classification of transfer rows

Candidate rule, fee-independent: `loan_return` if the row mirrors a move within the previous 24
months; else `loan_out` if a B→A mirror follows within 24 months; else `loan_then_bought` if a paid
A→B follows within 24 months (option exercised); else `free` when fee = 0; `paid` when fee > 0;
`internal` when fee is NULL. Criterion: the classes must be exhaustive, the free class must look
like genuine free transfers on inspection, and `loan_then_bought` must be non-trivial.

In [17]:
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
),
flags AS (
  SELECT m.*,
    EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id
            AND r.to_club_id = m.from_club_id AND r.transfer_date < m.transfer_date
            AND r.transfer_date >= m.transfer_date - INTERVAL 24 MONTH) AS is_return,
    EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id
            AND r.to_club_id = m.from_club_id AND r.transfer_date > m.transfer_date
            AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH) AS has_mirror_after,
    EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.from_club_id
            AND r.to_club_id = m.to_club_id AND r.transfer_fee > 0 AND r.transfer_date > m.transfer_date
            AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH) AS bought_after
  FROM moves m
),
classed AS (
  SELECT *, CASE
    WHEN is_return THEN 'loan_return'
    WHEN has_mirror_after THEN 'loan_out'
    WHEN transfer_fee = 0 AND bought_after THEN 'loan_then_bought'
    WHEN transfer_fee = 0 THEN 'free'
    WHEN transfer_fee > 0 THEN 'paid'
    ELSE 'internal' END AS kind
  FROM flags
)
SELECT kind, COUNT(*) AS rows_, COUNT(DISTINCT player_id) AS players,
       quantile_cont(market_value_in_eur, 0.5) AS value_median
FROM classed GROUP BY 1 ORDER BY 2 DESC""").df()

,kind,rows_,players,value_median
0,internal,47119,19951,250000.0
1,loan_return,33087,13365,500000.0
2,free,31024,14052,300000.0
3,loan_out,26859,13365,500000.0
4,paid,13430,8806,1500000.0


In [18]:
# Inspect the 'free' class for players worth >= 5m: do they read as contract expiries?
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
),
free AS (
  SELECT m.* FROM moves m
  WHERE m.transfer_fee = 0
    AND NOT EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id
                    AND r.to_club_id = m.from_club_id AND abs(date_diff('day', r.transfer_date, m.transfer_date)) <= 730)
    AND NOT EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.from_club_id
                    AND r.to_club_id = m.to_club_id AND r.transfer_fee > 0 AND r.transfer_date > m.transfer_date
                    AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH)
)
SELECT player_name, transfer_date, from_club_name, to_club_name, market_value_in_eur
FROM free WHERE market_value_in_eur >= 5000000 ORDER BY hash(player_id) LIMIT 15""").df()

,player_name,transfer_date,from_club_name,to_club_name,market_value_in_eur
0,Houssem Aouar,2023-07-01,Lyon,Roma,13000000.0
1,Boubacar Kamara,2022-07-01,Marseille,Aston Villa,25000000.0
2,Alexander Nübel,2020-07-01,FC Schalke 04,Bayern Munich,9500000.0
3,Alexander Nübel,2023-07-25,Bayern Munich,Stuttgart,8000000.0
4,Alexander Nübel,2026-06-30,Stuttgart,Bayern Munich,12000000.0
5,Kevin Diks,2025-07-01,Copenhagen,Mönchengladbach,5000000.0
6,Nikola Maksimović,2021-08-31,SSC Napoli,Genoa,10000000.0
7,Christopher Lenz,2021-07-01,Union Berlin,Frankfurt,5000000.0
8,Tanguy Nianzou,2020-07-01,PSG,Bayern Munich,11000000.0
9,João Mário,2021-07-13,Inter,Benfica,12000000.0


In [19]:
# And the 'loan_then_bought' class: does it look like exercised options?
con.execute("""
WITH moves AS (
  SELECT player_id, player_name, transfer_date, from_club_id, to_club_id, from_club_name, to_club_name,
         transfer_fee, market_value_in_eur FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25
)
SELECT m.player_name, m.transfer_date AS loan_date, r.transfer_date AS purchase_date, m.from_club_name, m.to_club_name,
       r.transfer_fee AS purchase_fee, m.market_value_in_eur
FROM moves m JOIN moves r ON r.player_id = m.player_id AND r.from_club_id = m.from_club_id AND r.to_club_id = m.to_club_id
  AND r.transfer_fee > 0 AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH
WHERE m.transfer_fee = 0 ORDER BY hash(m.player_id) LIMIT 10""").df()

,player_name,loan_date,purchase_date,from_club_name,to_club_name,purchase_fee,market_value_in_eur
0,Leandro Lozano,2022-02-02,2023-01-02,Boston River,Nacional,598000.0,400000.0
1,Iván Angulo,2022-07-25,2024-01-31,Palmeiras,Orlando,1400000.0,800000.0
2,Louis Patris,2024-09-04,2025-07-01,RSC Anderlecht,Sint-Truiden,1750000.0,2000000.0
3,Cenk Özkacar,2022-08-23,2023-07-01,Lyon,Valencia,5000000.0,2000000.0
4,Daniel Luna,2023-01-31,2023-07-01,Deportivo Cali,RCD Mallorca B,1000000.0,700000.0
5,Pep Biel,2024-08-14,2026-01-26,Olympiacos,Charlotte,3500000.0,3000000.0
6,Dylan Levitt,2021-08-20,2022-07-07,Man Utd U23,Dundee United,355000.0,650000.0
7,George Edmundson,2024-08-30,2025-01-25,Ipswich,Middlesbrough,710000.0,350000.0
8,Ali Sowe,2021-02-16,2021-07-01,CSKA-Sofia,Rostov,3000000.0,2000000.0
9,Sebastiano Luperto,2021-08-13,2023-07-01,Napoli,FC Empoli,2500000.0,2200000.0


**Note (Part 2c, read 2026-08-27).** Fee-independent mirror rule, 2014-15 → 2025-26:
`internal` 47,119 · `loan_return` 33,087 · `free` 31,024 · `loan_out` 26,859 · `paid` 13,430.
`loan_then_bought` is empty — Transfermarkt records an exercised option as loan → return → paid,
so the mirror rule covers it; class dropped. The `free` class reads as contract expiries in 13/15
(Aouar, Kamara, Donnarumma…); the two misses are one three-year loan (Nübel, Bayern → Stuttgart
2023-25-26) whose return falls outside the 24-month window. Open: the mirror window, and whether
`internal` hides club-to-club moves with an undisclosed (NULL) fee.

## Part 2d — Mirror window and undisclosed fees

Candidates for the window: 24, 36, 48 months. Criterion: choose the smallest window that captures
nearly all loan lengths (share of mirrors beyond it < ~2%) while the paid-side mirror share (false
loans) stays near its 24-month level. Undisclosed fees: if NULL-fee rows between two senior clubs
are material, they get their own class `undisclosed` and are priced at market value (spec §4.I).

In [20]:
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
),
gaps AS (
  SELECT m.transfer_fee, MIN(date_diff('day', m.transfer_date, r.transfer_date)) AS gap_days
  FROM moves m JOIN moves r ON r.player_id = m.player_id AND r.from_club_id = m.to_club_id
    AND r.to_club_id = m.from_club_id AND r.transfer_date > m.transfer_date
    AND r.transfer_date <= m.transfer_date + INTERVAL 48 MONTH
  GROUP BY m.player_id, m.transfer_date, m.from_club_id, m.to_club_id, m.transfer_fee
)
SELECT CASE WHEN transfer_fee = 0 THEN '0' WHEN transfer_fee > 0 THEN '>0' ELSE 'NULL' END AS fee_class,
       COUNT(*) AS mirrored_moves_48m,
       AVG(CAST(gap_days > 730 AS INTEGER)) AS share_beyond_24m,
       AVG(CAST(gap_days > 1095 AS INTEGER)) AS share_beyond_36m,
       quantile_cont(gap_days, [0.5, 0.9, 0.99]) AS gap_p50_p90_p99
FROM gaps GROUP BY 1 ORDER BY 1""").df()

,fee_class,mirrored_moves_48m,share_beyond_24m,share_beyond_36m,gap_p50_p90_p99
0,0,31798,0.010976,0.002201,"[183.0, 364.0, 734.0]"
1,>0,737,0.251018,0.088195,"[365.0, 1063.8, 1399.4799999999996]"
2,NULL,1191,0.130982,0.041142,"[305.0, 740.0, 1283.4999999999995]"


In [21]:
# Paid moves that mirror: how does the false-loan share grow with the window?
con.execute("""
WITH moves AS (
  SELECT player_id, transfer_date, from_club_id, to_club_id, transfer_fee FROM transfers
  WHERE from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
)
SELECT COUNT(*) AS paid_moves,
       AVG(CAST(EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
                        AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 24 MONTH) AS INTEGER)) AS mirror_24m,
       AVG(CAST(EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
                        AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 36 MONTH) AS INTEGER)) AS mirror_36m,
       AVG(CAST(EXISTS (SELECT 1 FROM moves r WHERE r.player_id = m.player_id AND r.from_club_id = m.to_club_id AND r.to_club_id = m.from_club_id
                        AND r.transfer_date > m.transfer_date AND r.transfer_date <= m.transfer_date + INTERVAL 48 MONTH) AS INTEGER)) AS mirror_48m
FROM moves m WHERE m.transfer_fee > 0""").df()

,paid_moves,mirror_24m,mirror_36m,mirror_48m
0,16105,0.034586,0.042099,0.045762


In [22]:
# NULL-fee rows between two senior clubs (neither side youth/reserve/without club): undisclosed fees?
con.execute("""
WITH senior AS (
  SELECT * FROM transfers
  WHERE transfer_fee IS NULL AND from_club_id IS NOT NULL AND to_club_id IS NOT NULL
    AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 14 AND 25
    AND NOT regexp_matches(from_club_name, '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
    AND NOT regexp_matches(to_club_name,   '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
)
SELECT COUNT(*) AS null_fee_senior_rows, COUNT(DISTINCT player_id) AS players,
       quantile_cont(market_value_in_eur, [0.5, 0.9]) AS value_p50_p90,
       AVG(CAST(market_value_in_eur >= 1000000 AS INTEGER)) AS share_value_ge_1m
FROM senior""").df()

,null_fee_senior_rows,players,value_p50_p90,share_value_ge_1m
0,10887,7142,"[300000.0, 1000000.0]",0.117404


In [23]:
con.execute("""
SELECT player_name, transfer_date, from_club_name, to_club_name, market_value_in_eur FROM transfers
WHERE transfer_fee IS NULL AND from_club_id IS NOT NULL AND to_club_id IS NOT NULL
  AND TRY_CAST(SUBSTR(transfer_season, 1, 2) AS INTEGER) BETWEEN 20 AND 25 AND market_value_in_eur >= 3000000
  AND NOT regexp_matches(from_club_name, '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
  AND NOT regexp_matches(to_club_name,   '(U1[5-9]|U2[0-3]|Youth|Yth|Jgd|Juvenil|Academy|\\bB$|\\bII$|Without Club|Retired|Unknown)')
ORDER BY hash(player_id) LIMIT 12""").df()

,player_name,transfer_date,from_club_name,to_club_name,market_value_in_eur
0,Franco Petroli,2026-01-15,Godoy Cruz,Lanús,3000000.0
1,Jesús Angulo,2021-07-01,Santos Laguna,Atlas,3500000.0
2,Luis Abram,2023-02-02,Granada CF,Atlanta,4000000.0
3,Fabio Miretti,2022-07-01,Juve Next Gen,Juventus,5000000.0
4,Álvaro Medrán,2024-01-26,Al-Taawoun,Al-Ettifaq,4500000.0
5,Lucas Rodríguez,2024-07-03,Club Tijuana,Querétaro FC,3000000.0
6,Antonio Blanco,2022-07-01,RM Castilla,Real Madrid,5000000.0
7,Junya Ito,2020-07-01,Kashiwa Reysol,Genk,4500000.0
8,Francesco Camarda,2025-06-01,Milan Futuro,AC Milan,10000000.0
9,Dro Fernández,2025-10-01,Barça Atlètic,Barcelona,5000000.0


**Note (Part 2d, read 2026-08-27).** Loan lengths (31,798 mirrored fee-0 moves): median 183 d,
p90 364, p99 734; 1.1% beyond 24 months, 0.22% beyond 36. Paid-side mirrors 3.5% → 4.2% at 36 m,
median gap 365 d — paid loans, correctly loans. NULL fees between "senior" clubs: 10,887 rows,
median value €300k, 11.7% ≥ €1m; sample = reserve-team promotions the name pattern missed (Juve
Next Gen, Castilla, Milan Futuro, Barça Atlètic) and undisclosed non-European moves.

**DECIDED — transfer classification**, fee-independent, **36-month mirror window**:
`loan_return` (mirrors an earlier move) → `loan_out` (a mirror follows) → `paid` (fee > 0) →
`free` (fee = 0) → `undisclosed` (fee NULL, both clubs in `clubs`) → `internal` (fee NULL
otherwise). Cost (spec §4.I): fee for `paid`; market value at transfer for `free` and
`undisclosed`; loans and internal moves are neither purchases nor sales. Rejected: 24-month window
(misses 1.1% of loans incl. multi-year); 48 months (0.2% more loans, more re-purchases swept in);
fee-based loan detection (paid loans exist); name-regex reserve detection (leaky).

## Part 3 — `players.contract_expiration_date` and `game_lineups`

3a: is contract end current-only, and how many players lack it? (feeds whether contract remaining
can be a feature at all — spec §4.D already excludes it from the backtested market model).
3b: which lineup rows count as "in the squad", what positions are recorded, and do lineups agree
with appearances on who played? (feeds the appearances ∪ lineups union in the identity table).

In [24]:
con.execute("""
SELECT last_season, COUNT(*) AS players, SUM(CAST(contract_expiration_date IS NOT NULL AS INTEGER)) AS with_contract_end,
       MIN(contract_expiration_date) AS earliest_end, MAX(contract_expiration_date) AS latest_end
FROM players GROUP BY 1 ORDER BY 1 DESC LIMIT 8""").df()

,last_season,players,with_contract_end,earliest_end,latest_end
0,2025,22292,17221.0,2026-01-10,2035-06-30
1,2024,6075,3908.0,2026-01-31,2031-06-30
2,2023,2092,1455.0,2000-05-31,2029-06-30
3,2022,1877,1186.0,2023-05-31,2028-06-30
4,2021,1817,1357.0,2023-03-15,2027-07-31
5,2020,1843,1182.0,2023-03-31,2027-12-31
6,2019,1566,931.0,2023-01-31,2027-06-30
7,2018,2065,1085.0,2023-01-01,2026-12-31


In [25]:
con.execute("SELECT type, COUNT(*) AS rows_ FROM game_lineups GROUP BY 1").df()

,type,rows_
0,substitutes,1398257
1,starting_lineup,1780759


In [26]:
con.execute("SELECT position, COUNT(*) AS rows_ FROM game_lineups GROUP BY 1 ORDER BY 2 DESC").df()

,position,rows_
0,Centre-Back,551970
1,Centre-Forward,394661
2,Goalkeeper,357844
3,Central Midfield,329361
4,Defensive Midfield,279364
5,Right-Back,230225
6,Left-Back,219387
7,Left Winger,205392
8,Attacking Midfield,203736
9,Right Winger,203171


In [27]:
# Do lineups and appearances agree on who played? All PL 2023-24 games.
con.execute("""
WITH g AS (SELECT game_id FROM games WHERE competition_id = 'GB1' AND season = '2023'),
a AS (SELECT DISTINCT CAST(game_id AS VARCHAR) AS game_id, player_id FROM appearances
      WHERE CAST(game_id AS VARCHAR) IN (SELECT game_id FROM g)),
l AS (SELECT DISTINCT CAST(game_id AS VARCHAR) AS game_id, player_id, type FROM game_lineups
      WHERE CAST(game_id AS VARCHAR) IN (SELECT game_id FROM g))
SELECT
  (SELECT COUNT(*) FROM a) AS appearance_rows,
  (SELECT COUNT(*) FROM l WHERE type = 'starting_lineup') AS starting_rows,
  (SELECT COUNT(*) FROM l WHERE type <> 'starting_lineup') AS bench_rows,
  (SELECT COUNT(*) FROM a LEFT JOIN l USING (game_id, player_id) WHERE l.player_id IS NULL) AS played_but_not_in_lineups,
  (SELECT COUNT(*) FROM l LEFT JOIN a USING (game_id, player_id) WHERE l.type = 'starting_lineup' AND a.player_id IS NULL) AS started_but_no_appearance""").df()

,appearance_rows,starting_rows,bench_rows,played_but_not_in_lineups,started_but_no_appearance
0,11384,8360,6820,0,0


In [28]:
# Minutes for bench players who came on vs. who did not: is 'substitutes' usable as 'in squad' only?
con.execute("""
WITH g AS (SELECT game_id FROM games WHERE competition_id = 'GB1' AND season = '2023')
SELECT l.type, COUNT(*) AS rows_, SUM(CAST(a.player_id IS NOT NULL AS INTEGER)) AS with_appearance,
       quantile_cont(a.minutes_played, 0.5) AS minutes_median
FROM game_lineups l
LEFT JOIN appearances a ON CAST(a.game_id AS VARCHAR) = CAST(l.game_id AS VARCHAR) AND a.player_id = l.player_id
WHERE CAST(l.game_id AS VARCHAR) IN (SELECT game_id FROM g) GROUP BY 1""").df()

,type,rows_,with_appearance,minutes_median
0,starting_lineup,8360,8360.0,90.0
1,substitutes,6820,3024.0,16.0


**Note (Part 3, read 2026-08-27).** 3a: `contract_expiration_date` is current-only — present for
77% of active players (17,221 / 22,292 with `last_season` 2025), stale or absent for anyone whose
last season is older (a 2023 player shows 2000-05-31). Usable only as a descriptive field for
current candidates; never a backtest feature (spec §4.D stands).
3b: lineups and appearances agree exactly on PL 2023-24 — 11,384 appearance rows = 8,360 starters
+ 3,024 used substitutes; 0 played-without-lineup, 0 started-without-appearance; 3,796 unused
substitutes exist only in lineups. `type` ∈ {starting_lineup, substitutes}. `position` is
Transfermarkt's detailed per-match role (Centre-Back … Second Striker; 0.4% generic leftovers).

**DECIDED** — identity's Transfermarkt side = appearances ∪ lineups (both types): adds bench-only
squad members, which is right for matching; minutes always from Understat. Transfermarkt lineup
`position` is a candidate role vocabulary for Step 5a alongside Understat's codes.

## Part 4 — Understat: one match, then the vocabularies

One match's player rows and shots printed whole; then, over the first 30 matches of PL 2023-24,
the result / situation / body-part / position vocabularies and how own goals are recorded;
then the season-level player table and the team-match table (style-vector candidates).

In [29]:
from pathlib import Path

if Path("/kaggle/working").exists():  # Kaggle only; locally uv.lock provides these
    get_ipython().run_line_magic("pip", "install -q soccerdata==1.9.1")

In [30]:
import soccerdata as sd

understat = sd.Understat(leagues=["ENG-Premier League"], seasons=["2023-2024"], data_dir=WORK / "understat")
schedule = understat.read_schedule().reset_index()
print(len(schedule), "matches;", list(schedule.columns))
schedule.head(3)

[08/28/26 11:24:08] INFO     No custom team name replacements found. You can configure these in       ]8;id=604984;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=604985;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/mihailandreev/soccerdata/config/teamname_replacements.json.                    

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=604991;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=604992;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_config.py#189\189]8;;\
                             /Users/mihailandreev/soccerdata/config/league_dict.json.                              

                    INFO     Saving cached data to                                                   ]8;id=604999;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=605000;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\
                             /Users/mihailandreev/football-player-scouting/data/cache/understat                    

[2026-08-28 11:24:08] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: /Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib


                    INFO     Successfully loaded TLS library:                                      ]8;id=605007;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py\libraries.py]8;;\:]8;id=605008;file:///Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/tls_requests/models/libraries.py#397\397]8;;\
                             /Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/si                 
                             te-packages/tls_requests/bin/tls-client-darwin-arm64-1.13.1.dylib                     

380

matches;

['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'is_result', 'has_data', 'url']

,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,home_goals,away_goals,home_xg,away_xg,is_result,has_data,url
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,1,2023,22275,2023-08-11 19:00:00,92,88,Burnley,Manchester City,MCI,BUR,0,3,0.311032,2.40074,True,True,https://understat.com/match/22275
1,ENG-Premier League,2324,2023-08-12 Arsenal-Nottingham Forest,1,2023,22276,2023-08-12 11:30:00,83,249,Arsenal,Nottingham Forest,NOT,ARS,2,1,0.84262,0.966305,True,True,https://understat.com/match/22276
2,ENG-Premier League,2324,2023-08-12 Bournemouth-West Ham,1,2023,22277,2023-08-12 14:00:00,73,81,Bournemouth,West Ham,WHU,BOU,1,1,1.51025,1.4834,True,True,https://understat.com/match/22277


In [31]:
match_id = schedule.game_id.iloc[0]
player_rows = understat.read_player_match_stats(match_id=[match_id]).reset_index()
print(list(player_rows.columns))
player_rows

['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'team_id', 'player_id', 'position', 'position_id', 'minutes', 'goals', 'own_goals', 'shots', 'xg', 'xg_chain', 'xg_buildup', 'assists', 'xa', 'key_passes', 'yellow_cards', 'red_cards']

,league,season,game,team,player,league_id,season_id,game_id,team_id,player_id,position,position_id,minutes,goals,own_goals,shots,xg,xg_chain,xg_buildup,assists,xa,key_passes,yellow_cards,red_cards
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Ameen Al Dakhil,1,2023,22275,92,11699,DC,3,90,0,0,0,0.0,0.122039,0.122039,0,0.0,0,0,0
1,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Anass Zaroury,1,2023,22275,92,11703,Sub,17,24,0,0,1,0.063984,0.063984,0.0,0,0.013008,1,0,1
2,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Benson Manuel,1,2023,22275,92,11702,Sub,17,11,0,0,0,0.0,0.063984,0.063984,0,0.0,0,0,0
3,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Connor Roberts,1,2023,22275,92,5568,DR,2,90,0,0,0,0.0,0.122039,0.122039,0,0.0,0,0,0
4,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Dara O'Shea,1,2023,22275,92,8756,DC,3,90,0,0,0,0.0,0.122039,0.122039,0,0.0,0,0,0
5,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Jacob Bruun Larsen,1,2023,22275,92,5355,Sub,17,24,0,0,1,0.013008,0.063984,0.063984,0,0.0,0,0,0
6,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,James Trafford,1,2023,22275,92,9077,GK,1,90,0,0,0,0.0,0.063984,0.063984,0,0.0,0,0,0
7,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Josh Brownhill,1,2023,22275,92,8323,Sub,17,1,0,0,0,0.0,0.0,0.0,0,0.0,0,0,0
8,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Josh Cullen,1,2023,22275,92,1018,MC,9,90,0,0,0,0.0,0.063984,0.063984,0,0.0,0,0,0
9,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Louis Beyer,1,2023,22275,92,6557,DC,3,79,0,0,0,0.0,0.0,0.0,0,0.0,0,0,0


In [32]:
match_shots = understat.read_shot_events(match_id=[match_id]).reset_index()
print(list(match_shots.columns))
match_shots

['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']

,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,team_id,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Anass Zaroury,1,2023,22275,2023-08-11 19:00:00,531947,92,11703,603612,Lyle Foster,0.063984,0.817,0.536,79,Left Foot,Open Play,Blocked Shot
1,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Jacob Bruun Larsen,1,2023,22275,2023-08-11 19:00:00,531948,92,5355,603615,Anass Zaroury,0.013008,0.958,0.376,81,<NA>,From Corner,Missed Shot
2,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Luca Koleosho,1,2023,22275,2023-08-11 19:00:00,531932,92,10620,603608,Vitinho,0.058055,0.86,0.635,14,Right Foot,Open Play,Missed Shot
3,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Lyle Foster,1,2023,22275,2023-08-11 19:00:00,531934,92,7498,<NA>,<NA>,0.017112,0.87,0.751,28,Left Foot,Open Play,Missed Shot
4,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Lyle Foster,1,2023,22275,2023-08-11 19:00:00,531936,92,7498,603609,Luca Koleosho,0.097616,0.86,0.455,36,Right Foot,Open Play,Blocked Shot
5,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Burnley,Zeki Amdouni,1,2023,22275,2023-08-11 19:00:00,531933,92,11701,603609,Luca Koleosho,0.061258,0.886,0.672,17,Left Foot,Open Play,Saved Shot
6,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Aymeric Laporte,1,2023,22275,2023-08-11 19:00:00,531951,88,2498,603627,Julián Álvarez,0.112282,0.913,0.527,94,<NA>,Set Piece,Saved Shot
7,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Erling Haaland,1,2023,22275,2023-08-11 19:00:00,531929,88,8260,603625,Rodri,0.505846,0.937,0.506,3,Left Foot,From Corner,Goal
8,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Erling Haaland,1,2023,22275,2023-08-11 19:00:00,531931,88,8260,603624,Kevin De Bruyne,0.130247,0.952,0.421,10,Right Foot,Open Play,Missed Shot
9,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,Manchester City,Erling Haaland,1,2023,22275,2023-08-11 19:00:00,531935,88,8260,603627,Julián Álvarez,0.113214,0.891,0.385,35,Left Foot,Open Play,Goal


In [33]:
sample_ids = schedule.game_id.head(30).tolist()
try:
    sample_shots = understat.read_shot_events(match_id=sample_ids).reset_index()
    sample_players = understat.read_player_match_stats(match_id=sample_ids).reset_index()
    print("shot result:", sample_shots.result.value_counts().to_dict())
    print("shot situation:", sample_shots.situation.value_counts().to_dict())
    print("shot body_part:", sample_shots.body_part.value_counts(dropna=False).to_dict())
    print("player-match positions:", sample_players.position.value_counts().to_dict())
    print("own_goals column total:", int(sample_players.own_goals.sum()), "| rows with minutes == 0:", int((sample_players.minutes == 0).sum()))
    display(sample_shots[sample_shots.result == "Own Goal"].head(5))
    display(sample_players[sample_players.own_goals > 0].head(5))
except Exception as exc:  # diagnostics never raise (CLAUDE.md §8)
    print("vocabulary pull failed:", type(exc).__name__, exc)

shot result:

{'Missed Shot': 305, 'Blocked Shot': 250, 'Saved Shot': 209, 'Goal': 88, 'Shot On Post': 20, 'Own Goal': 1}

shot situation:

{'Open Play': 637, 'From Corner': 145, 'Set Piece': 59, 'Direct Freekick': 20}

shot body_part:

...

1

| rows with minutes == 0:

0

,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,team_id,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
418,ENG-Premier League,2324,2023-08-19 Tottenham-Manchester United,Manchester United,Lisandro Martínez,1,2023,22290,2023-08-19 16:30:00,533350,89,10802,<NA>,<NA>,0.0,0.047,0.487,82,Right Foot,Open Play,Own Goal


,league,season,game,team,player,league_id,season_id,game_id,team_id,player_id,position,position_id,minutes,goals,own_goals,shots,xg,xg_chain,xg_buildup,assists,xa,key_passes,yellow_cards,red_cards
435,ENG-Premier League,2324,2023-08-19 Tottenham-Manchester United,Manchester United,Lisandro Martínez,1,2023,22290,89,10802,DC,3,90,0,1,0,0.0,0.176897,0.156797,0,0.0201,1,0,0


In [34]:
season_players = understat.read_player_season_stats().reset_index()
print(len(season_players), "players;", list(season_players.columns))
print("season-level position strings:", season_players.position.value_counts().head(15).to_dict())
season_players.head(5)

570

players;

['league', 'season', 'team', 'player', 'league_id', 'season_id', 'team_id', 'player_id', 'position', 'matches', 'minutes', 'goals', 'xg', 'np_goals', 'np_xg', 'assists', 'xa', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'xg_chain', 'xg_buildup']

season-level position strings:

{'D S': 124, 'M S': 117, 'S': 82, 'F M S': 65, 'F S': 47, 'D M S': 36, 'D': 33, 'GK': 30, 'M': 11, 'GK S': 8, 'D F M S': 8, 'F M': 4, 'F': 2, 'D M': 2, 'D F S': 1}

,league,season,team,player,league_id,season_id,team_id,player_id,position,matches,minutes,goals,xg,np_goals,np_xg,assists,xa,shots,key_passes,yellow_cards,red_cards,xg_chain,xg_buildup
0,ENG-Premier League,2324,Arsenal,Aaron Ramsdale,1,2023,83,5603,GK,6,540,0,0.0,0,0.0,0,0.0,0,0,0,0,0.783274,0.783274
1,ENG-Premier League,2324,Arsenal,Ben White,1,2023,83,7298,D S,37,3024,4,2.015914,4,2.015914,4,4.758832,13,38,8,0,20.149292,17.29045
2,ENG-Premier League,2324,Arsenal,Bukayo Saka,1,2023,83,7322,F,35,2990,16,16.807056,10,12.240043,8,11.325986,107,89,4,0,30.357597,13.22862
3,ENG-Premier League,2324,Arsenal,Cédric Soares,1,2023,83,847,S,3,44,0,0.0,0,0.0,0,0.0,0,0,0,0,0.196755,0.196755
4,ENG-Premier League,2324,Arsenal,David Raya,1,2023,83,9676,GK,32,2880,0,0.0,0,0.0,0,0.0,0,0,2,0,7.517592,7.517592


In [35]:
team_matches = understat.read_team_match_stats().reset_index()
print(len(team_matches), "team-matches;", list(team_matches.columns))
team_matches.head(4)

380

team-matches;

['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'away_points', 'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg', 'away_np_xg_difference', 'away_ppda', 'away_deep_completions', 'home_points', 'home_expected_points', 'home_goals', 'home_xg', 'home_np_xg', 'home_np_xg_difference', 'home_ppda', 'home_deep_completions']

,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,away_points,away_expected_points,away_goals,away_xg,away_np_xg,away_np_xg_difference,away_ppda,away_deep_completions,home_points,home_expected_points,home_goals,home_xg,home_np_xg,home_np_xg_difference,home_ppda,home_deep_completions
0,ENG-Premier League,2324,2023-08-11 Burnley-Manchester City,1,2023,22275,2023-08-11 19:00:00,92,88,Burnley,Manchester City,MCI,BUR,3,2.7761,3,2.40074,2.40074,2.089708,15.8,9,0,0.1385,0,0.311032,0.311032,-2.089708,33.071429,4
1,ENG-Premier League,2324,2023-08-12 Arsenal-Nottingham Forest,1,2023,22276,2023-08-12 11:30:00,83,249,Arsenal,Nottingham Forest,NOT,ARS,0,1.4883,1,0.966305,0.966305,0.123685,44.1,4,3,1.1754,2,0.84262,0.84262,-0.123685,4.0,9
2,ENG-Premier League,2324,2023-08-12 Bournemouth-West Ham,1,2023,22277,2023-08-12 14:00:00,73,81,Bournemouth,West Ham,WHU,BOU,1,1.2985,1,1.4834,1.4834,-0.02685,9.733333,8,1,1.3846,1,1.51025,1.51025,0.02685,5.111111,9
3,ENG-Premier League,2324,2023-08-12 Brighton-Luton,1,2023,22278,2023-08-12 14:00:00,220,256,Brighton,Luton,LUT,BRI,0,0.1878,1,1.88594,1.12477,-2.48154,23.0625,5,3,2.7246,4,4.36748,3.60631,2.48154,6.875,17


**Note (Part 4, read 2026-08-27).** Understat per-match player rows: `minutes, goals, own_goals,
shots, xg, xg_chain, xg_buildup, assists, xa, key_passes, cards`, Understat `player_id`/`team_id`,
and `position` — a real role code only for starters (16 codes incl. `FWL/FWR`, `DML/DMR`); every
substitute is `Sub` (27% of rows, 242/902 over 30 matches). Shots: `xg, location_x/y` (0–1),
`minute, body_part, situation, result`; `assist_player_id` is not in the player-id space (use the
name). Vocabularies over 873 shots: result {Missed, Blocked, Saved, Goal, Shot On Post, Own Goal};
situation {Open Play, From Corner, Set Piece, Direct Freekick} — **no "Penalty"** although
`xg ≠ np_xg` in Brighton–Luton proves penalties are present; body_part NA in 17.5% (headers?).
Own goals: shot row `result = Own Goal`, `xg = 0`, at the scorer's own end, plus `own_goals = 1`
on the player row. Season table: 570 players, coarse position strings (`D S`, `F M S`),
`np_goals/np_xg/xg_chain/xg_buildup`. Team-match: wide, both sides — `points, expected_points,
goals, xg, np_xg, np_xg_difference, ppda, deep_completions`; no possession.

**DECIDED:** `Own Goal` shot rows are excluded from every shot-based metric (shots, xG faced,
keeper proxy). Sub-minute role assignment is Step 5a's decision, with 27% as the share it must
handle. Deferred to Part 4b: how penalties are labelled; what the NA body part is.

## Part 4b — Penalties and the NA body part

Understat gives every penalty xG = 0.7612. Criterion: if every shot with xg in [0.75, 0.78] shares
one `situation` label, that label is how penalties arrive and `np_xg` can be reproduced from it;
if the NA body-part shots sit close to goal and never in the Direct Freekick situation, NA = header.

In [36]:
penalty_like = sample_shots[sample_shots.xg.between(0.75, 0.78)]
print(len(penalty_like), "shots with xg in [0.75, 0.78]")
print("their situation:", penalty_like.situation.value_counts().to_dict())
print("their body_part:", penalty_like.body_part.value_counts(dropna=False).to_dict())
print("their result:", penalty_like.result.value_counts().to_dict())
penalty_like[["game", "player", "xg", "location_x", "location_y", "minute", "body_part", "situation", "result"]]

12

shots with xg in [0.75, 0.78]

their situation:

{}

their body_part:

{'Left Foot': 7, 'Right Foot': 5}

their result:

{'Goal': 9, 'Saved Shot': 2, 'Shot On Post': 1}

,game,player,xg,location_x,location_y,minute,body_part,situation,result
87,2023-08-12 Brighton-Luton,João Pedro,0.761169,0.885,0.5,70,Right Foot,<NA>,Goal
103,2023-08-12 Brighton-Luton,Carlton Morris,0.761169,0.885,0.5,80,Right Foot,<NA>,Goal
204,2023-08-13 Brentford-Tottenham,Bryan Mbeumo,0.761169,0.885,0.5,26,Left Foot,<NA>,Goal
316,2023-08-19 Fulham-Brentford,Bryan Mbeumo,0.761169,0.885,0.5,65,Left Foot,<NA>,Goal
370,2023-08-19 Liverpool-Bournemouth,Mohamed Salah,0.761169,0.885,0.5,35,Left Foot,<NA>,Saved Shot
474,2023-08-20 Aston Villa-Everton,Douglas Luiz,0.761169,0.885,0.5,23,Right Foot,<NA>,Goal
501,2023-08-20 West Ham-Chelsea,Enzo Fernández,0.761169,0.885,0.5,42,Right Foot,<NA>,Saved Shot
516,2023-08-20 West Ham-Chelsea,Lucas Paquetá,0.761169,0.885,0.5,94,Left Foot,<NA>,Goal
534,2023-08-21 Crystal Palace-Arsenal,Martin Odegaard,0.761132,0.885,0.5,53,Left Foot,<NA>,Goal
584,2023-08-26 Arsenal-Fulham,Bukayo Saka,0.761169,0.885,0.5,69,Left Foot,<NA>,Goal


In [37]:
na_body = sample_shots[sample_shots.body_part.isna()]
print("NA body-part shots:", len(na_body))
print("situation:", na_body.situation.value_counts().to_dict())
print("location_x quantiles (1 = goal line):", na_body.location_x.quantile([0.1, 0.5, 0.9]).round(3).tolist(),
      "| all shots:", sample_shots.location_x.quantile([0.1, 0.5, 0.9]).round(3).tolist())
print("share that are goals:", round(float((na_body.result == 'Goal').mean()), 3), "| all shots:", round(float((sample_shots.result == 'Goal').mean()), 3))

NA body-part shots:

153

situation:

{'From Corner': 71, 'Open Play': 49, 'Set Piece': 33}

location_x quantiles (1 = goal line):

...

[0.756, 0.874, 0.953]

share that are goals:

0.085

| all shots:

0.101

**Note (Part 4b, read 2026-08-27).** Penalties: soccerdata drops Understat's `Penalty` label, so
they arrive as `situation = NA` with the fixed signature `xg = 0.7612`, `location (0.885, 0.5)` —
12 in 30 matches, exactly the 12 NAs missing from the situation counts (861 of 873). NA body part
(153 shots): never Direct Freekick, 68% from corners/set pieces, closer to goal (median
`location_x` 0.92 vs 0.874), converting 8.5% vs 10.1% — headers and "other body part", both
dropped by the same mapping.

**DECIDED:** `is_penalty` = (`situation` NA ∧ `xg` ∈ [0.75, 0.78] ∧ location ≈ (0.885, 0.5));
`np_xg` = `xg` − penalty xG, verified against Understat's own `np_xg` in the port; NA body part
labelled `head_or_other` (not asserted to be headers).

## Part 5 — Sofascore: one league-season statistics page, without and with a `fields` list

Needs the browser-TLS client (`wrapper-tls-requests`, pinned to `uv.lock`); whether it works on
Kaggle is itself a parity finding. Questions: what one page returns by default; which candidate
fields the endpoint accepts; what `None` means; the page cap; whether keepers share the list.
Then the tournament ids for all 13 leagues, verified through the seasons endpoint.

In [38]:
from pathlib import Path

if Path("/kaggle/working").exists():  # Kaggle only; locally uv.lock provides these
    get_ipython().run_line_magic("pip", "install -q wrapper-tls-requests==1.2.5 requests==2.34.2")

In [39]:
import json
import time

import requests
import tls_requests

UA = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
    "Referer": "https://www.sofascore.com/",
    "Origin": "https://www.sofascore.com",
}
SOFASCORE = "https://api.sofascore.com/api/v1"


def sofascore_get(path, params=None):
    """One polite request through the browser-TLS client (plain clients get 403)."""
    time.sleep(3)
    url = f"{SOFASCORE}{path}" + ("?" + requests.compat.urlencode(params) if params else "")
    response = tls_requests.get(url, headers=UA, timeout=30)
    return response.status_code, response.text


status, body = sofascore_get("/unique-tournament/17/seasons")
print(status, body[:200])
pl_seasons = json.loads(body)["seasons"] if status == 200 else []
pd.DataFrame(pl_seasons).head(12)

200

{"seasons":[{"name":"Premier League 26\/27","year":"26\/27","editor":false,"id":96668},{"name":"Premier League 25\/26","year":"25\/26","editor":false,"id":76986},{"name":"Premier League 24\/25","year"

,name,year,editor,id,seasonCoverageInfo
0,Premier League 26/27,26/27,False,96668,NaN
1,Premier League 25/26,25/26,False,76986,NaN
2,Premier League 24/25,24/25,False,61627,NaN
3,Premier League 23/24,23/24,False,52186,NaN
4,Premier League 22/23,22/23,False,41886,{}
5,Premier League 21/22,21/22,False,37036,NaN
6,Premier League 20/21,20/21,False,29415,NaN
7,Premier League 19/20,19/20,False,23776,NaN
8,Premier League 18/19,18/19,False,17359,NaN
9,Premier League 17/18,17/18,False,13380,NaN


In [40]:
season_2324 = next(s["id"] for s in pl_seasons if s["year"] == "23/24")
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 0, "accumulation": "total"})
print(status)
default_page = json.loads(body)
print("top-level keys:", list(default_page.keys()))
print("page / pages:", default_page.get("page"), default_page.get("pages"), "| results on page:", len(default_page.get("results", [])))
print("keys of one result:", list(default_page["results"][0].keys()))
pd.json_normalize(default_page["results"]).head(3).T

200

top-level keys:

['results', 'page', 'pages']

page / pages:

1

...

| results on page:

100

keys of one result:

['player', 'team']

,0,1,2
player.name,Rodri,Arijanet Murić,Kevin De Bruyne
player.slug,rodri,arijanet-muric,kevin-de-bruyne
player.userCount,185039,2219,337117
player.gender,M,M,M
player.id,827606,888971,70996
player.fieldTranslations.nameTranslation.ar,رودري,أريجانيت موريش,كيفن دي بروين
player.fieldTranslations.nameTranslation.bn,রডরি,আরিজানেট মুরিচ,কেভিন ডি ব্রুইন
player.fieldTranslations.nameTranslation.hi,रोड्री,अरिजानेट मुरिक,केविन डी ब्रूने
player.fieldTranslations.nameTranslation.ru,Родри,Ариянет Мурич,Кевин Де Брёйне
player.fieldTranslations.shortNameTranslation.ar,رودري,أ. موريش,ك. دي بروين


In [41]:
CANDIDATE_FIELDS = ["minutesPlayed","appearances","rating","goals","assists","expectedGoals","expectedAssists","keyPasses",
    "bigChancesCreated","accuratePasses","accuratePassesPercentage","accurateFinalThirdPasses","accurateLongBalls",
    "accurateCrosses","successfulDribbles","tackles","interceptions","clearances","ballRecovery","possessionWonAttThird",
    "possessionLost","groundDuelsWon","groundDuelsWonPercentage","aerialDuelsWon","aerialDuelsWonPercentage","dribbledPast",
    "fouls","wasFouled","errorLeadToShot","errorLeadToGoal","touches","saves","goalsConceded","savedShotsFromInsideTheBox",
    "savedShotsFromOutsideTheBox","highClaims","punches","penaltyFaced","penaltySave","cleanSheet"]
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 0, "accumulation": "total", "fields": ",".join(CANDIDATE_FIELDS)})
print(status, body[:300] if status != 200 else "")
fields_page = json.loads(body) if status == 200 else {"results": []}
fields_rows = pd.json_normalize(fields_page["results"])
accepted = [f for f in CANDIDATE_FIELDS if f in fields_rows.columns]
print("accepted:", len(accepted), "of", len(CANDIDATE_FIELDS), "| rejected:", [f for f in CANDIDATE_FIELDS if f not in fields_rows.columns])
print("page / pages:", fields_page.get("page"), fields_page.get("pages"))
print("None counts on this page:", fields_rows[accepted].isna().sum()[lambda s: s > 0].to_dict())
fields_rows.head(3).T

200

accepted:

40

of

40

...

1

6

None counts on this page:

{'expectedGoals': 3}

,0,1,2
minutesPlayed,2938,900,1235
appearances,34,10,18
rating,8.01,8.0,7.93
goals,8,0,4
assists,9,0,10
...,...,...,...
team.fieldTranslations.nameTranslation.ru,Манчестер Сити,Бернли,Манчестер Сити
team.fieldTranslations.shortNameTranslation.ar,مانشستر سيتي,NaN,مانشستر سيتي
team.fieldTranslations.shortNameTranslation.bn,ম্যান সিটি,NaN,ম্যান সিটি
team.fieldTranslations.shortNameTranslation.hi,मैन सिटी,NaN,मैन सिटी


In [42]:
position_col = "player.position"
print("positions on page:", fields_rows[position_col].value_counts().to_dict() if position_col in fields_rows else fields_rows.columns.tolist()[:12])
keepers = fields_rows[fields_rows[position_col] == "G"] if position_col in fields_rows else fields_rows.iloc[0:0]
print("keepers on page:", len(keepers))
keepers[["player.name", "minutesPlayed", "tackles", "saves", "goalsConceded"]].head(3) if len(keepers) else None

positions on page:

['minutesPlayed', 'appearances', 'rating', 'goals', 'assists', 'expectedGoals', 'expectedAssists', 'keyPasses', 'bigChancesCreated', 'accuratePasses', 'accuratePassesPercentage', 'accurateFinalThirdPasses']

keepers on page:

0

In [43]:
# Page depth: how many players does the whole league-season list hold, and does 'pages' cover them?
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 100 * (fields_page.get("pages", 1) - 1), "accumulation": "total", "fields": "minutesPlayed"})
last_page = json.loads(body)
print(status, "last page:", last_page.get("page"), "of", last_page.get("pages"), "| results:", len(last_page.get("results", [])))
print("→ players listed for PL 23/24 ≈", 100 * (last_page.get("pages", 1) - 1) + len(last_page.get("results", [])))

200

last page:

6

of

6

| results:

70

→ players listed for PL 23/24 ≈

570

In [44]:
# Tournament ids for all 13 leagues: verify each by its recent seasons (a wrong id shows another competition or 404).
TOURNAMENT_GUESSES = {"GB1": 17, "ES1": 8, "IT1": 23, "L1": 35, "FR1": 34,
                      "BE1": 38, "NL1": 37, "PO1": 238, "TR1": 52, "C1": 215, "BRA1": 325, "A1": 45, "DK1": 39}
for comp, tournament_id in TOURNAMENT_GUESSES.items():
    status, body = sofascore_get(f"/unique-tournament/{tournament_id}/seasons")
    years = [s["year"] for s in json.loads(body)["seasons"]][:4] if status == 200 else body[:80]
    print(comp, tournament_id, status, years)

GB1

17

200

['26/27', '25/26', '24/25', '23/24']

ES1

...

['26/27', '25/26', '24/25', '23/24']

DK1

39

200

['26/27', '25/26', '24/25', '23/24']

In [45]:
# Names behind the ids, to confirm the guesses are the right competitions.
for comp, tournament_id in TOURNAMENT_GUESSES.items():
    status, body = sofascore_get(f"/unique-tournament/{tournament_id}")
    name = json.loads(body).get("uniqueTournament", {}).get("name") if status == 200 else body[:80]
    print(comp, tournament_id, status, name)

GB1

17

200

Premier League

ES1

...

Austrian Bundesliga

DK1

39

200

Danish Superliga

**Note (Part 5, run locally 2026-08-27).** Sofascore returns 403 from Kaggle even through the
browser-TLS client — an IP-range block — so all Sofascore work runs locally. From here: the seasons
endpoint lists PL back to 15/16 (ids e.g. 23/24 = 52186); Brazil is calendar-year. All 13
tournament ids verified by name. The default statistics page carries only `player` and `team`
objects — no stats and **no position**; with the field list all 40 candidates are accepted
(page 1 of 6, 100 per page, 570 players for PL 23/24 = Understat's 570 exactly). The only `None`s
on page 1: 3 × `expectedGoals`.

## Part 5b — Keepers via the position filter, what  means, season depth per league

Criterion: keepers must be reachable () with their keeper fields populated;
 must have one meaning (stat not tracked for that player) so it can stay ; each
league's earliest season fixes the pull range.

In [46]:
status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                             params={"limit": 100, "offset": 0, "accumulation": "total", "fields": ",".join(CANDIDATE_FIELDS),
                                     "filters": "position.in.G"})
print(status, body[:200] if status != 200 else "")
keeper_page = json.loads(body) if status == 200 else {"results": []}
keeper_rows = pd.json_normalize(keeper_page["results"])
print("keepers:", len(keeper_rows), "| pages:", keeper_page.get("pages"))
print("None counts among keepers:", keeper_rows[accepted].isna().sum()[lambda s: s > 0].to_dict())
keeper_rows[["player.name", "team.name", "minutesPlayed", "saves", "goalsConceded", "highClaims", "tackles", "expectedGoals"]].head(6)

200

keepers:

40

| pages:

1

None counts among keepers:

{'expectedGoals': 16, 'expectedAssists': 1}

,player.name,team.name,minutesPlayed,saves,goalsConceded,highClaims,tackles,expectedGoals
0,Arijanet Murić,Burnley,900,64,16,19,1,NaN
1,Thomas Strakosha,Brentford,135,6,2,2,0,0.0
2,Alphonse Aréola,West Ham United,2699,138,53,20,1,0.0
3,André Onana,Manchester United,3420,149,58,37,2,0.0
4,Jordan Pickford,Everton,3420,121,51,26,1,0.0
5,Martin Dúbravka,Newcastle United,1994,88,42,18,0,NaN


In [47]:
for position in ["D", "M", "F"]:
    status, body = sofascore_get(f"/unique-tournament/17/season/{season_2324}/statistics",
                                 params={"limit": 100, "offset": 0, "accumulation": "total", "fields": "minutesPlayed", "filters": f"position.in.{position}"})
    page = json.loads(body) if status == 200 else {}
    print(position, status, "pages:", page.get("pages"), "| first page results:", len(page.get("results", [])))

D

200

pages:

2

| first page results:

...

200

pages:

2

| first page results:

100

In [48]:
none_xg = fields_rows[fields_rows.expectedGoals.isna()]
none_xg[["player.name", "team.name", "minutesPlayed", "appearances", "goals", "tackles", "rating"]]

,player.name,team.name,minutesPlayed,appearances,goals,tackles,rating
1,Arijanet Murić,Burnley,900,10,0,1,8.00
56,Martin Dúbravka,Newcastle United,1994,23,0,0,7.20
60,Mark Travers,Bournemouth,360,4,0,0,7.18


In [49]:
earliest = {}
for comp, tournament_id in TOURNAMENT_GUESSES.items():
    status, body = sofascore_get(f"/unique-tournament/{tournament_id}/seasons")
    years = [s["year"] for s in json.loads(body)["seasons"]] if status == 200 else []
    earliest[comp] = (years[-1] if years else None, len(years))
earliest

{'GB1': ('92/93', 35),
 'ES1': ('1969/1970', 58),
 'IT1': ('1965/1966', 62),
 'L1': ('70/71', 57),
 'FR1': ('70/71', 57),
 'BE1': ('80/81', 47),
 'NL1': ('88/89', 39),
 'PO1': ('02/03', 21),
 'TR1': ('80/81', 47),
 'C1': ('08/09', 19),
 'BRA1': ('2001', 25),
 'A1': ('08/09', 19),
 'DK1': ('08/09', 19)}

**Note (Part 5b, run locally 2026-08-27).** Keepers: 40 for PL 23/24 on one page via
`filters=position.in.G`, keeper fields populated (saves, goals conceded, high claims); position
pages D 2 / M 3 / F 2 / G 1, so pulling per position group yields position at ~8 pages per
league-season. `None` = stat not computed for that player (16 keepers lack `expectedGoals`; Onana
has 0.0 because he took a shot) — stays `NaN`; moot for modelling since xG is Understat's.
Season lists reach back decades (PL 92/93, PT 02/03, CH/AT/DK 08/09); listing ≠ stats, Phase 0
fixed full stats at 15/16, and every list covers 2015-16 → 2025-26.

**DECIDED:** request all 40 accepted fields (which become features is Step 5d); pull per position
filter (G, D, M, F); `None` → `NaN`; seasons 2015-16 → 2025-26 (Brazil 2016 → 2026); Sofascore
pulls run locally only. Tournament ids → `config.py`.

## Part 6 — FotMob: one outfield player's and one keeper's full season-stats JSON; league-level endpoint probe

Plain HTTP (no TLS client). Questions: does Kaggle reach it; the full structure of a season's
stats JSON (sections, per-90, percentiles), for an outfield player and a keeper; whether a
league-season endpoint exists that would replace per-player pulls for Belgium/Denmark.

In [50]:
import re

import requests

PLAIN_UA = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
}


def fotmob_get(url, params=None):
    time.sleep(3)
    response = requests.get(url, params=params, headers=PLAIN_UA, timeout=30)
    return response.status_code, response.text


def fotmob_search(name):
    status, body = fotmob_get("https://apigw.fotmob.com/searchapi/suggest", params={"term": name, "lang": "en"})
    ids = re.findall(r'"id"\s*:\s*"?(\d{4,8})', body)
    return status, (int(ids[0]) if ids else None), body[:200]


def fotmob_page_data(player_id):
    status, body = fotmob_get(f"https://www.fotmob.com/players/{player_id}/x")
    match = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', body, re.S)
    return status, (json.loads(match.group(1))["props"]["pageProps"]["data"] if match else None)


fotmob_ids = {}
for name in ["Hans Vanaken", "Simon Mignolet"]:
    status, player_id, head = fotmob_search(name)
    fotmob_ids[name] = player_id
    print(name, status, player_id, "|", head[:120])

Hans Vanaken

200

203666

|

{
  "took": 1,
  "total": 1,
  "squadMemberSuggest": [
    {
      "text": "Hans Vanaken",
      "offset": 0,
      "len

Simon Mignolet

200

37868

|

{
  "took": 2,
  "total": 1,
  "squadMemberSuggest": [
    {
      "text": "Simon Mignolet",
      "offset": 0,
      "l

In [51]:
fotmob_pages = {}
for name, player_id in fotmob_ids.items():
    if player_id is None:
        continue
    status, data = fotmob_page_data(player_id)
    fotmob_pages[name] = data
    print(name, status, "| data keys:", list(data.keys()) if data else None)
    if data:
        recent = [s for s in data["statSeasons"] if s["seasonName"] in ("2023/2024", "2024/2025")]
        for s in recent:
            print("  ", s["seasonName"], [(t["name"], t["entryId"], t.get("hasDeepStats")) for t in s["tournaments"]])

Hans Vanaken

200

| data keys:

['id', 'name', 'birthDate', 'contractEnd', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr']

2024/2025

...

['id', 'name', 'birthDate', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr']

2024/2025

[('Belgian Pro League', '1-0', True), ('Champions League', '1-1', True)]

2023/2024

[('Belgian Pro League', '2-0', True), ('Super Cup', '2-1', True), ('Conference League', '2-2', True)]

In [52]:
def fotmob_season_json(player_id, entry_id):
    status, body = fotmob_get("https://www.fotmob.com/api/data/playerStats", params={"playerId": player_id, "seasonId": entry_id})
    return status, (json.loads(body) if status == 200 else body[:200])


def flatten_stats(stats_json):
    rows = []

    def walk(obj, section):
        if isinstance(obj, dict):
            if "title" in obj and "statValue" in obj:
                rows.append({"section": section, **{k: obj.get(k) for k in ["title", "statValue", "per90", "percentileRank", "percentileRankPer90", "statFormat"]}})
            for key, value in obj.items():
                walk(value, obj.get("title", section) if key == "items" else section)
        elif isinstance(obj, list):
            for value in obj:
                walk(value, section)

    walk(stats_json, "")
    return pd.DataFrame(rows)


season_stats = {}
for name, data in fotmob_pages.items():
    season = next((s for s in data["statSeasons"] if s["seasonName"] == "2023/2024"), None)
    if season is None:
        print(name, "no 2023/2024 season"); continue
    domestic = season["tournaments"][0]
    status, stats_json = fotmob_season_json(fotmob_ids[name], domestic["entryId"])
    season_stats[name] = stats_json
    print(name, domestic["name"], status, "| top-level keys:", list(stats_json.keys()) if isinstance(stats_json, dict) else stats_json)

Hans Vanaken

Belgian Pro League

200

| top-level keys:

['sectionOrder', 'shotmap', 'statsSection', 'topStatCard', 'keeperShotmap']

Simon Mignolet

Belgian Pro League

200

| top-level keys:

['sectionOrder', 'keeperShotmap', 'statsSection', 'topStatCard', 'shotmap']

In [53]:
for name, stats_json in season_stats.items():
    if not isinstance(stats_json, dict):
        continue
    flat = flatten_stats(stats_json)
    print(name, "—", len(flat), "stats; sections:", flat.section.value_counts().to_dict())
    display(flat)

Hans Vanaken

—

46

stats; sections:

{'Possession': 11, 'Defending': 11, 'Passing': 10, 'Shooting': 6, '': 6, 'Discipline': 2}

...

—

24

stats; sections:

{'Goalkeeping': 11, '': 6, 'Distribution': 5, 'Discipline': 2}

,section,title,statValue,per90,percentileRank,percentileRankPer90,statFormat
0,Goalkeeping,Saves,71,2.282143,57.142857,14.285714,number
1,Goalkeeping,Save percentage,71.0,71.000000,62.857143,62.857143,percent
2,Goalkeeping,Goals conceded,29,0.932143,34.285714,88.571429,number
3,Goalkeeping,Goals prevented,-0.23,-0.007303,40.000000,45.714286,fraction
4,Goalkeeping,Clean sheets,11,0.353571,82.857143,82.857143,number
5,Goalkeeping,Penalty saves,0,0.000000,0.000000,0.000000,number
6,Goalkeeping,Penalties conceded,1,0.032143,48.571429,62.857143,number
7,Goalkeeping,Penalty save %,0.0,0.000000,25.714286,25.714286,percent
8,Goalkeeping,Error led to goal,0,0.000000,100.000000,100.000000,number
9,Goalkeeping,Acted as sweeper,16,0.514286,88.571429,77.142857,number


In [54]:
# Raw shape of one section, to see what else sits beside title/statValue (e.g. per90 basis, percentile group)
outfield_json = season_stats.get("Hans Vanaken")
print(json.dumps(outfield_json, ensure_ascii=False)[:2500] if isinstance(outfield_json, dict) else outfield_json)

{"sectionOrder": ["top-stat-card", "shotmap", "stats-section"], "shotmap": [{"id": 2571877883, "playerName": "Hans Vanaken", "eventType": "AttemptSaved", "shotType": "LeftFoot", "situation": "RegularPlay", "teamId": 8342, "playerId": 203666, "x": 89.1, "y": 23.4387261191, "min": 15, "period": "FirstHalf", "isOwnGoal": false, "isBlocked": true, "isOnTarget": true, "isSavedOffLine": false, "isFromInsideBox": true, "blockedX": 93.7, "blockedY": 26.4009375, "goalCrossedY": 35.22, "goalCrossedZ": 1.219999994, "expectedGoals": 0.0435284972190857, "onGoalShot": {"x": 0.677248677248676, "y": 0.322751321164021, "zoomRatio": 1}, "box": "InsideBox", "homeTeamId": 8342, "awayTeamId": 8203, "homeTeamName": "Club Brugge", "awayTeamName": "KV Mechelen", "homeScore": 1, "awayScore": 1, "matchId": 4206189, "matchDate": "2023-07-30T16:30:00Z", "teamColor": "#0572FF", "teamColorDark": "#006ad5"}, {"id": 2571879487, "playerName": "Hans Vanaken", "eventType": "AttemptSaved", "shotType": "Header", "situatio

In [55]:
# League-level endpoint probe: does FotMob serve a whole league-season of player stats?
for url, params in [
    ("https://www.fotmob.com/api/leagueseasondeepstats", {"id": 40, "season": "2023/2024", "type": "players", "stat": "tackles"}),
    ("https://www.fotmob.com/api/data/leagueseasondeepstats", {"id": 40, "season": "2023/2024", "type": "players", "stat": "tackles"}),
    ("https://www.fotmob.com/api/data/leagues", {"id": 40, "tab": "stats"}),
    ("https://www.fotmob.com/api/data/leagues", {"id": 40}),
]:
    status, body = fotmob_get(url, params=params)
    print(status, url, params, "|", body[:200].replace("\n", " "))

404

https://www.fotmob.com/api/leagueseasondeepstats

{'id': 40, 'season': '2023/2024', 'type': 'players', 'stat': 'tackles'}

|

<!DOCTYPE html><html lang="en" dir="ltr"><head><meta name="apple-itunes-app" content="app-id=488575683"/><link rel="alternate" href="android-app://com.mobilefootie.wc2010/http"/><link rel="apple-touch

...

200

https://www.fotmob.com/api/data/leagues

{'id': 40}

|

{"tabs":["overview","table","fixtures","stats","transfers","seasons"],"allAvailableSeasons":["2026/2027","2025/2026","2024/2025","2023/2024","2022/2023","2021/2022","2020/2021","2019/2020","2018/2019"

**Note (Part 6, read 2026-08-27).** FotMob is reachable from Kaggle (plain HTTP, 200s). Search →
id (Vanaken 203666, Mignolet 37868); the player page's `statSeasons` lists tournaments per season
with `entryId` and `hasDeepStats`; `/api/data/playerStats?playerId&seasonId=<entryId>` returns
`topStatCard`, `statsSection` and a **`shotmap`** — every shot with xG, xGOT, on-target, body
part, situation, coordinates, match id — i.e. shot-level data for feeder leagues Understat does
not cover. Outfield: 46 stats in Shooting / Passing / Possession / Defending / Discipline, each
with `statValue`, `per90`, `percentileRank` (incl. "Goals conceded while on pitch", "xG against
while on pitch"). Keeper: 24 stats — saves, save %, goals conceded, **goals prevented**, clean
sheets, penalties faced/saved, errors led to goal, sweeper actions, high claims, distribution.
Page data also carries `marketValues`, `contractEnd`, `injuryInformation`, `careerHistory`.
`/api/data/leagueseasondeepstats` exists and wants a season *id* (it lists them) — Part 6b.

**DECIDED:** FotMob keeper "goals prevented" is the second keeper metric alongside the Understat
proxy (Phase 0 Q4). Open until 6b: per-player vs per-league pull for Belgium/Denmark.

## Part 6b — The FotMob league-season endpoint: does it list every player per stat?

`/api/data/leagueseasondeepstats` wants a season *id* (listed in its own response) and a stat
name. Criterion: if one request returns all players of a league-season for one stat (not a
top-N), Belgium/Denmark are pulled per stat (~50 stats × 9 seasons × 2 leagues) instead of per
player; the `leagues?tab=stats` response should name the available stats.

In [56]:
status, body = fotmob_get("https://www.fotmob.com/api/data/leagues", params={"id": 40, "tab": "stats"})
league_json = json.loads(body)
print(status, "top-level keys:", list(league_json.keys()))
stats_block = league_json.get("stats", {})
print("stats keys:", list(stats_block.keys()) if isinstance(stats_block, dict) else type(stats_block))
print(json.dumps(stats_block, ensure_ascii=False)[:2500])

200

top-level keys:

['tabs', 'allAvailableSeasons', 'details', 'seostr', 'QAData', 'table', 'transfers', 'overview', 'stats', 'fixtures', 'playoff', 'seasons']

stats keys:

['players', 'teams', 'seasonStatLinks', 'seasonsWithLinks']

{"players": [{"header": "Top scorer", "participant": {"id": 304860, "name": "Michael Frey", "rank": 1, "ccode": "SUI", "teamId": 9988, "teamName": "Royal Antwerp", "value": 4, "stat": {"name": "goals", "value": 4, "format": "number", "fractions": 0}}, "fetchAllUrl": "https://data.fotmob.com/stats/40/season/37803/goals.json", "topThree": [{"id": 304860, "name": "Michael Frey", "rank": 1, "ccode": "SUI", "teamId": 9988, "teamName": "Royal Antwerp", "value": 4, "stat": {"name": "goals", "value": 4, "format": "number", "fractions": 0}, "teamColors": {"darkMode": "#DB2823", "lightMode": "#DB2823", "fontDarkMode": "rgba(255, 255, 255, 1.0)", "fontLightMode": "rgba(255, 255, 255, 1.0)"}}, {"id": 1353364, "name": "Anthony Valencia", "rank": 2, "ccode": "ECU", "teamId": 9988, "teamName": "Royal Antwerp", "value": 3, "stat": {"name": "goals", "value": 3, "format": "number", "fractions": 0}, "teamColors": {"darkMode": "#DB2823", "lightMode": "#DB2823", "fontDarkMode": "rgba(255, 255, 255, 1.0)", 

In [57]:
status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats",
                          params={"id": 40, "season": "2023/2024", "type": "players", "stat": "tackles"})
season_list = json.loads(body).get("seasons", [])
season_ids = {s["name"]: s["id"] for s in season_list}
print("seasons offered:", season_ids)
belgium_2324 = season_ids.get("2023/2024")
for stat in ["tackles", "interceptions", "goals", "rating", "expected_goals", "xg"]:
    status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats",
                              params={"id": 40, "season": belgium_2324, "type": "players", "stat": stat})
    deep = json.loads(body) if status == 200 else {}
    rows = deep.get("statsData", [])
    print(stat, status, "| players returned:", len(rows), "| first row:", json.dumps(rows[0], ensure_ascii=False)[:300] if rows else None)

seasons offered:

{'2026/2027': 37803, '2025/2026': 27152, '2024/2025': 23650, '2023/2024': 20957, '2022/2023': 17862, '2021/2022': 16419, '2020/2021': 15312, '2019/2020': 14160, '2018/2019': 12768}

tackles

200

| players returned:

...

200

| players returned:

0

| first row:

None

In [58]:
# What stat names does the endpoint know? They usually sit in the response's own metadata.
status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats",
                          params={"id": 40, "season": belgium_2324, "type": "players", "stat": "tackles"})
deep = json.loads(body)
print("response keys:", list(deep.keys()))
for key in deep:
    if key not in ("statsData", "seasons"):
        print(key, "→", json.dumps(deep[key], ensure_ascii=False)[:1500])

response keys:

['statsData', 'seasons', 'statsList', 'leagueDetails', 'type', 'teamName', 'currentSeasonId', 'currentStatName']

statsList

→

[{"name": "goals", "localizedTitleId": "goals_title", "title": "Top scorer", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "goal_assist", "localizedTitleId": "goal_assist_title", "title": "Assists", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "_goals_and_goal_assist", "localizedTitleId": "goals_and_assists", "title": "Goals + Assists", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "rating", "localizedTitleId": "rating_title", "title": "FotMob rating", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "mins_played", "localizedTitleId": "minutes_played", "title": "Minutes played", "category": "Top Stat", "localizedCategoryId": "top_stats"}, {"name": "goals_per_90", "localizedTitleId": "goals_per_90_title", "title": "Goals per 90", "category": "Attacking", "localizedCategoryId": "attack"}, {"name": "expected_goals", "localizedTitleId": "expected_goals", "title": "Expected goals (xG)", "category":

...

→

"20957"

currentStatName

→

"tackles"

## Part 6c — FotMob league-season stats: full stat list, the static JSON, completeness, position codes

Criterion: if the static per-stat JSON lists every player of the league-season (`mins_played`
count ≈ the league's player count) with a stable row shape, Belgium/Denmark are pulled per stat
(statsList × seasons); the numeric `position` code needs a mapping read off the data.

In [59]:
stats_list = pd.DataFrame(deep["statsList"])
print(len(stats_list), "stats;", "categories:", stats_list.category.value_counts().to_dict())
print(stats_list[["name", "title", "category"]].to_string())

37

stats;

categories:

{'Attacking': 16, 'Defending': 8, 'Top Stat': 5, 'Goalkeeping': 5, 'Discipline': 3}

                                           name                            title     category
0                                         goals                       Top scorer     Top Stat
1                                   goal_assist                          Assists     Top Stat
2                        _goals_and_goal_assist                  Goals + Assists     Top Stat
3                                        rating                    FotMob rating     Top Stat
4                                   mins_played                   Minutes played     Top Stat
5                                  goals_per_90                     Goals per 90    Attacking
6                                expected_goals              Expected goals (xG)    Attacking
7                         expected_goals_per_90       Expected goals (xG) per 90    Attacking
8                        expected_goalsontarget  Expected goals on target (xGOT)    Attacking
9                          ontarget_scoring_att           Sh

In [60]:
static_url = "https://data.fotmob.com/stats/40/season/{season}/{stat}.json"
for stat in ["mins_played", "rating", "total_tackle", "interception", "won_tackle", "expected_goals"]:
    if stat not in set(stats_list.name):
        print(stat, "— not in statsList"); continue
    status, body = fotmob_get(static_url.format(season=belgium_2324, stat=stat))
    payload = json.loads(body) if status == 200 else {}
    rows = payload.get("TopLists", [{}])[0].get("StatList", []) if "TopLists" in payload else payload.get("statsData", payload if isinstance(payload, list) else [])
    print(stat, status, "| top-level:", list(payload.keys())[:6] if isinstance(payload, dict) else type(payload).__name__, "| rows:", len(rows), "| first:", json.dumps(rows[0], ensure_ascii=False)[:250] if rows else None)

mins_played

200

| top-level:

['TopLists', 'LeagueName']

| rows:

...

['TopLists', 'LeagueName']

| rows:

403

| first:

{"ParticipantName": "Kévin Denkey", "ParticiantId": 820477, "TeamId": 9984, "TeamColor": "#00521f", "StatValue": 25.0, "SubStatValue": 27.0, "MinutesPlayed": 3389, "MatchesPlayed": 38, "StatValueCount": 37, "Rank": 1, "ParticipantCountryCode": "TOG",

In [61]:
status, body = fotmob_get("https://www.fotmob.com/api/data/leagueseasondeepstats", params={"id": 40, "season": belgium_2324, "type": "players", "stat": "mins_played"})
minutes_rows = pd.DataFrame(json.loads(body)["statsData"])
minutes_rows["minutes"] = minutes_rows.statValue.map(lambda d: d["value"])
print(len(minutes_rows), "players with minutes | min minutes listed:", minutes_rows.minutes.min(), "| teams:", minutes_rows.teamId.nunique())
print("position codes:", minutes_rows.position.value_counts().to_dict())
minutes_rows.sort_values("minutes", ascending=False).head(5)

490

players with minutes | min minutes listed:

1

| teams:

16

position codes:

{11: 37, 115: 37, 36: 33, 34: 30, 38: 29, 64: 25, 85: 22, 66: 22, 32: 21, 87: 19, 105: 16, 83: 16, 106: 13, 103: 13, 33: 13, 77: 12, 35: 11, 37: 11, 107: 11, 104: 9, 68: 8, 84: 8, 3: 7, 86: 6, 76: 6, 75: 6, 78: 6, 73: 5, 74: 5, 72: 5, 62: 4, 65: 3, 71: 3, 79: 3, 1: 3, 2: 3, 82: 2, 88: 1, 51: 1, 53: 1, 67: 1, 94: 1, 58: 1, 59: 1}

,id,teamId,name,position,substatValue,statValue,rank,type,minutes
0,972200,9987,Maarten Vandevoordt,11,"{'value': 41, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3690, 'format...",1,players,3690
2,810487,9984,Warleson,11,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3600, 'format...",2,players,3600
1,1390101,9997,Matte Smets,36,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3600, 'format...",2,players,3600
3,445873,8342,Brandon Mechele,36,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3583, 'format...",4,players,3583
4,1280741,9997,Mathias Delorge,64,"{'value': 40, 'format': 'number', 'fractions': 0}","{'name': 'mins_played', 'value': 3551, 'format...",5,players,3551


In [62]:
# Position code → role: read it off known players
known = {"Simon Mignolet": "GK", "Hans Vanaken": "CM/AM", "Kévin Denkey": "ST", "Cameron Puertas": "AM"}
minutes_rows[minutes_rows.name.isin(known)][["name", "teamId", "position", "minutes"]].assign(expected=lambda d: d.name.map(known))

,name,teamId,position,minutes,expected
9,Kévin Denkey,9984,106,3389,ST
10,Hans Vanaken,8342,85,3348,CM/AM
13,Cameron Puertas,7978,77,3323,AM
49,Simon Mignolet,8342,11,2800,GK


## Part 6d — Absence semantics, oldest season, and FotMob league ids

Criterion: a per-90 list's minimum `MinutesPlayed` is its floor — players below it are `NaN`,
players in `mins_played` but absent from a totals list are 0. The oldest offered season must
return the same row shape. League ids are verified by the name the API returns.

In [63]:
def static_stat(league_id, season_id, stat):
    status, body = fotmob_get(f"https://data.fotmob.com/stats/{league_id}/season/{season_id}/{stat}.json")
    rows = json.loads(body)["TopLists"][0]["StatList"] if status == 200 else []
    return status, pd.DataFrame(rows)


minutes_list = static_stat(40, belgium_2324, "mins_played")[1]
for stat in ["total_tackle", "rating", "goals", "expected_goals"]:
    status, rows = static_stat(40, belgium_2324, stat)
    floor = rows.MinutesPlayed.min()
    eligible = int((minutes_list.StatValue >= floor).sum())
    print(f"{stat:16s} {status} rows={len(rows):3d} min MinutesPlayed={floor:5.0f} | players in mins_played with >= that: {eligible}")

total_tackle     200 rows=236 min MinutesPlayed=  437 | players in mins_played with >= that: 341

rating           200 rows=224 min MinutesPlayed=  683 | players in mins_played with >= that: 303

goals            200 rows=233 min MinutesPlayed=   45 | players in mins_played with >= that: 445

expected_goals   200 rows=403 min MinutesPlayed=    3 | players in mins_played with >= that: 481

In [64]:
oldest_season = min(season_ids.values())
status, rows = static_stat(40, oldest_season, "mins_played")
print("oldest season", oldest_season, [k for k, v in season_ids.items() if v == oldest_season], status, "| players:", len(rows), "| columns:", list(rows.columns))

oldest season

12768

['2018/2019']

200

| players:

439

| columns:

['ParticipantName', 'ParticiantId', 'TeamId', 'TeamColor', 'StatValue', 'SubStatValue', 'MinutesPlayed', 'MatchesPlayed', 'StatValueCount', 'Rank', 'ParticipantCountryCode', 'TeamName', 'Positions']

In [65]:
FOTMOB_LEAGUE_GUESSES = {"GB1": 47, "ES1": 87, "IT1": 55, "L1": 54, "FR1": 53,
                         "BE1": 40, "NL1": 57, "PO1": 61, "TR1": 71, "C1": 69, "BRA1": 268, "A1": 38, "DK1": 46}
for comp, league_id in FOTMOB_LEAGUE_GUESSES.items():
    status, body = fotmob_get("https://www.fotmob.com/api/data/leagues", params={"id": league_id})
    details = json.loads(body).get("details", {}) if status == 200 else {}
    seasons = json.loads(body).get("allAvailableSeasons", [])[-1:] if status == 200 else []
    print(comp, league_id, status, details.get("name"), details.get("country"), "| oldest season:", seasons)

GB1

47

200

Premier League

ENG

...

200

Superligaen

DEN

| oldest season:

['2010/2011']

**Note (Parts 6b–6d, run locally 2026-08-27).** `/api/data/leagueseasondeepstats?id&season=<id>&type=players&stat=<name>`
returns whole-league lists for the 37 names in its `statsList` (Attacking 16, Defending 8, Top
Stat 5, Goalkeeping 5 incl. `_goals_prevented`, Discipline 3); the same lists are static JSON at
`data.fotmob.com/stats/{league}/season/{season_id}/{stat}.json`, rows with `ParticipantName,
ParticiantId, TeamId, StatValue, SubStatValue, MinutesPlayed, MatchesPlayed, Rank, Positions`.
`mins_played` lists every player (Belgium 23/24: 490, 16 teams, down to 1 minute). Absence
semantics differ: per-90 lists apply an undocumented eligibility rule (`total_tackle` 236 rows
while 341 players clear its 437-minute minimum) → absent = `NaN`; totals lists have tiny floors
(xG 3 min, goals 45) → absent while in `mins_played` = 0. Oldest offered season (2018/19) has the
same shape. All 13 FotMob league ids verified by name. Local run reproduced Kaggle's Part 1
numbers (parity).

**DECIDED:** FotMob pulled per stat via the static JSON (≈37 × seasons × leagues), never per
player; per-90 absence → `NaN`, totals absence → 0 when the player is in `mins_played`; league
ids → `config.FOTMOB_LEAGUES`. Rejected: per-player pulls for Belgium/Denmark (~20k requests for
the same numbers). Position labels (`Positions`) inspected in the next run.

In [66]:
status, rows = static_stat(40, belgium_2324, "mins_played")
print("Positions values (Belgium 23/24):", rows.Positions.astype(str).value_counts().head(15).to_dict())
rows[["ParticipantName", "TeamName", "Positions", "MinutesPlayed"]].head(5)

Positions values (Belgium 23/24):

{'[11]': 37, '[36]': 27, '[115]': 27, '[34]': 17, '[105]': 12, '[33]': 11, '[37]': 11, '[38]': 11, '[34, 32]': 10, '[106]': 9, '[35]': 9, '[32]': 8, '[3]': 7, '[64]': 6, '[104]': 6}

,ParticipantName,TeamName,Positions,MinutesPlayed
0,Maarten Vandevoordt,KRC Genk,[11],3690
1,Matte Smets,Sint-Truidense VV,[36],3600
2,Warleson,Cercle Brugge,[11],3600
3,Brandon Mechele,Club Brugge,[36],3583
4,Mathias Delorge,Sint-Truidense VV,"[64, 75]",3551


## Part 7 — ClubElo naming/coverage and the Transfermarkt injury page

ClubElo: one club history (columns, date intervals) and one date snapshot (naming convention,
countries covered, levels) — decides whether feeder clubs get Elo at all. Injury page: one
player's table, and what '?' and '-' mean in `until` / `Days` / `Games missed`; a player with a
long history to see whether the page is paginated.

In [67]:
from io import StringIO


def clubelo_get(key):
    time.sleep(3)
    response = requests.get(f"http://api.clubelo.com/{key}", headers=PLAIN_UA, timeout=90)
    return response.status_code, response.text


status, body = clubelo_get("Arsenal")
arsenal_elo = pd.read_csv(StringIO(body))
print(status, len(arsenal_elo), "rows |", arsenal_elo.From.min(), "→", arsenal_elo.To.max(), "| columns:", list(arsenal_elo.columns))
arsenal_elo.tail(5)

EmptyDataError: No columns to parse from file

In [68]:
status, body = clubelo_get("2024-08-01")
elo_snapshot = pd.read_csv(StringIO(body))
print(status, len(elo_snapshot), "clubs on 2024-08-01 | levels:", elo_snapshot.Level.value_counts().to_dict())
print("clubs per country:", elo_snapshot.Country.value_counts().to_dict())

EmptyDataError: No columns to parse from file

In [69]:
ours = ["ENG", "ESP", "ITA", "GER", "FRA", "BEL", "NED", "POR", "TUR", "SUI", "AUT", "DEN", "BRA"]
print("countries missing from ClubElo:", [c for c in ours if c not in set(elo_snapshot.Country)])
elo_snapshot[elo_snapshot.Country.isin(ours) & (elo_snapshot.Level == 1)].groupby("Country").head(3).sort_values(["Country", "Rank"])[["Rank", "Club", "Country", "Level", "Elo"]]

NameError: name 'elo_snapshot' is not defined

In [70]:
kane_url = con.execute("SELECT url FROM players WHERE name = 'Harry Kane'").fetchone()[0].replace("/profil/", "/verletzungen/")
time.sleep(3)
response = requests.get(kane_url, headers=PLAIN_UA, timeout=30)
injury_tables = pd.read_html(StringIO(response.text))
kane_injuries = next(t for t in injury_tables if any("injury" in str(c).lower() for c in t.columns))
print(response.status_code, kane_url, "|", len(kane_injuries), "rows | columns:", list(kane_injuries.columns))
kane_injuries

200

https://www.transfermarkt.co.uk/harry-kane/verletzungen/spieler/132098

|

15

rows | columns:

['Season', 'Injury', 'from', 'until', 'Days', 'Games missed']

,Season,Injury,from,until,Days,Games missed
0,25/26,Ankle problems,31/03/2026,06/04/2026,7 days,2
1,25/26,Calf problems,05/03/2026,08/03/2026,4 days,1
2,24/25,Torn muscle fiber,01/12/2024,15/12/2024,15 days,4
3,23/24,Back problems,09/05/2024,01/06/2024,24 days,2
4,23/24,Ankle injury,17/03/2024,27/03/2024,11 days,2
5,20/21,Ankle injury,16/04/2021,22/04/2021,7 days,1
6,20/21,Ankle injury,29/01/2021,05/02/2021,8 days,2
7,20/21,unknown injury,30/11/2020,05/12/2020,6 days,1
8,19/20,Fitness,09/03/2020,01/04/2020,24 days,1
9,19/20,Torn thigh muscle,02/01/2020,09/03/2020,68 days,14


In [71]:
print("raw 'until':", kane_injuries["until"].astype(str).unique()[:10])
print("raw 'Days':", kane_injuries["Days"].astype(str).unique()[:10])
print("raw 'Games missed':", kane_injuries["Games missed"].astype(str).unique()[:10])

raw 'until':

<ArrowStringArray>
['06/04/2026', '08/03/2026', '15/12/2024', '01/06/2024', '27/03/2024', '22/04/2021', '05/02/2021', '05/12/2020', '01/04/2020', '09/03/2020']
Length: 10, dtype: str

raw 'Days':

<ArrowStringArray>
['7 days', '4 days', '15 days', '24 days', '11 days', '8 days', '6 days', '68 days', '52 days', '41 days']
Length: 10, dtype: str

raw 'Games missed':

<ArrowStringArray>
['2', '1', '4', '14', '9', '7', '-', '3']
Length: 8, dtype: str

In [72]:
# A long injury history: is the page paginated?
reus_url = con.execute("SELECT url FROM players WHERE name = 'Marco Reus'").fetchone()[0].replace("/profil/", "/verletzungen/")
time.sleep(3)
response = requests.get(reus_url, headers=PLAIN_UA, timeout=30)
reus_tables = pd.read_html(StringIO(response.text))
reus_injuries = next(t for t in reus_tables if any("injury" in str(c).lower() for c in t.columns))
print(response.status_code, len(reus_injuries), "rows | earliest season:", reus_injuries["Season"].astype(str).min(), "| pagination markers on page:", "page/2" in response.text or "pager" in response.text.lower())
reus_injuries.tail(3)

200

15

rows | earliest season:

21/22

| pagination markers on page:

True

,Season,Injury,from,until,Days,Games missed
12,21/22,Ill,07/03/2022,28/03/2022,22 days,3
13,21/22,minor knock,24/02/2022,07/03/2022,12 days,1
14,21/22,Knee problems,23/09/2021,27/09/2021,5 days,1


## Part 7b — Injury pagination and ClubElo key format

Criterion: if `/page/2` returns different rows, the injury pull walks pages until an empty or
repeated page; record how many pages a long history needs and whether `?` appears. ClubElo keys
for multi-word clubs are confirmed by a 200 with matching `Club` values; Brazil's absence by a
lookup of Flamengo.

In [73]:
def injury_page(base_url, page):
    time.sleep(3)
    url = base_url if page == 1 else f"{base_url}/page/{page}"
    response = requests.get(url, headers=PLAIN_UA, timeout=30)
    tables = pd.read_html(StringIO(response.text)) if response.status_code == 200 else []
    table = next((t for t in tables if any("injury" in str(c).lower() for c in t.columns)), pd.DataFrame())
    return response.status_code, table


for name, base_url in [("Harry Kane", kane_url), ("Marco Reus", reus_url)]:
    pages = []
    for page in range(1, 8):
        status, table = injury_page(base_url, page)
        if table.empty or (pages and table.equals(pages[-1])):
            break
        pages.append(table)
    history = pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()
    print(name, "| pages fetched:", len(pages), "| rows:", len(history), "| seasons:", history["Season"].astype(str).min() if len(history) else None, "→", history["Season"].astype(str).max() if len(history) else None)
    print("   distinct 'Games missed' values:", sorted(history["Games missed"].astype(str).unique().tolist()) if len(history) else None)
    print("   distinct non-numeric 'Days':", [v for v in history["Days"].astype(str).unique() if "day" not in v] if len(history) else None)
    print("   distinct non-date 'until':", [v for v in history["until"].astype(str).unique() if "/" not in v] if len(history) else None)

Harry Kane

| pages fetched:

2

| rows:

21

...

['-', '1', '10', '2', '3', '30', '4', '40', '5', '7', '8', '9']

   distinct non-numeric 'Days':

[]

   distinct non-date 'until':

[]

In [74]:
for key in ["ManCity", "Man City", "ParisSG", "Paris SG", "FCKobenhavn", "Bueyueksehir", "Flamengo"]:
    status, body = clubelo_get(key)
    head = pd.read_csv(StringIO(body)) if status == 200 and body.startswith("Rank") else pd.DataFrame()
    print(f"{key:14s} {status} rows={len(head):5d}", "| club:", head.Club.iloc[0] if len(head) else body[:80].replace("\n", " "))

ReadTimeout: HTTPConnectionPool(host='api.clubelo.com', port=80): Read timed out. (read timeout=90)

**Note (Parts 7–7b, run locally 2026-08-27).** ClubElo: club history = intervals `From`/`To`
with `Elo`, `Rank` (top ~100 only), `Level`; Arsenal 6,507 intervals 1946 → 2026-12-31 (the
tail is a projection; reading Elo at past match dates never touches it). Snapshot 2024-08-01:
629 clubs, first divisions of all 12 European countries we need (BEL 16, NED 18, POR 18, TUR 19,
SUI/AUT/DEN 12); **Brazil absent** (`Flamengo` → 200 with an empty CSV). Key = display name
without spaces (`ManCity`, `ParisSG`, `FCKobenhavn`, `Bueyueksehir`); a wrong key also returns
200 + header only, so missing is detected from the empty body. Injury page: columns
`Season, Injury, from, until, Days, Games missed`; dates `dd/mm/yyyy`, `Days` always "N days",
`Games missed` numeric or `-`; **paginated at 15 rows** (`/page/N`): Kane 2 pages / 21 spells to
12/13, Reus 5 pages / 72 spells to 09/10 — Phase 0's "complete to 2009-10" held only for
players with < 15 spells. FotMob `Positions` is a list of numeric codes (`[34, 32]`), unmapped.

**DECIDED:** injury pull walks pages until empty/repeat; `Days` is the primary injury quantity,
`-` in `Games missed` → `NaN`; ClubElo key = name with spaces removed, empty CSV = missing,
Brazil → `NaN` opponent Elo (Tier-2 handling is Step 5a's decision). Step 1 complete: all six
sources read; fetch code (Step 2) is written against these notes.

# Step 2 — Fetch layer checks

Each fetcher lives in `src/scout/data/` with a unit test; this section imports it and records
what it returns on the real data, so the numbers the code is judged by are stored outputs
(CLAUDE.md §4: notebooks import from the package; §8: verify the port reproduces the numbers).

## 2.1 Transfermarkt — `scout.data.transfermarkt.load_player_club_seasons`

Expected: 33,061 appearance-based Big-5 rows (Phase 0), plus lineup-only rows with `NaN` minutes
that fill the appearance holes found in Part 3 (Atlético 2014-15 had 3 players).

In [75]:
from scout import config
from scout.data import transfermarkt as tm_loader

player_club_seasons = tm_loader.load_player_club_seasons(list(config.BIG5), config.SEASONS)
print("rows:", len(player_club_seasons), "| by source:", player_club_seasons.source.value_counts().to_dict(),
      "| players:", player_club_seasons.tm_player_id.nunique())
print("lineups-only rows with NaN minutes:", int(player_club_seasons[player_club_seasons.source == "lineups"].minutes.isna().sum()),
      "of", int((player_club_seasons.source == "lineups").sum()))
print("rows without a club name:", int(player_club_seasons.club_name.isna().sum()))
atletico_2014 = player_club_seasons[(player_club_seasons.club_id == 13) & (player_club_seasons.season == 2014)]
print("Atlético 2014-15 by source:", atletico_2014.source.value_counts().to_dict())

rows:

40299

| by source:

{'appearances': 33061, 'lineups': 7238}

| players:

...

7238

rows without a club name:

0

Atlético 2014-15 by source:

{'lineups': 23, 'appearances': 3}

## 2.2 Understat — `scout.data.understat.load` (fast kinds; per-match kinds land later)

Expected: 60 league-seasons per kind; 32,574 player-seasons (Phase 0); minutes per league-season
≈ 0.75M for 380-game leagues, 0.60M for the Bundesliga, 0.55M for the COVID-cut Ligue 1 2019-20.

In [76]:
from scout.data import understat

season_players = understat.load("player_season")
team_matches = understat.load("team_match")
schedules = understat.load("schedule")
print("player-seasons:", len(season_players), "| league-seasons:", season_players.groupby(["league", "season"]).ngroups,
      "| team-matches:", len(team_matches), "| scheduled matches:", len(schedules))
minutes = season_players.groupby(["league", "season"]).minutes.sum().unstack("league") / 1e6
minutes.round(2)

player-seasons:

32574

| league-seasons:

60

| team-matches:

21589

| scheduled matches:

21690

league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A
season,,,,,
2014,0.75,0.75,0.75,0.6,0.75
2015,0.75,0.75,0.75,0.61,0.75
2016,0.75,0.75,0.75,0.6,0.75
2017,0.75,0.75,0.75,0.6,0.75
2018,0.75,0.75,0.75,0.6,0.75
2019,0.75,0.75,0.55,0.6,0.75
2020,0.75,0.75,0.75,0.61,0.75
2021,0.75,0.75,0.75,0.61,0.75
2022,0.75,0.75,0.75,0.6,0.75


In [77]:
raw_counts = {kind: len(list((understat.RAW / "understat" / kind).glob("*.parquet"))) for kind in understat.KINDS}
print(raw_counts, "← per-match kinds fill in as the background pull runs")
no_stats = schedules[~schedules.game_id.isin(team_matches.game_id)]
print(len(no_stats), "scheduled matches without team-match stats | has_data:", no_stats.has_data.value_counts().to_dict(),
      "| is_result:", no_stats.is_result.value_counts().to_dict(), "| seasons:", no_stats.season.value_counts().to_dict())

{'player_season': 60, 'player_match': 16, 'shots': 16, 'team_match': 60, 'schedule': 60}

← per-match kinds fill in as the background pull runs

101

scheduled matches without team-match stats | has_data:

{np.False_: 101}

| is_result:

{np.False_: 101}

| seasons:

{2019: 101}

## 2.3 Sofascore — `scout.data.sofascore.load` (files land as the background pull runs)

Expected per league-season: every player who played, in four position groups (PL 23/24: 570,
G 40 / D 188 / M 232 / F 110); `None` only in keeper xG/xA.

In [78]:
from scout.data import sofascore

sofascore_players = sofascore.load()
print("league-seasons on disk:", sofascore_players.groupby(["competition_id", "season"]).ngroups, "of 143",
      "| players:", len(sofascore_players))
print("players per league-season (median):", sofascore_players.groupby(["competition_id", "season"]).size().median())
print("position groups:", sofascore_players.position_group.value_counts().to_dict())
print("fields with any None (share):", sofascore_players[sofascore.FIELDS].isna().mean()[lambda s: s > 0].round(3).to_dict())

league-seasons on disk:

142

of 143

| players:

71505

...

525.5

position groups:

{'M': 27550, 'D': 23333, 'F': 15202, 'G': 5420}

fields with any None (share):

{'minutesPlayed': 0.0, 'appearances': 0.0, 'rating': 0.079, 'goals': 0.0, 'assists': 0.1, 'expectedGoals': 0.733, 'expectedAssists': 0.698, 'keyPasses': 0.079, 'bigChancesCreated': 0.079, 'accuratePasses': 0.079, 'accuratePassesPercentage': 0.079, 'accurateFinalThirdPasses': 0.079, 'accurateLongBalls': 0.079, 'accurateCrosses': 0.079, 'successfulDribbles': 0.079, 'tackles': 0.079, 'interceptions': 0.079, 'clearances': 0.079, 'ballRecovery': 0.71, 'possessionWonAttThird': 0.079, 'possessionLost': 0.079, 'groundDuelsWon': 0.079, 'groundDuelsWonPercentage': 0.079, 'aerialDuelsWon': 0.079, 'aerialDuelsWonPercentage': 0.079, 'dribbledPast': 0.079, 'fouls': 0.079, 'wasFouled': 0.079, 'errorLeadToShot': 0.079, 'errorLeadToGoal': 0.079, 'touches': 0.079, 'saves': 0.079, 'goalsConceded': 0.104, 'savedShotsFromInsideTheBox': 0.079, 'savedShotsFromOutsideTheBox': 0.079, 'highClaims': 0.079, 'punches': 0.079, 'penaltyFaced': 0.079, 'penaltySave': 0.079, 'cleanSheet': 0.104}

## 2.4 FotMob — `scout.data.fotmob.load` (raw long form; files land as the background pull runs)

Expected per league-season: 37 stat lists; `mins_played` lists every player (Belgium 23/24: 490,
16 teams); keeper lists hold ~one keeper per team (an eligibility floor, like the per-90 lists).

In [79]:
from scout.data import fotmob

fotmob_rows = fotmob.load()
league_seasons = fotmob_rows.groupby(["competition_id", "season"])
print("league-seasons on disk:", league_seasons.ngroups, "| rows:", len(fotmob_rows), "| stats:", fotmob_rows.stat.nunique())
print("players per league-season via mins_played (median):", fotmob_rows[fotmob_rows.stat == "mins_played"].groupby(["competition_id", "season"]).size().median())
list_sizes = fotmob_rows.groupby("stat").size().sort_values()
print("smallest lists (eligibility floors):", list_sizes.head(6).to_dict())
print("earliest season per league:", fotmob_rows.groupby("competition_id").season.min().to_dict())

league-seasons on disk:

27

| rows:

240080

| stats:

...

555.0

smallest lists (eligibility floors):

{'phys_tdc_per_90': 210, '_goals_prevented': 302, 'phys_sprints_per_90': 333, 'phys_sprints': 514, 'goals_conceded': 531, 'saves': 531}

earliest season per league:

{'BE1': 2023, 'ES1': 2016, 'GB1': 2016, 'IT1': 2016}

## 2.5 ClubElo — `scout.data.clubelo.fetch_club` / `elo_on`

Expected: a known club returns decades of intervals; a multi-word name works through the
spaceless key; an unknown club (Flamengo) is an empty frame, not an error; Elo at a past match
date resolves, at a date before the history starts it is `NaN`.

In [80]:
from scout.data import clubelo

arsenal = clubelo.fetch_club("Arsenal")
man_city = clubelo.fetch_club("Man City")
flamengo = clubelo.fetch_club("Flamengo")
print("Arsenal intervals:", len(arsenal), arsenal.From.min().date(), "→", arsenal.To.max().date(),
      "| Man City intervals:", len(man_city), "| Flamengo rows:", len(flamengo))
print("Arsenal Elo on 2023-08-12:", clubelo.elo_on(arsenal, "2023-08-12"), "| on 1900-01-01:", clubelo.elo_on(arsenal, "1900-01-01"),
      "| Flamengo on 2023-08-12:", clubelo.elo_on(flamengo, "2023-08-12"))

ReadTimeout: HTTPConnectionPool(host='api.clubelo.com', port=80): Read timed out. (read timeout=30)

## 2.6 Injuries — `scout.data.injuries.fetch_injuries` (paginated) and `load`

Expected: Kane 21 spells over 2 pages (Part 7b), Reus 72 over 5; `-` games missed → `NaN`;
the background pull appends 50 players at a time with a marker row for players without spells.

In [81]:
from scout.data import injuries, transfermarkt as tm_loader

players_table = tm_loader.load_table("players")
for name in ["Harry Kane", "Marco Reus"]:
    url = players_table.loc[players_table.name == name, "url"].iloc[0]
    spells = injuries.fetch_injuries(url)
    print(name, "| spells:", len(spells), "| seasons:", spells.season.min(), "→", spells.season.max(),
          "| games_missed NaN:", int(spells.games_missed.isna().sum()), "| days NaN:", int(spells.days.isna().sum()))
pulled = injuries.load()
print("pulled so far:", pulled.tm_player_id.nunique(), "players,", int(pulled.season.notna().sum()), "spells")

Harry Kane

| spells:

21

| seasons:

12/13

...

pulled so far:

50

players,

644

spells

**Note (Step 2 checks, 2026-08-28).** All six fetchers exist in `scout.data.*` with tests and
reproduce the Step 1 units. Findings from the first full-data checks: the 101 Understat fixtures
without team stats are all 2019-20 with `has_data = False` — the COVID-cancelled Ligue 1 games,
correctly absent. Sofascore: 143/143 league-seasons, 71,505 player-seasons, median 525 per
league-season; `expectedGoals` is `None` in 73% of player-seasons, `expectedAssists` 70%,
`ballRecovery` 71% (fields that exist only for recent seasons), and a flat 7.9% `None` across
almost every field (one league/era with a thinner set) — per-league/season coverage of each field
is a Step 5 question. Brazil 2020 is labelled `20/21` on Sofascore (COVID split season); the
fetcher tries both labels. FotMob: 42 distinct stats (the Premier League adds `phys_*` tracking
stats), eligibility floors visible in list sizes. ClubElo timed out at 30 s and 90 s in this run
(slow host; Phase 0 saw the same) — fetcher timeout raised to 120 s, cell 2.5 fills on rerun.
Injuries: Kane 21 spells, Reus 72, as walked by hand.

# Step 3 — Team lineage for the new provider names

`scout.identity.build_team_lineage` applies the Phase 0 rule (token-set score ≥ 90, shortest
name on ties, committed overrides win). Criterion: 0 unresolved names and 0 wrong pairs on
review — every auto match whose provider name differs from the Transfermarkt name is listed
and read; overrides are written for the leftovers.

In [2]:
from scout import config
from scout.data import fotmob, sofascore, transfermarkt as tm_loader, understat
from scout.identity import build_team_lineage, load_overrides, normalize_name

competitions = list(config.BIG5) + list(config.FEEDERS)
tm_clubs = (tm_loader.load_player_club_seasons(competitions, config.SEASONS)
            [["club_id", "club_name", "competition_id"]].drop_duplicates())
league_to_comp = {league: comp for comp, league in config.BIG5.items()}
understat_players = understat.load("player_season")
provider_teams = {
    "understat": understat_players.assign(competition_id=understat_players.league.map(league_to_comp))
                 [["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"}),
    "sofascore": sofascore.load()[["competition_id", "team_name", "sofascore_team_id"]].drop_duplicates()
                 .rename(columns={"sofascore_team_id": "provider_team_id"}),
    "fotmob": fotmob.load()[["competition_id", "team_name", "fotmob_team_id"]].drop_duplicates()
              .rename(columns={"fotmob_team_id": "provider_team_id"}),
}
lineage = build_team_lineage(tm_clubs, provider_teams, load_overrides("teams"))
print("teams per provider:", lineage.provider.value_counts().to_dict())
print("unresolved per provider:", lineage[lineage.club_id.isna()].provider.value_counts().to_dict())
print("sources:", lineage.source.value_counts().to_dict())

teams per provider: {'sofascore': 390, 'fotmob': 175, 'understat': 168}
unresolved per provider: {'sofascore': 34, 'fotmob': 3}
sources: {'auto': 689, 'override': 44}


In [3]:
unresolved = lineage[lineage.club_id.isna()].sort_values(["provider", "competition_id", "team_name"])
print(len(unresolved), "unresolved")
print(unresolved[["provider", "competition_id", "team_name", "provider_team_id", "score"]].to_string())

37 unresolved
      provider competition_id                  team_name  provider_team_id      score
630     fotmob            FR1                      Brest            8521.0  50.000000
620     fotmob            FR1                     Rennes            9851.0  53.333333
624     fotmob            FR1                 St Etienne            9853.0  82.352941
169  sofascore             A1              Admira Wacker            2050.0  41.176471
181  sofascore             A1        FC Wacker Innsbruck            2088.0  43.243243
185  sofascore             A1        SC Austria Lustenau            2062.0  70.000000
178  sofascore             A1             SKN St. Pölten            2119.0  40.909091
177  sofascore             A1                  SV Grödig            6002.0  62.500000
172  sofascore             A1             SV Mattersburg            2061.0  61.538462
180  sofascore             A1       Wiener Neustädter SC           23642.0  47.058824
212  sofascore            BE1           

In [4]:
# Suspicious: auto matches whose normalised names differ (the Barcelona→Espanyol failure mode)
resolved = lineage[lineage.club_id.notna() & (lineage.source == "auto")]
differing = resolved[resolved.team_name.map(normalize_name) != resolved.club_name.map(normalize_name)]
print(len(differing), "auto matches with differing names; lowest scores first")
print(differing.sort_values("score")[["provider", "competition_id", "team_name", "club_name", "score"]].head(80).to_string())

375 auto matches with differing names; lowest scores first
      provider competition_id                     team_name                       club_name       score
265  sofascore             C1             FC Lausanne-Sport    Football Club Lausanne-Sport   90.322581
508  sofascore            PO1              Portimonense SAD                 Portimonense SC   90.322581
504  sofascore            PO1            Vitória de Setúbal              Vitória Setúbal FC   90.909091
730     fotmob             L1         1. FC Heidenheim 1846  1. Fußballclub Heidenheim 1846   91.891892
194  sofascore            BE1         RC Sporting Charleroi   Royal Charleroi Sporting Club   92.307692
572     fotmob            BE1         Sporting de Charleroi   Royal Charleroi Sporting Club   92.307692
159  understat             L1           Fortuna Duesseldorf              Fortuna Düsseldorf   97.297297
309  sofascore            ES1         Deportivo de A Coruña          Deportivo de La Coruña   97.674419
490  

In [5]:
# Two provider names mapping to one club inside a competition: legitimate renames or wrong matches?
dupes = (lineage.dropna(subset=["club_id"]).groupby(["provider", "competition_id", "club_id"])
         .team_name.agg(list).loc[lambda s: s.str.len() > 1])
print(len(dupes), "clubs with several provider names")
print(dupes.to_string())

2 clubs with several provider names
provider   competition_id  club_id
sofascore  BE1             498        [KSC Lokeren Oost-Vlaanderen, KSC Lokeren]
understat  IT1             130                        [Parma, Parma Calcio 1913]


### 3b — Why do some provider teams have no Transfermarkt club at all?

After the spelling overrides, the leftovers are names with no candidate in the Transfermarkt
panel. Criterion: compare clubs per league on each side; if Transfermarkt lists fewer clubs than
the provider for a feeder league, the gap is dataset coverage (smaller/older clubs and play-off
sides), and those provider rows fall back to name-only identity matching.

In [6]:
lineage = build_team_lineage(tm_clubs, provider_teams, load_overrides("teams"))
still_unresolved = lineage[lineage.club_id.isna()]
print("unresolved after overrides:", still_unresolved.provider.value_counts().to_dict(), "| overrides applied:", int((lineage.source == "override").sum()))
tm_clubs_per_league = tm_clubs.groupby("competition_id").club_id.nunique().rename("tm_clubs")
provider_clubs_per_league = provider_teams["sofascore"].groupby("competition_id").team_name.nunique().rename("sofascore_teams")
unresolved_per_league = still_unresolved.groupby("competition_id").size().rename("unresolved")
print(pd.concat([tm_clubs_per_league, provider_clubs_per_league, unresolved_per_league], axis=1).fillna(0).astype(int).to_string())
print(still_unresolved.sort_values(["competition_id", "team_name"])[["provider", "competition_id", "team_name"]].to_string())

unresolved after overrides: {'sofascore': 34, 'fotmob': 3} | overrides applied: 44
                tm_clubs  sofascore_teams  unresolved
competition_id                                       
A1                    13               20           7
BE1                   26               32           5
BRA1                  24               35          12
C1                    13               17           4
DK1                   21               20           0
ES1                   32               31           0
FR1                   34               35           5
GB1                   35               34           0
IT1                   36               35           0
L1                    30               30           0
NL1                   29               33           4
PO1                   33               29           0
TR1                   41               39           0
      provider competition_id                  team_name
169  sofascore             A1              Admira 

**Note (Step 3, 2026-08-28).** 700 provider team names (Sofascore 390, Understat 168, FotMob 142)
against the Transfermarkt panel with the Phase 0 rule: 657 auto matches, 44 overrides, **34
unresolved** — all Sofascore, all in leagues where Transfermarkt's panel lists fewer clubs than
the provider (A1 13 vs 20, BRA1 24 vs 35, C1 13 vs 17) or play-off sides from second divisions
(Rodez, Dunkerque, Den Bosch, Deinze, Roeselare, Lommel). Review of every auto match with a
differing name found no wrong pair; the only club with two provider names is Parma (renamed).
Traps caught by hand: Sporting Braga → SC Braga (not Sporting CP), FC Eindhoven ≠ PSV, CD Aves →
Desportivo Aves, Athletico → Paranaense, AaB → Aalborg BK.

**DECIDED:** the 34 unresolved teams stay unresolved — a Transfermarkt coverage gap, not a naming
one; their players get no club-season key and fall back to name-only matching, measured in
Step 4. Rejected: lowering the score threshold (the candidates for these names are wrong clubs).

# Step 4 — Player identity policy (decision D1)

Candidates: (a) modal Transfermarkt id per provider id across seasons; (b) per-season matches
kept as-is; (c) modal with a share floor. Fuzzy threshold re-fit on Sofascore names in
{85, 88, 90, 92, 95}. Criterion: fewest provider ids mapped to more than one Transfermarkt id
without lowering the minutes-weighted rate; the highest threshold with zero errors in a
hand-checked sample of 30 fuzzy hits. Then: what a name-only fallback recovers for players of
the 34 unresolved teams, and what the reep alias register would add for mononyms.

In [7]:
from scout.identity import match_players

tm_panel = tm_loader.load_player_club_seasons(competitions, config.SEASONS)
tm_side = tm_panel.rename(columns={"tm_player_id": "right_id"})[["right_id", "name", "club_id", "season", "competition_id"]].copy()
tm_side["club_key"] = tm_side.club_id.astype("Int64").astype(str)

sofascore_side = sofascore.load().merge(
    lineage[lineage.provider == "sofascore"][["competition_id", "team_name", "club_id"]],
    on=["competition_id", "team_name"], how="left",
).rename(columns={"player_name": "name"})
sofascore_side["club_key"] = sofascore_side.club_id.astype("Int64").astype(str)
print("sofascore player-seasons:", len(sofascore_side), "| without a club key (unresolved teams):", int(sofascore_side.club_id.isna().sum()))


def run_cascade(comp, season, min_fuzzy):
    left = sofascore_side[(sofascore_side.competition_id == comp) & (sofascore_side.season == season)]
    right = tm_side[(tm_side.competition_id == comp) & (tm_side.season == season)]
    return match_players(left[["name", "club_key", "season", "sofascore_player_id", "minutesPlayed"]], right, min_fuzzy=min_fuzzy)


for comp, season in [("GB1", 2023), ("A1", 2023)]:
    for min_fuzzy in (85, 88, 90, 92, 95):
        out = run_cascade(comp, season, min_fuzzy)
        hit = out.right_id.notna()
        minutes_rate = out.loc[hit, "minutesPlayed"].sum() / out.minutesPlayed.sum()
        print(f"{comp} {season} fuzzy>={min_fuzzy}: methods={out.method.value_counts().to_dict()} | minutes-weighted rate={minutes_rate:.3%}")

sofascore player-seasons: 71505 | without a club key (unresolved teams): 2089
GB1 2023 fuzzy>=85: methods={'exact': 550, 'last_token': 14, 'fuzzy': 6} | minutes-weighted rate=100.000%
GB1 2023 fuzzy>=88: methods={'exact': 550, 'last_token': 14, 'fuzzy': 5, 'unmatched': 1} | minutes-weighted rate=99.905%
GB1 2023 fuzzy>=90: methods={'exact': 550, 'last_token': 14, 'fuzzy': 5, 'unmatched': 1} | minutes-weighted rate=99.905%
GB1 2023 fuzzy>=92: methods={'exact': 550, 'last_token': 14, 'fuzzy': 5, 'unmatched': 1} | minutes-weighted rate=99.905%
GB1 2023 fuzzy>=95: methods={'exact': 550, 'last_token': 14, 'fuzzy': 5, 'unmatched': 1} | minutes-weighted rate=99.905%
A1 2023 fuzzy>=85: methods={'unmatched': 330} | minutes-weighted rate=0.000%
A1 2023 fuzzy>=88: methods={'unmatched': 330} | minutes-weighted rate=0.000%


A1 2023 fuzzy>=90: methods={'unmatched': 330} | minutes-weighted rate=0.000%
A1 2023 fuzzy>=92: methods={'unmatched': 330} | minutes-weighted rate=0.000%
A1 2023 fuzzy>=95: methods={'unmatched': 330} | minutes-weighted rate=0.000%


In [8]:
# Hand-check: every fuzzy hit at the loosest threshold, with the score band it fell in
tm_names = tm_side.drop_duplicates("right_id").set_index("right_id").name
for comp, season in [("GB1", 2023), ("A1", 2023)]:
    loose = run_cascade(comp, season, 85)
    fuzzy_hits = loose[loose.method == "fuzzy"].copy()
    fuzzy_hits["tm_name"] = fuzzy_hits.right_id.map(tm_names)
    from rapidfuzz import fuzz
    fuzzy_hits["score"] = [fuzz.token_set_ratio(normalize_name(a), normalize_name(b)) for a, b in zip(fuzzy_hits.name, fuzzy_hits.tm_name)]
    print(f"== {comp} {season}: {len(fuzzy_hits)} fuzzy hits at >=85")
    print(fuzzy_hits.sort_values("score")[["name", "tm_name", "score", "minutesPlayed"]].head(40).to_string())

== GB1 2023: 6 fuzzy hits at >=85
                         name          tm_name       score  minutesPlayed
41420         Yehor Yarmoliuk  Yegor Yarmolyuk   86.666667          716.0
41074       Gabriel Magalhães          Gabriel  100.000000         3053.0
41359         Hannibal Mejbri         Hannibal  100.000000          138.0
41468  Jaden Philogene-Bidace  Jaden Philogene  100.000000           10.0
41484           Son Heung-min    Heung-min Son  100.000000         2946.0
41517          Hwang Hee-chan   Hee-chan Hwang  100.000000         2124.0
== A1 2023: 0 fuzzy hits at >=85
Empty DataFrame
Columns: [name, tm_name, score, minutesPlayed]
Index: []


In [9]:
# Across all seasons: does one Sofascore id map to more than one Transfermarkt id? (policy a vs b)
all_matches = []
for (comp, season), left in sofascore_side.groupby(["competition_id", "season"]):
    right = tm_side[(tm_side.competition_id == comp) & (tm_side.season == season)]
    out = match_players(left[["name", "club_key", "season", "sofascore_player_id", "minutesPlayed"]], right, min_fuzzy=92)
    out["competition_id"] = comp
    all_matches.append(out)
all_matches = pd.concat(all_matches, ignore_index=True)
matched = all_matches.dropna(subset=["right_id"])
ids_per_provider_id = matched.groupby("sofascore_player_id").right_id.nunique()
print("sofascore ids matched:", len(ids_per_provider_id), "| mapped to >1 TM id:", int((ids_per_provider_id > 1).sum()),
      f"({(ids_per_provider_id > 1).mean():.2%})")
print("overall minutes-weighted rate at 92:", f"{matched.minutesPlayed.sum() / all_matches.minutesPlayed.sum():.3%}",
      "| by method:", all_matches.method.value_counts().to_dict())
conflicts = matched[matched.sofascore_player_id.isin(ids_per_provider_id[ids_per_provider_id > 1].index)]
conflict_view = conflicts.assign(tm_name=conflicts.right_id.map(tm_names)).sort_values(["sofascore_player_id", "season"])
print(conflict_view[["sofascore_player_id", "name", "season", "competition_id", "right_id", "tm_name", "method"]].head(30).to_string())

sofascore ids matched: 17735 | mapped to >1 TM id: 30 (0.17%)
overall minutes-weighted rate at 92: 84.122% | by method: {'exact': 55337, 'unmatched': 13379, 'last_token': 1703, 'fuzzy': 859, 'name_unique': 227}
       sofascore_player_id               name  season competition_id  right_id           tm_name      method
24448              17787.0            Marcelo    2015            ES1   44501.0           Marcelo       exact
25027              17787.0            Marcelo    2016            ES1   44501.0           Marcelo       exact
25543              17787.0            Marcelo    2017            ES1   44501.0           Marcelo       exact
26117              17787.0            Marcelo    2018            ES1   44501.0           Marcelo       exact
26616              17787.0            Marcelo    2019            ES1   44501.0           Marcelo       exact
27190              17787.0            Marcelo    2020            ES1   44501.0           Marcelo       exact
27756              17787.0

In [10]:
# Unmatched: where are they, and what does a name-only fallback (unique normalised name within
# the competition-season, no club key) recover for players of the 34 unresolved teams?
unmatched = all_matches[all_matches.right_id.isna()]
print("unmatched player-seasons:", len(unmatched), "| minutes share:", f"{unmatched.minutesPlayed.sum() / all_matches.minutesPlayed.sum():.2%}")
print("unmatched with no club key (unresolved teams):", int((unmatched.club_key == '<NA>').sum()), "| by league:", unmatched.competition_id.value_counts().head(8).to_dict())
tm_side["norm"] = tm_side.name.map(normalize_name)
unique_names = tm_side.groupby(["competition_id", "season", "norm"]).right_id.nunique()
unique_names = unique_names[unique_names == 1].reset_index()[["competition_id", "season", "norm"]]
fallback = unmatched.assign(norm=unmatched.name.map(normalize_name)).merge(unique_names.assign(name_only_hit=True), on=["competition_id", "season", "norm"], how="left")
print("name-only fallback would recover:", int(fallback.name_only_hit.fillna(False).sum()), "of", len(unmatched))
print("still unmatched, top by minutes:")
print(fallback[fallback.name_only_hit.isna()].nlargest(15, "minutesPlayed")[["competition_id", "season", "name", "minutesPlayed"]].to_string())

unmatched player-seasons: 13379 | minutes share: 15.88%
unmatched with no club key (unresolved teams): 0 | by league: {'BRA1': 7022, 'A1': 2886, 'C1': 2673, 'BE1': 405, 'DK1': 164, 'PO1': 128, 'ES1': 41, 'TR1': 22}
name-only fallback would recover: 97 of 13379
still unmatched, top by minutes:
      competition_id  season              name  minutesPlayed
3328            BRA1    2015            Victor         3510.0
5357            BRA1    2018           Jandrei         3424.0
4615            BRA1    2017            Wilson         3420.0
4617            BRA1    2017    Jean Fernandes         3420.0
4637            BRA1    2017           Jandrei         3420.0
5341            BRA1    2018            Victor         3420.0
6760            BRA1    2020       Tiago Volpi         3420.0
7492            BRA1    2021       Tiago Volpi         3420.0
8209            BRA1    2022             Tadeu         3420.0
8214            BRA1    2022             Fábio         3420.0
8973            BRA1    

In [11]:
# reep register: what does the repository expose? (listing only; download decided after reading)
import json, time, requests

time.sleep(3)
listing = requests.get("https://api.github.com/repos/withqwerty/reep/contents", timeout=30)
print(listing.status_code, [item["name"] for item in listing.json()][:30] if listing.status_code == 200 else listing.text[:200])
time.sleep(3)
readme = requests.get("https://raw.githubusercontent.com/withqwerty/reep/main/README.md", timeout=30)
print(readme.status_code); print("\n".join(line for line in readme.text.split("\n") if "csv" in line.lower() or "duckdb" in line.lower() or "download" in line.lower())[:1500])

200 ['.github', '.gitignore', 'CHANGELOG.md', 'CLAUDE.md', 'LICENSE', 'README.md', 'data', 'openapi.yaml', 'package.json', 'pnpm-lock.yaml', 'src', 'tsconfig.json', 'wrangler.toml']


200
free to download and use.
| Download the register (CSV + DuckDB) | [reep.football/downloads](https://reep.football/downloads) |
> **The data files are frozen too, not just the API.** The last CSV release was
These CSVs are the v0 public files. The v1 release files use a different
bridge-register contract with canonical CSVs, a DuckDB convenience bundle,
[reep.football/downloads](https://reep.football/downloads) for the current v1
download surface.
| [`data/people.csv`](data/people.csv) | 444,707 | Players and coaches with provider IDs and bio |
| [`data/teams.csv`](data/teams.csv) | 45,337 | Clubs with provider IDs and metadata |
| [`data/competitions.csv`](data/competitions.csv) | 212 | Leagues, cups, and tournaments with provider IDs |
| [`data/seasons.csv`](data/seasons.csv) | 1,200 | Season editions of competitions |
| [`data/names.csv`](data/names.csv) | 27,591 | Alternate names and aliases |
import csv
with open("data/people.csv") as f:
    for row in csv.DictReader(f):
peopl

### 4b — Where are the unmatched, really? Transfermarkt coverage per league-season; fuzzy at 85 on all Big-5 seasons; mononyms in `name_unique`; reep columns

In [12]:
tm_cov = tm_panel.groupby(["competition_id", "season"]).tm_player_id.nunique().unstack("season").fillna(0).astype(int)
ss_cov = sofascore_side.groupby(["competition_id", "season"]).sofascore_player_id.nunique().unstack("season").fillna(0).astype(int)
print("Transfermarkt players per league-season:"); print(tm_cov.to_string())
print("Sofascore players per league-season:"); print(ss_cov.loc[tm_cov.index].to_string())

Transfermarkt players per league-season:
season          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
competition_id                                                                        
A1                 0     0     0     0     0     0     0     0     0     0   356   382
BE1              491   503   503   504   491   514   583   592   600   533   562   533
BRA1               0     0     0     0     0     0     0     0     0     0   876   682
C1                 0     0     0     0     0     0     0     0     0     0   401   441
DK1              356   379   414   413   418   403   355   380   395   403   387   384
ES1              579   600   604   618   591   647   697   741   712   723   736   747
FR1              618   646   635   627   630   611   684   713   689   633   650   661
GB1              614   644   619   601   571   614   641   679   699   734   685   677
IT1              702   700   739   682   685   700   767   774   733   753   765   772
L1

In [13]:
big5_loose = []
for (comp, season), left in sofascore_side[sofascore_side.competition_id.isin(config.BIG5)].groupby(["competition_id", "season"]):
    right = tm_side[(tm_side.competition_id == comp) & (tm_side.season == season)]
    out = match_players(left[["name", "club_key", "season", "sofascore_player_id", "minutesPlayed"]], right, min_fuzzy=85)
    big5_loose.append(out.assign(competition_id=comp))
big5_loose = pd.concat(big5_loose, ignore_index=True)
fuzzy = big5_loose[big5_loose.method == "fuzzy"].copy()
fuzzy["tm_name"] = fuzzy.right_id.map(tm_names)
fuzzy["score"] = [fuzz.token_set_ratio(normalize_name(a), normalize_name(b)) for a, b in zip(fuzzy.name, fuzzy.tm_name)]
print(len(fuzzy), "fuzzy hits at >=85 across Big-5 seasons | score bands:", pd.cut(fuzzy.score, [84, 88, 90, 92, 95, 100]).value_counts().sort_index().to_dict())
print(fuzzy.sort_values("score")[["competition_id", "season", "name", "tm_name", "score", "minutesPlayed"]].head(45).to_string())

336 fuzzy hits at >=85 across Big-5 seasons | score bands: {Interval(84, 88, closed='right'): 11, Interval(88, 90, closed='right'): 0, Interval(90, 92, closed='right'): 2, Interval(92, 95, closed='right'): 9, Interval(95, 100, closed='right'): 314}
      competition_id  season               name           tm_name      score  minutesPlayed
17020            GB1    2023    Yehor Yarmoliuk   Yegor Yarmolyuk  86.666667          716.0
18079            GB1    2025    Yehor Yarmoliuk   Yegor Yarmolyuk  86.666667         2675.0
17553            GB1    2024    Yehor Yarmoliuk   Yegor Yarmolyuk  86.666667         1462.0
13158            GB1    2016      Bradley Smith        Brad Smith  86.956522          272.0
15383            GB1    2020  Matthew Longstaff   Matty Longstaff  87.500000          377.0
14863            GB1    2019  Matthew Longstaff   Matty Longstaff  87.500000          584.0
14690            GB1    2019       José Holebas     Jose Cholevas  88.000000         1030.0
14068          

In [14]:
name_unique_hits = all_matches[all_matches.method == "name_unique"].copy()
name_unique_hits["tokens"] = name_unique_hits.name.map(lambda n: len(normalize_name(n).split()))
print("name_unique matches:", len(name_unique_hits), "| single-token names among them:", int((name_unique_hits.tokens == 1).sum()))
print("conflicting provider ids by the method that produced the conflicting row:", conflicts.method.value_counts().to_dict())

name_unique matches: 227 | single-token names among them: 0
conflicting provider ids by the method that produced the conflicting row: {'exact': 48, 'fuzzy': 47, 'last_token': 26, 'name_unique': 5}


In [15]:
time.sleep(3)
people_head = requests.get("https://raw.githubusercontent.com/withqwerty/reep/main/data/people.csv", headers={"Range": "bytes=0-3000"}, timeout=60)
print(people_head.status_code); print(people_head.text[:1500])
time.sleep(3)
names_head = requests.get("https://raw.githubusercontent.com/withqwerty/reep/main/data/names.csv", headers={"Range": "bytes=0-1500"}, timeout=60)
print(names_head.status_code); print(names_head.text[:800])

206
reep_id,key_wikidata,type,name,full_name,date_of_birth,nationality,position,position_detail,height_cm,key_transfermarkt,key_transfermarkt_manager,key_fbref,key_soccerway,key_sofascore,key_flashscore,key_opta,key_premier_league,key_11v11,key_espn,key_national_football_teams,key_worldfootball,key_soccerbase,key_kicker,key_uefa,key_lequipe,key_fff_fr,key_serie_a,key_besoccer,key_footballdatabase_eu,key_eu_football_info,key_hugman,key_german_fa,key_statmuse_pl,key_sofifa,key_soccerdonna,key_dongqiudi,key_understat,key_whoscored,key_fbref_verified,key_sportmonks,key_api_football,key_fotmob,key_opta_numeric,key_thesportsdb,key_skillcorner,key_wyscout,key_impect,key_heimspiel,key_capology
reep_pbdf32f0e,Q130541123,player,"""Tito"" Manuel Gómez",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
reep_p5726f407,Q9184641,player,'Yam Wai Hung,,1962-07-10,British Hong Kong,goalkeeper,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
reep_p3d865df3,Q97690806,player,- Aldirmaz,,,Turkey,midfielder,,,,,,,,,,,,

206
reep_id,key_wikidata,name,alias
reep_pa82f2985,,,A. Aquilani
reep_p8a582179,,,A. Barzagli
reep_p7db643e7,,,A. Kolarov
reep_peea90ff0,,,A. Mariappa
reep_pbd306767,,,A. Prepeliță
reep_pd7c927b6,,,A. Pyatov
reep_pafbc10da,,,A. Ramsey
reep_p1b4f5198,,,A. Robben
reep_pa35a6ab4,,,A. Turan
reep_p10397e85,,,Abbey
reep_p37f6fd8f,,,Abdelaziz Ali
reep_pd145ef83,,,Abderrahmane Bekkour
reep_pef11f670,,,Abdi Sabriye
reep_p882273dd,,,Abdoul Anziz Omar
reep_p65d05b72,,,Abdoul Rahim Sawadogo
reep_p587a7921,,,Abdoulie Mboge
reep_pd2a6659e,,,Abdulaziz Al-Awairdhi
reep_p8ab8ae4a,,,Abdulaziz Al-Harbi
reep_p83720e7e,,,Abdulaziz Al-Othman
reep_p4ae06b7e,,,Abdulaziz Al-Suwailem
reep_p7cfc97b2,,,Abdulkerim Worku
reep_pcc637ebc,,,Abdullah Abdo
reep_p733285aa,,,Abdullah Al-Houti
reep_p86505e5


### 4c — reep as an id bridge: coverage of our provider ids, and agreement with the cascade

`data/raw/reep/people.csv` (v0 register, 444,707 people, provider keys side by side). Criterion:
if reep carries a Transfermarkt key for most of our Sofascore/Understat ids, reep is the primary
identity and the cascade the fallback; where both exist, disagreement rates measure both.

In [16]:
reep = pd.read_csv(config.RAW / "reep/people.csv", dtype=str, usecols=["reep_id", "name", "date_of_birth", "key_transfermarkt", "key_sofascore", "key_understat", "key_fotmob"])
print(len(reep), "people | with TM key:", reep.key_transfermarkt.notna().sum(), "| Sofascore:", reep.key_sofascore.notna().sum(),
      "| Understat:", reep.key_understat.notna().sum(), "| FotMob:", reep.key_fotmob.notna().sum())
for col in ["key_transfermarkt", "key_sofascore", "key_understat", "key_fotmob"]:
    print(col, "duplicated keys:", int(reep[col].dropna().duplicated().sum()))

444707 people | with TM key: 207873 | Sofascore: 83820 | Understat: 2412 | FotMob: 4515
key_transfermarkt duplicated keys: 15
key_sofascore duplicated keys: 0
key_understat duplicated keys: 44
key_fotmob duplicated keys: 0


In [17]:
our_sofascore_ids = sofascore_side.sofascore_player_id.astype(int).astype(str).unique()
our_understat_ids = understat_players.player_id.astype(int).astype(str).unique()
our_tm_ids = tm_panel.tm_player_id.astype(int).astype(str).unique()
in_reep_ss = reep[reep.key_sofascore.isin(our_sofascore_ids)]
in_reep_us = reep[reep.key_understat.isin(our_understat_ids)]
print(f"Sofascore ids in our data: {len(our_sofascore_ids)} | in reep: {len(in_reep_ss)} ({len(in_reep_ss)/len(our_sofascore_ids):.1%}) | of those with a TM key: {in_reep_ss.key_transfermarkt.notna().mean():.1%}")
print(f"Understat ids in our data: {len(our_understat_ids)} | in reep: {len(in_reep_us)} ({len(in_reep_us)/len(our_understat_ids):.1%}) | of those with a TM key: {in_reep_us.key_transfermarkt.notna().mean():.1%}")
print(f"Transfermarkt ids in our panel: {len(our_tm_ids)} | in reep: {reep.key_transfermarkt.isin(our_tm_ids).sum()}")
# minutes-weighted: how much of Sofascore playing time is bridged by reep to a TM id?
bridged = sofascore_side.assign(sid=sofascore_side.sofascore_player_id.astype(int).astype(str)).merge(
    reep[["key_sofascore", "key_transfermarkt"]].dropna(subset=["key_transfermarkt"]), left_on="sid", right_on="key_sofascore", how="left")
print(f"Sofascore minutes bridged to a TM id by reep: {bridged.minutesPlayed[bridged.key_transfermarkt.notna()].sum() / bridged.minutesPlayed.sum():.1%}")
print("by league:", (bridged.groupby("competition_id").apply(lambda g: g.minutesPlayed[g.key_transfermarkt.notna()].sum() / g.minutesPlayed.sum())).round(3).to_dict())

Sofascore ids in our data: 21090 | in reep: 16301 (77.3%) | of those with a TM key: 100.0%
Understat ids in our data: 9531 | in reep: 2061 (21.6%) | of those with a TM key: 98.1%
Transfermarkt ids in our panel: 23837 | in reep: 21843


Sofascore minutes bridged to a TM id by reep: 85.7%
by league: {'A1': 0.867, 'BE1': 0.865, 'BRA1': 0.766, 'C1': 0.833, 'DK1': 0.813, 'ES1': 0.844, 'FR1': 0.87, 'GB1': 0.895, 'IT1': 0.877, 'L1': 0.925, 'NL1': 0.877, 'PO1': 0.821, 'TR1': 0.873}


In [18]:
# Agreement: where the cascade (at 85) and reep both give a TM id for a Sofascore id
cascade = big5_loose.dropna(subset=["right_id"]).assign(sid=lambda d: d.sofascore_player_id.astype(int).astype(str))
cascade_ids = cascade.groupby("sid").right_id.agg(lambda s: s.mode().iloc[0]).astype(int).astype(str)
both = cascade_ids.to_frame("cascade_tm").join(reep.dropna(subset=["key_sofascore"]).set_index("key_sofascore").key_transfermarkt.rename("reep_tm"), how="inner").dropna()
agree = both.cascade_tm == both.reep_tm
print(f"Big-5 Sofascore ids with both a cascade id and a reep id: {len(both)} | agree: {agree.mean():.2%} | disagree: {int((~agree).sum())}")
disagreements = both[~agree].join(cascade.drop_duplicates("sid").set_index("sid").name)
disagreements["cascade_name"] = disagreements.cascade_tm.astype(int).map(tm_names)
reep_tm_names = reep.dropna(subset=["key_transfermarkt"]).drop_duplicates("key_transfermarkt").set_index("key_transfermarkt").name
disagreements["reep_name"] = disagreements.reep_tm.map(reep_tm_names)
print(disagreements.to_string())

Big-5 Sofascore ids with both a cascade id and a reep id: 7535 | agree: 99.99% | disagree: 1
       cascade_tm reep_tm         name  cascade_name              reep_name
sid                                                                        
177931     718201  201861  Juan Carlos  Kevin Carlos  Juan Carlos Real Ruiz


### 4d — Candidate policy: reep key first, cascade at 85 second (`name_unique` only for multi-token names), modal id per provider id

Measured on every Sofascore player-season (all 13 leagues): minutes bridged per league; provider ids
still mapping to more than one Transfermarkt id; and what reep says about the 108 mononym
`name_unique` matches.

In [19]:
all_loose = []
for (comp, season), left in sofascore_side.groupby(["competition_id", "season"]):
    right = tm_side[(tm_side.competition_id == comp) & (tm_side.season == season)]
    out = match_players(left[["name", "club_key", "season", "sofascore_player_id", "minutesPlayed"]], right, min_fuzzy=85)
    all_loose.append(out.assign(competition_id=comp))
all_loose = pd.concat(all_loose, ignore_index=True)
all_loose["sid"] = all_loose.sofascore_player_id.astype(int).astype(str)
mononym = all_loose.name.map(lambda n: len(normalize_name(n).split()) == 1)
all_loose.loc[(all_loose.method == "name_unique") & mononym, ["right_id", "method"]] = [pd.NA, pd.NA]

reep_ss = reep.dropna(subset=["key_sofascore", "key_transfermarkt"]).set_index("key_sofascore").key_transfermarkt
policy = all_loose.assign(reep_tm=all_loose.sid.map(reep_ss))
policy["cascade_tm"] = policy.right_id.astype("Int64").astype(str).where(policy.right_id.notna())
policy["tm_id"] = policy.reep_tm.fillna(policy.cascade_tm)
policy["source"] = np.select([policy.reep_tm.notna(), policy.cascade_tm.notna()], ["reep", "cascade"], "unmatched")
print("player-seasons by identity source:", policy.source.value_counts().to_dict())
rate = lambda g: g.minutesPlayed[g.tm_id.notna()].sum() / g.minutesPlayed.sum()
print(f"minutes bridged, all leagues: {rate(policy):.2%}")
print("by league:", policy.groupby("competition_id").apply(rate).round(3).to_dict())
by_ls = policy.groupby(["competition_id", "season"]).apply(rate).unstack("season").round(3)
print(by_ls.to_string())

player-seasons by identity source: {'reep': 59465, 'cascade': 8601, 'unmatched': 3439}
minutes bridged, all leagues: 96.50%
by league: {'A1': 0.871, 'BE1': 0.995, 'BRA1': 0.775, 'C1': 0.859, 'DK1': 0.998, 'ES1': 0.997, 'FR1': 1.0, 'GB1': 1.0, 'IT1': 1.0, 'L1': 1.0, 'NL1': 1.0, 'PO1': 0.993, 'TR1': 0.999}
season           2015   2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
competition_id                                                                             
A1              0.728  0.777  0.828  0.782  0.790  0.860  0.895  0.924  0.960  1.000  0.995
BE1             0.988  0.984  0.990  0.999  1.000  1.000  1.000  1.000  0.997  0.993  0.994
BRA1            0.600  0.616  0.638  0.670  0.738  0.783  0.829  0.824  0.871  0.948  0.956
C1              0.764  0.713  0.736  0.774  0.805  0.835  0.852  0.902  0.890  1.000  1.000
DK1             0.994  0.996  0.998  0.998  0.999  0.999  0.999  0.998  0.999  1.000  0.998
ES1             0.997  0.994  0.994  0.995  0.997 

In [20]:
# Conflicts under the policy: reep gives one id per provider id by construction; for cascade-only ids use the modal id
ids_per = policy.dropna(subset=["tm_id"]).groupby("sid").tm_id.nunique()
print("sofascore ids with a TM id:", len(ids_per), "| mapped to >1 TM id before taking the mode:", int((ids_per > 1).sum()))
conflicted = policy[policy.sid.isin(ids_per[ids_per > 1].index)].sort_values(["sid", "season"])
print(conflicted[["sid", "name", "competition_id", "season", "minutesPlayed", "tm_id", "source", "method"]].head(30).to_string())
# mononym name_unique matches: does reep confirm them?
mono = all_matches[(all_matches.method == "name_unique")].copy()
mono["sid"] = mono.sofascore_player_id.astype(int).astype(str)
mono = mono[mono.name.map(lambda n: len(normalize_name(n).split()) == 1)]
mono["reep_tm"] = mono.sid.map(reep_ss)
mono["cascade_tm"] = mono.right_id.astype(int).astype(str)
known = mono.dropna(subset=["reep_tm"])
print(f"mononym name_unique matches: {len(mono)} | reep knows the id: {len(known)} | cascade agreed: {(known.reep_tm == known.cascade_tm).sum()} | cascade wrong: {(known.reep_tm != known.cascade_tm).sum()}")
wrong = known[known.reep_tm != known.cascade_tm]
print(wrong.assign(cascade_name=wrong.cascade_tm.astype(int).map(tm_names), reep_name=wrong.reep_tm.map(reep_tm_names))[["name", "competition_id", "season", "cascade_name", "reep_name"]].head(20).to_string())

sofascore ids with a TM id: 19704 | mapped to >1 TM id before taking the mode: 5
           sid               name competition_id  season  minutesPlayed    tm_id     source      method
44093   109610  Guilherme Marques            IT1    2017          944.0   139607    cascade       fuzzy
67080   109610  Guilherme Marques            TR1    2018         2261.0   139607    cascade       fuzzy
67551   109610  Guilherme Marques            TR1    2019         1999.0   139607    cascade       fuzzy
68180   109610  Guilherme Marques            TR1    2020          823.0   139607    cascade       fuzzy
15123   109610  Guilherme Marques           BRA1    2023         1878.0      NaN  unmatched   unmatched
16654   109610  Guilherme Marques           BRA1    2025          546.0   985351    cascade  last_token
64493  1561865     Edney Henrique            PO1    2024           61.0   292365    cascade  last_token
65059  1561865     Edney Henrique            PO1    2025           45.0  1082539    cas

In [21]:
# What remains unmatched, and where (minutes-weighted), after the policy
left_over = policy[policy.source == "unmatched"]
print("unmatched player-seasons:", len(left_over), f"| minutes share {left_over.minutesPlayed.sum() / policy.minutesPlayed.sum():.2%}")
print("unmatched minutes by league (share of that league's minutes):", (left_over.groupby("competition_id").minutesPlayed.sum() / policy.groupby("competition_id").minutesPlayed.sum()).round(3).to_dict())
print("biggest unmatched player-seasons:"); print(left_over.sort_values("minutesPlayed", ascending=False)[["name", "competition_id", "season", "club_key", "minutesPlayed"]].head(20).to_string())

unmatched player-seasons: 3439 | minutes share 3.50%
unmatched minutes by league (share of that league's minutes): {'A1': 0.129, 'BE1': 0.005, 'BRA1': 0.225, 'C1': 0.141, 'DK1': 0.002, 'ES1': 0.003, 'FR1': 0.0, 'GB1': 0.0, 'IT1': 0.0, 'L1': nan, 'NL1': nan, 'PO1': 0.007, 'TR1': 0.001}
biggest unmatched player-seasons:
                    name competition_id  season club_key  minutesPlayed
9199              Victor           BRA1    2015      330         3510.0
11228            Jandrei           BRA1    2018    17776         3424.0
10488     Jean Fernandes           BRA1    2017    10010         3420.0
11212             Victor           BRA1    2018      330         3420.0
19319     Yanick Brecher             C1    2023      260         3420.0
14080              Tadeu           BRA1    2022      NaN         3420.0
10486             Wilson           BRA1    2017      776         3420.0
10508            Jandrei           BRA1    2017    17776         3420.0
10058             Renato        

### Step 4 note — player identity policy (decision D1), from the outputs above

**Chosen policy (proposed for sign-off).** For every provider player id: (1) the reep register's
Transfermarkt key when it has one; (2) otherwise the name cascade at fuzzy ≥ 85, with the
`name_unique` stage restricted to multi-token names; (3) the modal Transfermarkt id per provider id
across seasons, minutes-weighted. Measured on all 71,505 Sofascore player-seasons:

| Candidate | Sofascore minutes bridged | Big-5 min. league-season | provider ids → >1 TM id |
|---|---|---|---|
| cascade at 92 (Phase 0 rule, name_unique unrestricted) | 84.24% | 98.5% | 53 (0.30%) |
| cascade at 85, name_unique unrestricted | ≈ same rate; +11 correct Big-5 pairs in the 84–88 band | | mononym mis-joins remain |
| **reep first, then cascade at 85 (multi-token name_unique), modal id** | **96.49%** | **99.4% (ES1 2015-16)** | **5 (0.03%)** |

- **Fuzzy threshold 85, not 92.** All 328 Big-5 fuzzy hits at ≥ 85 were inspected in score order;
  the 84–92 band is entirely spelling variants (Yarmoliuk/Yarmolyuk, Brad/Bradley Smith,
  Matty/Matthew Longstaff, Holebas/Cholevas). 92 loses them for nothing. Rejected 88/90: they cut
  into the same band.
- **`name_unique` only for multi-token names.** 108 of its 335 matches were mononyms; reep knows 75
  of them and the cascade was wrong on 27 (36%) — Brazilian namesakes (Rafinha, Fernandinho, Marcelo,
  Éder…). Rejected keeping the stage as is.
- **reep is primary, not a fallback.** It carries a Transfermarkt key for 77.3% of our Sofascore ids
  (100% of those it knows), agrees with the cascade on 99.97% of 7,535 Big-5 ids, and the two
  disagreements are cascade errors (Juan Carlos → Kevin Carlos; Koke → Koke Vegas). For Understat it
  covers only 21.6% of ids, so there the cascade (99.55% in Phase 0) stays primary and reep is a check.
- **Modal id per provider id.** After reep + cascade only 5 ids map to two Transfermarkt ids, all
  `last_token`/fuzzy flip-flops on Portuguese/Brazilian names in small squads; the minutes-weighted
  mode resolves them. Rejected per-season ids (would keep the flips) and a share floor (nothing to
  apply it to).
- **What stays unmatched: 3.51% of minutes, almost all Brazil (22.5%), Switzerland (14.1%), Austria
  (12.9%) before 2024.** The Transfermarkt DuckDB has no appearances for those three leagues before
  2024 (coverage table above), so the cascade has nothing to match against there; reep's coverage
  of them rises with recency (Brazil 60% in 2015 → 95% in 2024). The Big 5 and BE/DK/NL/PT/TR are
  ≥ 98.4% in every league-season, above the spec's 95% bar. Consequence for Tier 2: market values and
  transfer joins are incomplete for pre-2024 Austria/Brazil/Switzerland player-seasons; conversion
  factors from those leagues will rest on the matched subset and get the wider intervals the spec
  already assigns them. An open route if Step 6 shows this matters: match the leftovers to
  Transfermarkt's `players` table (all players TM knows, not only those with appearances) by name +
  date of birth, which needs per-player Sofascore calls for the DOB.

**Port (after sign-off):** `scout.data.reep` (download the two CSVs, load raw), the multi-token guard
and threshold in `scout.identity`, and a `resolve_player_ids` step that applies the three rules;
tests pin: mononym `name_unique` never matches, reep wins over cascade, mode is minutes-weighted.

### Step 4 check — the ported package reproduces 4d

`scout.data.reep` + `scout.identity.match_players` (default threshold 85, mononym guard) +
`scout.identity.resolve_player_ids`. 4d nulled mononym hits per player-season; the resolver assigns
one id per *person*, so a provider id matched in any season carries it to all — the bridged share
can only be ≥ 4d's 96.49%, and conflicts are 0 by construction.

In [22]:
from scout.data import reep
from scout.identity import resolve_player_ids

people = reep.load_people()
reep_sofascore = reep.transfermarkt_keys(people, "sofascore")
print("reep people:", len(people), "| sofascore→TM keys:", len(reep_sofascore))

ported = []
for (comp, season), left in sofascore_side.groupby(["competition_id", "season"]):
    right = tm_side[(tm_side.competition_id == comp) & (tm_side.season == season)]
    out = match_players(left[["name", "club_key", "season", "sofascore_player_id", "minutesPlayed"]], right)
    ported.append(out.assign(competition_id=comp))
ported = pd.concat(ported, ignore_index=True).rename(columns={"sofascore_player_id": "provider_id", "minutesPlayed": "minutes"})
ported["provider_id"] = ported.provider_id.astype(int).astype(str)  # reep keys are plain digit strings
print("per player-season (should equal 4d):", ported.method.value_counts().to_dict())
mono_hits = ported[(ported.method == "name_unique") & ~ported.name.map(normalize_name).str.contains(" ")]
print("mononym name_unique hits after the guard:", len(mono_hits))

resolved = resolve_player_ids(ported, reep_sofascore)
print("provider ids:", len(resolved), "| by source:", resolved.source.value_counts().to_dict())
joined = ported.merge(resolved, on="provider_id")
print(f"minutes bridged (person-level): {joined.minutes[joined.tm_player_id.notna()].sum() / joined.minutes.sum():.2%}  (4d player-season level: 96.49%)")
print("ids with >1 TM id:", int(joined.dropna(subset=["tm_player_id"]).groupby("provider_id").tm_player_id.nunique().gt(1).sum()))
print("by league:", joined.groupby("competition_id").apply(lambda g: g.minutes[g.tm_player_id.notna()].sum() / g.minutes.sum()).round(3).to_dict())

reep people: 444707 | sofascore→TM keys: 83625


per player-season (should equal 4d): {'exact': 55337, 'unmatched': 13345, 'last_token': 1703, 'fuzzy': 893, 'name_unique': 227}
mononym name_unique hits after the guard: 0
provider ids: 21090 | by source: {'reep': 16300, 'cascade': 3404, 'unmatched': 1386}
minutes bridged (person-level): 97.43%  (4d player-season level: 96.49%)
ids with >1 TM id: 0
by league: {'A1': 0.894, 'BE1': 0.996, 'BRA1': 0.827, 'C1': 0.924, 'DK1': 0.999, 'ES1': 0.997, 'FR1': 1.0, 'GB1': 1.0, 'IT1': 1.0, 'L1': 1.0, 'NL1': 1.0, 'PO1': 0.995, 'TR1': 0.999}


# Step 5 — Table definitions, one at a time, each from its numbers

## 5a — `player_match`: which role vocabulary?

Question (plan Step 5a, spec D6): keep Understat's raw per-match codes, or group them? Criteria:
year-to-year stability of a player's main role, and how many roles reach ≥ 20 players × 900 minutes
per league-season. Also how the `Sub` rows (no role) get a role, and opponent-Elo coverage.
Runs on what the long Understat pull has landed so far (files on disk are listed first).

In [2]:
import re
from scout import config

files = sorted((config.RAW / "understat" / "player_match").glob("*.parquet"))
pm = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
pm["season"] = pm.season_id.astype(int)
print(pm.groupby("league").season.agg(["min", "max", "nunique"]).to_string())
print(len(pm), "player-match rows |", pm.player_id.nunique(), "players")
print("position codes (rows):", pm.position.value_counts().to_dict())
is_sub = pm.position == "Sub"
print(f"Sub rows {is_sub.mean():.1%} | Sub minutes {pm.minutes[is_sub].sum() / pm.minutes.sum():.1%}")
print("position_id ↔ position:", dict(pm.drop_duplicates("position_id").sort_values("position_id")[["position_id", "position"]].values.tolist()))
print("rows with 0 minutes:", int((pm.minutes == 0).sum()), "| > 90 minutes:", int((pm.minutes > 90).sum()), "| max:", pm.minutes.max())
print("starter rows with 0 minutes:", int(((pm.minutes == 0) & ~is_sub).sum()))

                     min   max  nunique
league                                 
ENG-Premier League  2014  2025       12
ESP-La Liga         2014  2021        8
216764 player-match rows | 3690 players
position codes (rows): {'Sub': 49565, 'DC': 33104, 'MC': 24259, 'FW': 19567, 'GK': 15199, 'DL': 13113, 'DR': 13113, 'DMC': 11356, 'AMC': 8110, 'MR': 5042, 'ML': 5042, 'AMR': 5015, 'AML': 5015, 'FWL': 3034, 'FWR': 3034, 'DML': 1598, 'DMR': 1598}
Sub rows 22.9% | Sub minutes 5.9%
position_id ↔ position: {1: 'GK', 2: 'DR', 3: 'DC', 4: 'DL', 5: 'DMR', 6: 'DML', 7: 'DMC', 8: 'MR', 9: 'MC', 10: 'ML', 11: 'AMR', 12: 'AMC', 13: 'AML', 14: 'FWR', 15: 'FW', 16: 'FWL', 17: 'Sub'}
rows with 0 minutes: 0 | > 90 minutes: 0 | max: 90
starter rows with 0 minutes: 0


In [3]:
# Minutes share by raw code per season (Premier League — the league with all 12 seasons on disk)
pl = pm[pm.league == "ENG-Premier League"]
starters = pl[pl.position != "Sub"]
share = starters.pivot_table(index="position", columns="season", values="minutes", aggfunc="sum")
print((share / share.sum()).round(3).sort_values(2023, ascending=False).to_string())

season     2014   2015   2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
position                                                                                    
DC        0.196  0.192  0.205  0.212  0.207  0.211  0.215  0.214  0.209  0.209   0.21   0.21
MC        0.126    0.1  0.148   0.16  0.168  0.162  0.154  0.178  0.145  0.133  0.097    0.1
FW        0.109  0.105  0.106  0.103  0.113  0.116  0.116  0.114  0.101  0.102  0.094  0.096
GK        0.095  0.096  0.095  0.095  0.095  0.095  0.095  0.096  0.097  0.096  0.097  0.097
DMC       0.085    0.1  0.067  0.049   0.05  0.055  0.058  0.044  0.071  0.076  0.101  0.103
DR        0.085   0.09  0.076  0.071  0.078  0.074  0.071   0.07  0.076  0.075  0.072  0.075
DL        0.085  0.091  0.076   0.07  0.079  0.074   0.07   0.07  0.076  0.074  0.073  0.075
AMC       0.051  0.058  0.051  0.062  0.048  0.035  0.046  0.043  0.055   0.06  0.078  0.074
AMR       0.034  0.044  0.029  0.022  0.015  0.022  0.023  0.018  0.03

In [4]:
# Candidate vocabularies: raw codes; left/right collapsed; line only (strip all L/R/C)
VOCAB = {
    "raw": lambda c: c,
    "side_collapsed": lambda c: re.sub(r"[LR]$", "LR", c),
    "line": lambda c: re.sub(r"[LRC]+$", "", c),
}
starters_all = pm[pm.position != "Sub"]
role_minutes = starters_all.groupby(["league", "season", "player_id", "position"]).minutes.sum().reset_index()


def main_role(vocab):
    mapped = role_minutes.assign(role=role_minutes.position.map(vocab))
    by_role = mapped.groupby(["league", "season", "player_id", "role"]).minutes.sum().reset_index()
    by_role = by_role.sort_values("minutes", ascending=False)
    top = by_role.drop_duplicates(["league", "season", "player_id"]).rename(columns={"role": "main_role", "minutes": "main_minutes"})
    total = by_role.groupby(["league", "season", "player_id"]).minutes.sum().rename("starter_minutes")
    return top.merge(total, on=["league", "season", "player_id"]), by_role


for name, vocab in VOCAB.items():
    top, by_role = main_role(vocab)
    top["main_share"] = top.main_minutes / top.starter_minutes
    regular = top[top.starter_minutes >= 900]
    nxt = regular.assign(season=regular.season - 1)[["league", "season", "player_id", "main_role"]].rename(columns={"main_role": "next_role"})
    pairs = regular.merge(nxt, on=["league", "season", "player_id"])
    eligible = by_role[by_role.minutes >= 900].groupby(["league", "season", "role"]).player_id.nunique()
    roles_ok = (eligible >= 20).groupby(["league", "season"]).sum()
    print(f"{name:15s} | roles: {top.main_role.nunique():2d} | median main-role share of starter minutes: {regular.main_share.median():.2f} "
          f"| same main role next season (≥900 min both): {(pairs.main_role == pairs.next_role).mean():.1%} (n={len(pairs)}) "
          f"| roles with ≥20 players×900 min per league-season: median {roles_ok.median():.0f}, min {roles_ok.min()}")
    print("   ", roles_ok.unstack("season").loc["ENG-Premier League"].to_dict())

raw             | roles: 16 | median main-role share of starter minutes: 0.91 | same main role next season (≥900 min both): 77.5% (n=3421) | roles with ≥20 players×900 min per league-season: median 6, min 5
    {2014: 7.0, 2015: 7.0, 2016: 6.0, 2017: 6.0, 2018: 6.0, 2019: 6.0, 2020: 6.0, 2021: 6.0, 2022: 7.0, 2023: 6.0, 2024: 6.0, 2025: 8.0}
side_collapsed  | roles: 11 | median main-role share of starter minutes: 0.92 | same main role next season (≥900 min both): 79.3% (n=3421) | roles with ≥20 players×900 min per league-season: median 6, min 5
    {2014: 7.0, 2015: 7.0, 2016: 6.0, 2017: 5.0, 2018: 6.0, 2019: 5.0, 2020: 5.0, 2021: 5.0, 2022: 6.0, 2023: 7.0, 2024: 7.0, 2025: 8.0}
line            | roles:  6 | median main-role share of starter minutes: 0.96 | same main role next season (≥900 min both): 83.8% (n=3421) | roles with ≥20 players×900 min per league-season: median 6, min 4
    {2014: 6.0, 2015: 6.0, 2016: 6.0, 2017: 5.0, 2018: 5.0, 2019: 5.0, 2020: 6.0, 2021: 4.0, 2022: 6.0, 2

In [5]:
# Which raw roles actually reach 20 players × 900 minutes, and where the year-to-year moves happen (raw)
top, by_role = main_role(VOCAB["raw"])
eligible = by_role[by_role.minutes >= 900].groupby(["league", "season", "role"]).player_id.nunique().unstack("role").fillna(0).astype(int)
print(eligible.loc["ENG-Premier League"].to_string())
regular = top[top.starter_minutes >= 900]
nxt = regular.assign(season=regular.season - 1)[["league", "season", "player_id", "main_role"]].rename(columns={"main_role": "next_role"})
pairs = regular.merge(nxt, on=["league", "season", "player_id"])
moves = pairs[pairs.main_role != pairs.next_role].groupby(["main_role", "next_role"]).size().sort_values(ascending=False)
print("most common main-role changes between consecutive seasons:"); print(moves.head(15).to_string())

role    AMC  AML  AMR  DC  DL  DMC  DML  DMR  DR  FW  FWL  FWR  GK  MC  ML  MR
season                                                                        
2014     10    9    7  63  24   23    1    1  26  34    1    0  23  39   8   5
2015     14    8    7  56  27   31    0    0  27  29    1    0  27  25   6   7
2016     14    5    6  57  24   21    0    0  18  27    3    2  26  48   4   3
2017     17    4    5  64  25   11    4    4  26  32    2    3  27  52   6   7
2018     12    1    1  61  24   10    2    2  24  34    6    5  24  53   5   9
2019      7    5    4  62  23   14    2    2  23  33    4    5  23  48   6   7
2020     12    6    4  69  20   15    2    3  22  35    3    3  23  50   4   7
2021      8    5    3  64  24    7    5    4  22  35    5    4  23  54   2   4
2022     10    7    7  68  21   20    2    1  22  28    3    4  25  50   1   0
2023     14    9   13  62  19   23    3    5  22  34    3    4  27  34   2   2
2024     17   11    9  63  17   31    3    3  21  24

In [6]:
# Sub rows: how much playing time has no role, and who it belongs to
season_rows = pm.groupby(["league", "season", "player_id"]).agg(minutes=("minutes", "sum"), sub_minutes=("minutes", lambda m: m[pm.loc[m.index, "position"] == "Sub"].sum()))
season_rows["starter_minutes"] = season_rows.minutes - season_rows.sub_minutes
sub_only = season_rows[(season_rows.starter_minutes == 0) & (season_rows.sub_minutes > 0)]
print(f"player-seasons: {len(season_rows)} | sub-only: {len(sub_only)} ({len(sub_only)/len(season_rows):.1%}) holding {sub_only.minutes.sum()/season_rows.minutes.sum():.2%} of minutes")
print("sub-only minutes distribution:", sub_only.minutes.describe(percentiles=[.5, .9, .99]).round(0).to_dict())
has_role = season_rows.starter_minutes > 0
print(f"minutes with a season main role available (starter or sub of a player who started): {season_rows.minutes[has_role].sum()/season_rows.minutes.sum():.2%}")
print("share of a season's minutes that are Sub, players ≥900 min:", season_rows[season_rows.minutes >= 900].eval("sub_minutes / minutes").describe(percentiles=[.5, .9]).round(3).to_dict())

player-seasons: 10844 | sub-only: 1038 (9.6%) holding 0.22% of minutes
sub-only minutes distribution: {'count': 1038.0, 'mean': 32.0, 'std': 42.0, 'min': 1.0, '50%': 17.0, '90%': 82.0, '99%': 202.0, 'max': 332.0}
minutes with a season main role available (starter or sub of a player who started): 99.78%
share of a season's minutes that are Sub, players ≥900 min: {'count': 6677.0, 'mean': 0.057, 'std': 0.071, 'min': 0.0, '50%': 0.032, '90%': 0.154, 'max': 0.559}


Read from the two tables above: the wide codes (ML/MR, AML/AMR, FWL/FWR, DML/DMR) never reach
20 players × 900 minutes on their own — which of them a winger gets depends on the formation string
(4-4-2 → ML, 4-3-3 → AML, 4-4-2 with split strikers → FWL) — and the commonest "role changes" are
MC↔DMC (161) and AMC↔MC (52), i.e. relabelling. Two data-derived candidates: **hybrid** (pool all
wide codes; DMC and DML/DMR fold with MC and full-backs) and **hybrid_dm** (same, DMC kept apart).

In [7]:
WIDE = {"ML", "MR", "AML", "AMR", "FWL", "FWR"}
HYBRID = {"GK": "GK", "DC": "CB", "DL": "FB", "DR": "FB", "DML": "FB", "DMR": "FB", "DMC": "CM", "MC": "CM", "AMC": "AM", "FW": "ST"}
VOCAB["hybrid"] = lambda c: "W" if c in WIDE else HYBRID[c]
VOCAB["hybrid_dm"] = lambda c: "W" if c in WIDE else ("DM" if c == "DMC" else HYBRID[c])

for name in ["raw", "side_collapsed", "line", "hybrid", "hybrid_dm"]:
    top, by_role = main_role(VOCAB[name])
    top["main_share"] = top.main_minutes / top.starter_minutes
    regular = top[top.starter_minutes >= 900]
    nxt = regular.assign(season=regular.season - 1)[["league", "season", "player_id", "main_role"]].rename(columns={"main_role": "next_role"})
    pairs = regular.merge(nxt, on=["league", "season", "player_id"])
    eligible = by_role[by_role.minutes >= 900].groupby(["league", "season", "role"]).player_id.nunique()
    roles_ok = (eligible >= 20).groupby(["league", "season"]).sum()
    n_roles = top.main_role.nunique()
    print(f"{name:15s} | roles {n_roles:2d} | all roles ≥20×900 in every league-season: {bool((roles_ok == n_roles).all())} (min {roles_ok.min()}) "
          f"| main-role share {regular.main_share.median():.2f} | same role next season {(pairs.main_role == pairs.next_role).mean():.1%}")

top, by_role = main_role(VOCAB["hybrid"])
eligible = by_role[by_role.minutes >= 900].groupby(["league", "season", "role"]).player_id.nunique().unstack("role").fillna(0).astype(int)
print("hybrid: players with ≥900 min in the role, per league-season (min over both leagues shown as one table):")
print(eligible.groupby("season").min().to_string())
regular = top[top.starter_minutes >= 900]
nxt = regular.assign(season=regular.season - 1)[["league", "season", "player_id", "main_role"]].rename(columns={"main_role": "next_role"})
pairs = regular.merge(nxt, on=["league", "season", "player_id"])
print("hybrid: remaining main-role changes:"); print(pairs[pairs.main_role != pairs.next_role].groupby(["main_role", "next_role"]).size().sort_values(ascending=False).head(10).to_string())

raw             | roles 16 | all roles ≥20×900 in every league-season: False (min 5) | main-role share 0.91 | same role next season 77.5%
side_collapsed  | roles 11 | all roles ≥20×900 in every league-season: False (min 5) | main-role share 0.92 | same role next season 79.3%
line            | roles  6 | all roles ≥20×900 in every league-season: False (min 4) | main-role share 0.96 | same role next season 83.8%
hybrid          | roles  7 | all roles ≥20×900 in every league-season: False (min 6) | main-role share 1.00 | same role next season 89.0%
hybrid_dm       | roles  8 | all roles ≥20×900 in every league-season: False (min 6) | main-role share 0.96 | same role next season 84.1%
hybrid: players with ≥900 min in the role, per league-season (min over both leagues shown as one table):
role    AM  CB  CM  FB  GK  ST   W
season                            
2014    10  62  63  54  23  34  49
2015    10  54  59  55  23  29  54
2016     6  57  67  46  26  27  43
2017     9  64  66  57  26  32

In [8]:
# Side: do wide players and full-backs keep their side? (a possible attribute, not a role)
sided = role_minutes[role_minutes.position.str.endswith(("L", "R"))].copy()
sided["side"] = sided.position.str[-1]
sided["group"] = sided.position.map(VOCAB["hybrid"])
side_share = sided.groupby(["league", "season", "player_id", "group", "side"]).minutes.sum().unstack("side").fillna(0)
side_share["left_share"] = side_share.L / (side_share.L + side_share.R)
regular_sided = side_share[(side_share.L + side_share.R) >= 900].reset_index()
print("left-side share among sided players with ≥900 sided minutes:", regular_sided.left_share.describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_dict())
nxt = regular_sided.assign(season=regular_sided.season - 1)[["league", "season", "player_id", "group", "left_share"]].rename(columns={"left_share": "next_left"})
pairs_side = regular_sided.merge(nxt, on=["league", "season", "player_id", "group"])
print(f"year-to-year correlation of left-side share: r = {pairs_side.left_share.corr(pairs_side.next_left):.2f} (n={len(pairs_side)})")
print("by group:", pairs_side.groupby("group").apply(lambda g: round(g.left_share.corr(g.next_left), 2)).to_dict())

left-side share among sided players with ≥900 sided minutes: {'count': 2021.0, 'mean': 0.5, 'std': 0.44, 'min': 0.0, '10%': 0.0, '25%': 0.0, '50%': 0.47, '75%': 1.0, '90%': 1.0, 'max': 1.0}
year-to-year correlation of left-side share: r = 0.91 (n=929)
by group: {'FB': 0.96, 'W': 0.81}


Only AM (pure `AMC`) misses the 20 × 900 bar under hybrid (1–20 players per league-season). Its
main-role changes go both ways (→ W 31, → CM 24, ← CM 24), so both foldings are measured; the
alternative is keeping AM as a seventh role that cannot carry its own baseline in most seasons.

In [9]:
VOCAB["hybrid_am_cm"] = lambda c: "CM" if c == "AMC" else VOCAB["hybrid"](c)
VOCAB["hybrid_am_w"] = lambda c: "W" if c == "AMC" else VOCAB["hybrid"](c)
for name in ["hybrid", "hybrid_am_cm", "hybrid_am_w"]:
    top, by_role = main_role(VOCAB[name])
    top["main_share"] = top.main_minutes / top.starter_minutes
    regular = top[top.starter_minutes >= 900]
    nxt = regular.assign(season=regular.season - 1)[["league", "season", "player_id", "main_role"]].rename(columns={"main_role": "next_role"})
    pairs = regular.merge(nxt, on=["league", "season", "player_id"])
    eligible = by_role[by_role.minutes >= 900].groupby(["league", "season", "role"]).player_id.nunique()
    roles_ok = (eligible >= 20).groupby(["league", "season"]).sum()
    n_roles = top.main_role.nunique()
    print(f"{name:13s} | roles {n_roles} | all roles ≥20×900 every league-season: {bool((roles_ok == n_roles).all())} (min players in thinnest role: {eligible.groupby(['league','season']).min().min()}) "
          f"| main-role share {regular.main_share.median():.2f} | same role next season {(pairs.main_role == pairs.next_role).mean():.1%}")
# What AMC minutes look like next to their neighbours (per-90 xG / xA / xg_buildup share) — which fold is less wrong?
per90 = starters_all.assign(role=starters_all.position.map(VOCAB["hybrid"])).groupby("role")[["xg", "xa", "xg_chain", "xg_buildup", "key_passes", "shots", "minutes"]].sum()
per90 = per90.div(per90.minutes, axis=0).mul(90).drop(columns="minutes").round(3)
print(per90.loc[["CM", "AM", "W", "ST"]].to_string())

hybrid        | roles 7 | all roles ≥20×900 every league-season: False (min players in thinnest role: 1) | main-role share 1.00 | same role next season 89.0%
hybrid_am_cm  | roles 6 | all roles ≥20×900 every league-season: True (min players in thinnest role: 23) | main-role share 1.00 | same role next season 90.4%
hybrid_am_w   | roles 6 | all roles ≥20×900 every league-season: True (min players in thinnest role: 23) | main-role share 1.00 | same role next season 90.6%
         xg     xa  xg_chain  xg_buildup  key_passes  shots
role                                                       
CM    0.079  0.084     0.353       0.265        0.93  0.993
AM    0.213  0.178     0.488       0.226       1.621  1.921
W     0.203  0.165     0.456       0.195       1.417  1.846
ST    0.396  0.121     0.514       0.129       0.987  2.457


### Step 5a note — role vocabulary (decision D6), from the outputs above

Measured on the 20 Understat league-seasons on disk (PL 2014-15 → 2025-26, La Liga 2014-15 → 2021-22;
216,764 player-match rows). Criteria: every role has ≥ 20 players × 900 minutes in every league-season;
the highest year-to-year stability of a player's main role among vocabularies that pass.

| Vocabulary | roles | passes 20 × 900 everywhere | same main role next season |
|---|---|---|---|
| raw Understat codes | 16 | no (min 5 of 16) | 77.5% |
| left/right collapsed | 11 | no (min 5) | 79.3% |
| line only (GK/D/DM/M/AM/FW) | 6 | no (min 4) | 83.8% |
| hybrid (GK, CB, FB, CM, AM, W, ST) | 7 | no — AM 1–20 players | 89.0% |
| hybrid, AM → CM | 6 | yes (min 23) | 90.4% |
| **hybrid, AM → W** | **6** | **yes (min 23)** | **90.6%** |

**Chosen (proposed): six roles** — `GK`; `CB` = DC; `FB` = DL, DR, DML, DMR; `CM` = MC, DMC;
`W` = ML, MR, AML, AMR, FWL, FWR, AMC; `ST` = FW.

- The raw codes are formation slots, not roles: the commonest "changes" are MC↔DMC (161 pairs) and
  AMC↔MC (52), and a winger is ML in a 4-4-2, AML in a 4-3-3, FWL with split strikers — ML/MR
  carried 3% of minutes in 2014-15 and 1% by 2024-25 while AML/AMR rose in step. No wide code reaches
  20 players on its own in any season. Rejected: raw and side-collapsed (fail the bar, less stable).
- AM (pure AMC) folds into W, not CM: its per-90 output (xG 0.21, xA 0.18, key passes 1.6, buildup
  share 0.23) matches the wingers (0.20, 0.17, 1.4, 0.20) and not the central midfielders (0.08,
  0.08, 0.9, 0.27). Stability is a coin-flip between the two folds (90.4 vs 90.6); the profile decides.
- The LW-vs-LM distinction the spec wanted from the codes is not in the codes — it is in the stats
  (defensive and work-rate shares), which is where Phase 2 reads it. Side is kept as an attribute:
  left-side share of sided minutes has year-to-year r = 0.91 (FB 0.96, W 0.81).
- `Sub` rows (22.9% of rows, 5.9% of minutes) get the player's season main role at that club; the
  1,038 sub-only player-seasons hold 0.22% of minutes and are the only rows without a role.
- To re-check in Step 6 on all 60 league-seasons: the 20 × 900 bar per role (Bundesliga has 18
  clubs; thinnest role there is the risk) and the stability number.

**Port (after sign-off):** `scout.panel.player_match` — Understat rows + `role` from the mapping,
`Sub` → season main role, `side`; tests pin the mapping table, the sub rule, and that every code in
the data is mapped. Opponent Elo joins once the ClubElo coverage cell has run (pull in progress).

### 5a (cont.) — opponent Elo coverage per league-season

Fetching histories by Transfermarkt name fails (ClubElo keys are its own ASCII display names; the
host answers in ~90 s). Instead: one date listing per season (1 July) gives every club ClubElo rates
that day with its country and level; those names map onto Transfermarkt clubs with the Step 3
lineage rule, and coverage = share of Transfermarkt club-seasons with a mapped ClubElo club.

In [2]:
from scout.data import clubelo

ELO_COUNTRY = {"ENG": "GB1", "ESP": "ES1", "ITA": "IT1", "GER": "L1", "FRA": "FR1", "BEL": "BE1", "NED": "NL1",
               "POR": "PO1", "TUR": "TR1", "SUI": "C1", "BRA": "BRA1", "AUT": "A1", "DEN": "DK1"}
listings = []
for season in config.SEASONS:
    day = f"{season}-07-01"
    frame = retry.until_done(lambda day=day: clubelo.list_clubs_on(day))
    listings.append(frame.assign(season=season))
    print(day, len(frame), "clubs |", frame.Country.isin(ELO_COUNTRY).sum(), "in our 13 countries | levels:", frame.Level.value_counts().to_dict())
listings = pd.concat(listings, ignore_index=True)

2014-07-01 730 clubs | 337 in our 13 countries | levels: {1: 475, 0: 132, 2: 123}
2015-07-01 717 clubs | 333 in our 13 countries | levels: {1: 472, 0: 123, 2: 122}
2016-07-01 684 clubs | 335 in our 13 countries | levels: {1: 463, 2: 123, 0: 98}
2017-07-01 690 clubs | 338 in our 13 countries | levels: {1: 466, 2: 123, 0: 101}
2018-07-01 711 clubs | 341 in our 13 countries | levels: {1: 470, 2: 122, 0: 119}
2019-07-01 711 clubs | 341 in our 13 countries | levels: {1: 469, 2: 121, 0: 121}
2020-07-01 688 clubs | 317 in our 13 countries | levels: {1: 448, 0: 132, 2: 108}
2021-07-01 731 clubs | 341 in our 13 countries | levels: {1: 483, 0: 128, 2: 120}
2022-07-01 726 clubs | 340 in our 13 countries | levels: {1: 477, 0: 129, 2: 120}
2023-07-01 718 clubs | 341 in our 13 countries | levels: {1: 466, 0: 132, 2: 120}
2024-07-01 630 clubs | 308 in our 13 countries | levels: {1: 422, 2: 104, 0: 104}
2025-07-01 634 clubs | 306 in our 13 countries | levels: {1: 426, 0: 106, 2: 102}


In [3]:
elo_clubs = listings[listings.Country.isin(ELO_COUNTRY)].assign(competition_id=lambda d: d.Country.map(ELO_COUNTRY))
elo_names = elo_clubs[["competition_id", "Club"]].drop_duplicates().rename(columns={"Club": "team_name"})
elo_lineage = build_team_lineage(tm_clubs, {"clubelo": elo_names}, load_overrides("teams"))
print("ClubElo names in our countries:", len(elo_lineage), "| resolved:", elo_lineage.club_id.notna().sum())
print("unresolved (top 40 by score):"); print(elo_lineage[elo_lineage.club_id.isna()].sort_values("score", ascending=False)[["competition_id", "team_name", "score"]].head(40).to_string(index=False))
# coverage: Transfermarkt club-seasons (from the player panel) that have a ClubElo club rated on 1 July of that season
tm_club_seasons = tm_panel[["competition_id", "season", "club_id", "club_name"]].drop_duplicates()
rated = elo_clubs.merge(elo_lineage.dropna(subset=["club_id"])[["competition_id", "team_name", "club_id"]], left_on=["competition_id", "Club"], right_on=["competition_id", "team_name"])
covered = tm_club_seasons.merge(rated[["competition_id", "season", "club_id"]].drop_duplicates().assign(has_elo=True), on=["competition_id", "season", "club_id"], how="left")
covered["has_elo"] = covered.has_elo.fillna(False).astype(bool)
print("club-seasons with Elo, by league-season:"); print(covered.groupby(["competition_id", "season"]).has_elo.mean().unstack("season").round(2).to_string())
print("missing clubs (club-seasons without Elo), per league:"); print(covered[~covered.has_elo].groupby("competition_id").club_name.agg(lambda s: ", ".join(sorted(set(s)))).to_string())

ClubElo names in our countries: 490 | resolved: 340
unresolved (top 40 by score):
competition_id       team_name     score
           FR1       CA Bastia 88.888889
           ES1      Sociedad B 88.888889
           ES1       Sevilla B 87.500000
           ES1        Bilbao B 85.714286
            L1   FSV Frankfurt 81.818182
           GB1  Sheffield Weds 80.000000
           GB1    Bristol City 72.727273
            C1  Lausanne Ouchy 72.727273
           ES1      Andorra CF 70.000000
           GB1        Barnsley 66.666667
           IT1         Cosenza 66.666667
           TR1      Elazigspor 66.666667
            L1        Duisburg 63.157895
           FR1        Red Star 63.157895
           BE1          Bergen 62.500000
           IT1           Lecco 61.538462
            L1       Magdeburg 60.000000
           ES1        Lorca FC 60.000000
           IT1         Vicenza 58.823529
           ES1         Leonesa 58.823529
           IT1           Siena 57.142857
           FR1  

### Step 5a check — `scout.panel.player_match` reproduces the note's numbers

In [2]:
from scout.panel import player_match

built = player_match.build()
print(len(built), "rows (5a: 216,764) | roles:", built.role.value_counts(dropna=False).to_dict())
print(f"minutes with a role: {built.minutes[built.role.notna()].sum() / built.minutes.sum():.2%} (5a: 99.78%)")
starter_role_minutes = built[~built.is_sub].groupby(["league", "season", "player_id", "role"]).minutes.sum().reset_index()
eligible = starter_role_minutes[starter_role_minutes.minutes >= 900].groupby(["league", "season", "role"]).player_id.nunique()
print("thinnest role, players with ≥900 min in a league-season:", int(eligible.min()), "(5a: 23) | roles per league-season:", eligible.groupby(["league", "season"]).size().min())
print("side:", built.side.value_counts(dropna=False).to_dict())

240519 rows (5a: 216,764) | roles: {'W': 55451, 'CM': 51818, 'CB': 40436, 'FB': 39159, 'ST': 33225, 'GK': 16880, nan: 3550}
minutes with a role: 99.76% (5a: 99.78%)
thinnest role, players with ≥900 min in a league-season: 23 (5a: 23) | roles per league-season: 6
side: {<NA>: 179440, 'L': 30540, 'R': 30539}


## 5b — stints and values

Questions (plan 5b, spec D5): how many player-seasons span two clubs, how small the smaller stint is,
and which minimum stint (0 / 90 / 300 minutes) drops what; whether `value_at` should be read at
1 July or at the stint's first match. Bar: no player who later departs in a paid transfer (the
Phase 5 backtest population) falls below the floor in the season before leaving.

In [2]:
# Multi-club seasons in the Transfermarkt panel (13 leagues; a stint = player × club × season)
stints = tm_panel.copy()
clubs_per = stints.groupby(["tm_player_id", "season"]).club_id.nunique()
multi = clubs_per[clubs_per > 1]
print(f"player-seasons: {len(clubs_per)} | with 2+ clubs in the panel: {len(multi)} ({len(multi)/len(clubs_per):.1%}) | 3+: {int((clubs_per > 2).sum())}")
stints["n_clubs"] = stints.set_index(["tm_player_id", "season"]).index.map(clubs_per)
smaller = stints[stints.n_clubs > 1].sort_values("minutes").drop_duplicates(["tm_player_id", "season"])
print("minutes in the smaller stint:", smaller.minutes.describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_dict(), "| NaN (lineup-only):", int(smaller.minutes.isna().sum()))
total_minutes = stints.minutes.sum()
for floor in [0, 90, 300, 900]:
    dropped = stints[stints.minutes.fillna(0) < max(floor, 1)] if floor else stints[stints.minutes.isna()]
    print(f"floor {floor:3d} min: drops {len(dropped):6d} stints ({len(dropped)/len(stints):.1%} of rows, {dropped.minutes.sum()/total_minutes:.2%} of minutes)")

player-seasons: 73179 | with 2+ clubs in the panel: 4528 (6.2%) | 3+: 65
minutes in the smaller stint: {'count': 4377.0, 'mean': 416.0, 'std': 506.0, 'min': 1.0, '10%': 19.0, '25%': 80.0, '50%': 225.0, '75%': 572.0, '90%': 1091.0, 'max': 3240.0} | NaN (lineup-only): 151
floor   0 min: drops  15645 stints (20.1% of rows, 0.00% of minutes)
floor  90 min: drops  23070 stints (29.7% of rows, 0.31% of minutes)
floor 300 min: drops  30826 stints (39.6% of rows, 2.18% of minutes)
floor 900 min: drops  43834 stints (56.3% of rows, 12.42% of minutes)


In [3]:
# Within-league moves in Understat (files on disk): the same player for two teams in one league-season
pm_teams = pm.groupby(["league", "season", "player_id"]).agg(teams=("team_id", "nunique"), minutes=("minutes", "sum"))
two = pm_teams[pm_teams.teams > 1]
print(f"Understat player-seasons: {len(pm_teams)} | two teams same league: {len(two)} ({len(two)/len(pm_teams):.1%}) holding {two.minutes.sum()/pm_teams.minutes.sum():.2%} of minutes")
first_match = pm.groupby(["league", "season", "player_id", "team_id"]).game.min().str[:10]
print("first-match dates of the second stint (month):", pd.to_datetime(first_match.loc[first_match.index.droplevel('team_id').isin(two.index)]).dt.month.value_counts().sort_index().to_dict())

Understat player-seasons: 12616 | two teams same league: 272 (2.2%) holding 2.04% of minutes


first-match dates of the second stint (month): {1: 81, 2: 108, 3: 2, 4: 2, 5: 1, 6: 2, 7: 2, 8: 206, 9: 106, 10: 24, 11: 4, 12: 6}


In [4]:
# value_at: 1 July of the season vs the stint's first appearance date, for stints that start after 1 September
con = tm_loader.connect()
comp_list = ",".join(f"'{c}'" for c in competitions)
stint_start = con.execute(f'''
  SELECT a.player_id, CAST(a.player_club_id AS INTEGER) AS club_id, CAST(g.season AS INTEGER) AS season,
         MIN(a.date) AS first_date
  FROM appearances a JOIN games g ON a.game_id = g.game_id
  WHERE g.competition_id IN ({comp_list}) AND CAST(g.season AS INTEGER) BETWEEN 2014 AND 2025
  GROUP BY 1, 2, 3''').df()
valuations = con.execute("SELECT player_id, date, market_value_in_eur FROM player_valuations").df()
valuations["date"] = pd.to_datetime(valuations.date)
valuations = valuations.sort_values("date")  # merge_asof wants the right side sorted on the key


def value_at(frame, date_col):
    keyed = frame[["player_id", date_col]].rename(columns={date_col: "date"}).assign(date=lambda d: pd.to_datetime(d.date)).sort_values("date")
    hit = pd.merge_asof(keyed.reset_index(), valuations, on="date", by="player_id", direction="backward")
    return hit.set_index("index").market_value_in_eur.reindex(frame.index)


stint_start["july_first"] = pd.to_datetime(stint_start.season.astype(str) + "-07-01")
stint_start["value_july"] = value_at(stint_start, "july_first")
stint_start["value_start"] = value_at(stint_start, "first_date")
late = stint_start[pd.to_datetime(stint_start.first_date) >= pd.to_datetime(stint_start.season.astype(str) + "-09-01")].dropna(subset=["value_july", "value_start"])
ratio = late.value_start / late.value_july
print(f"stints starting after 1 Sept with both values: {len(late)} of {len(stint_start)} | value identical: {(ratio == 1).mean():.1%} | ratio start/July percentiles:", ratio.quantile([.1, .25, .5, .75, .9]).round(2).to_dict())
winter = late[pd.to_datetime(late.first_date).dt.month.isin([1, 2])]
print(f"January/February joiners: {len(winter)} | median ratio {(winter.value_start / winter.value_july).median():.2f} | |log ratio| > 0.25 for {(np.abs(np.log(winter.value_start / winter.value_july)) > 0.25).mean():.1%}")

stints starting after 1 Sept with both values: 22219 of 62149 | value identical: 59.5% | ratio start/July percentiles: {0.1: 0.75, 0.25: 1.0, 0.5: 1.0, 0.75: 1.0, 0.9: 1.4}
January/February joiners: 6042 | median ratio 1.00 | |log ratio| > 0.25 for 43.0%


In [5]:
# The backtest population: players sold for a fee from a Big-5 club (transfer seasons 15/16 → 24/25) — minutes at that club the season before
sales = con.execute(f'''
  SELECT t.player_id, t.from_club_id AS club_id, t.transfer_date, t.transfer_fee,
         CAST('20' || SUBSTR(t.transfer_season, 1, 2) AS INTEGER) AS transfer_season
  FROM transfers t
  WHERE t.transfer_fee > 0 AND t.from_club_id IN (SELECT DISTINCT club_id FROM (SELECT CAST(club_id AS INTEGER) AS club_id, domestic_competition_id FROM clubs) WHERE domestic_competition_id IN ('GB1','ES1','IT1','L1','FR1'))
    AND t.transfer_season BETWEEN '15/16' AND '24/25' ''').df()
# summer sales close the previous season; winter sales (Jan–Mar) close the current one
sales["prev_season"] = sales.transfer_season - (~pd.to_datetime(sales.transfer_date).dt.month.isin([1, 2, 3])).astype(int)
prev = sales.merge(tm_panel[["tm_player_id", "club_id", "season", "minutes"]], left_on=["player_id", "club_id", "prev_season"], right_on=["tm_player_id", "club_id", "season"], how="left")
print(f"paid departures from Big-5 clubs: {len(sales)} | with a panel row at that club the season before: {prev.minutes.notna().sum()} | no row at all (youth/reserve/out on loan): {prev.tm_player_id.isna().sum()}")
present = prev.dropna(subset=["minutes"])
for floor in [90, 300, 900]:
    below = present[present.minutes < floor]
    print(f"floor {floor:3d}: {len(below):4d} departing players below it ({len(below)/len(present):.1%}) | their fees sum {below.transfer_fee.sum()/1e6:.0f} M€ of {present.transfer_fee.sum()/1e6:.0f} M€")
print("departing players' previous-season minutes:", present.minutes.describe(percentiles=[.05, .1, .25, .5]).round(0).to_dict())

paid departures from Big-5 clubs: 3678 | with a panel row at that club the season before: 1959 | no row at all (youth/reserve/out on loan): 1586
floor  90:  159 departing players below it (8.1%) | their fees sum 1256 M€ of 28678 M€
floor 300:  352 departing players below it (18.0%) | their fees sum 2574 M€ of 28678 M€
floor 900:  681 departing players below it (34.8%) | their fees sum 5526 M€ of 28678 M€
departing players' previous-season minutes: {'count': 1959.0, 'mean': 1483.0, 'std': 1018.0, 'min': 1.0, '5%': 41.0, '10%': 110.0, '25%': 534.0, '50%': 1439.0, 'max': 3420.0}


### Step 5b note — stints and values (decision D5), from the outputs above

**Chosen (proposed for sign-off).** A stint is player × club × season; **no minutes floor in the
table** — every stint with ≥ 1 minute is a row, lineup-only stints (bench, 20.1% of rows, 0.00% of
minutes) stay as identity rows with `minutes` NaN and no per-90 quantities. `value_at` is stored
twice: at the stint's first appearance (`value_at_start`) and at 1 July of the season
(`value_july`, the season-level value from Part 1b), each with its `value_age_days`.

- 6.2% of player-seasons span two clubs (65 span three); the smaller stint has a median of 225
  minutes, a quarter are under 80. A 90-minute floor drops 0.31% of minutes, 300 drops 2.18%, 900
  drops 12.4%.
- The backtest population is where a floor bites: of 3,678 paid departures from Big-5 clubs
  (15/16 → 24/25), 8.1% had under 90 minutes at the selling club in their last season and 18.0%
  under 300 (1.26 / 2.57 bn € of 28.7 bn € in fees). Those players are profiled from earlier
  seasons, not dropped — so the floor belongs to the model (a Phase 2 decision with these numbers
  as its reference), not to the table. Rejected: 90 / 300 / 900 as table floors.
- 1,586 of the 3,678 paid departures (43%) have **no panel row at the selling club** the season
  before — youth and reserve players, loanees sold from the parent club. The backtest population is
  defined on panel presence (Phase 5 note; the count goes in the writeup).
- `value_at` at 1 July is the wrong price for a mid-season move: among 22,219 stints that start after
  1 September, 40.5% carry a different valuation at their first match than at 1 July; for the 6,042
  January/February joiners 43.0% differ by more than 25% (log ratio). Rejected: 1 July only.
  Both timestamps stay because they answer different questions (what the season's profile is worth
  vs what the move cost).

**Port (after sign-off):** `scout.panel.stints` — the Transfermarkt panel rows + first/last
appearance date + the two valuations with ages; tests pin: lineup-only rows kept with NaN minutes,
`value_at` = last valuation on or before the date (never after), both timestamps present.

### Step 5b check — `scout.panel.stints` reproduces the note's numbers

In [2]:
from scout.panel import stints as stints_panel

built_stints = stints_panel.build(competitions, list(config.SEASONS))
print(len(built_stints), "stints | lineup-only (NaN minutes):", f"{built_stints.minutes.isna().mean():.1%} (5b: 20.1%)")
late = built_stints[pd.to_datetime(built_stints.first_date) >= pd.to_datetime(built_stints.season.astype(str) + "-09-01")].dropna(subset=["value_july", "value_at_start"])
print(f"post-September stints with both values: {len(late)} (5b: 22,219) | value differs from 1 July: {(late.value_at_start != late.value_july).mean():.1%} (5b: 40.5%)")
print("value_age_days at 1 July, median by season:", built_stints.groupby("season").value_age_days_july.median().astype("Int64").to_dict())
print("value coverage: start", f"{built_stints.value_at_start.notna().mean():.1%}", "| July", f"{built_stints.value_july.notna().mean():.1%}", "| never read ahead:", bool((built_stints.value_age_days_start.dropna() >= 0).all() and (built_stints.value_age_days_july.dropna() >= 0).all()))

77794 stints | lineup-only (NaN minutes): 20.1% (5b: 20.1%)
post-September stints with both values: 22219 (5b: 22,219) | value differs from 1 July: 40.5% (5b: 40.5%)
value_age_days at 1 July, median by season: {2014: 155, 2015: 0, 2016: 143, 2017: 22, 2018: 28, 2019: 23, 2020: 84, 2021: 27, 2022: 24, 2023: 15, 2024: 27, 2025: 25}
value coverage: start 76.7% | July 87.5% | never read ahead: True


### Step 5a note (cont.) — opponent Elo coverage

ClubElo rates 630–730 clubs on any 1 July, 306–341 in our 13 countries; **Brazil is absent**
(Part 7), so Brazilian match rows carry NaN opponent Elo — decided, not a gap to fix. Its names
are ASCII abbreviations ("Man City", "Koeln", "Paris SG", "Gladbach"), so the Step 3 lineage rule
resolved 303 of 490; 40 hand overrides (`overrides/teams.csv`, provider `clubelo`) fix every
Big-5 miss and the recognisable feeder ones (Kayseri = Kayserispor, not Erciyesspor; Verona =
Hellas, not Chievo; Brugge = Club, not Cercle — all three were 100-score ties broken the wrong way).

Result: **every Big-5 club-season 2014-15 → 2025-26 has a ClubElo club (100%)**; Belgium, Denmark,
Netherlands, Portugal, Turkey, Switzerland, Austria 78–100% on 1 July. The feeder misses are
newly promoted clubs ClubElo does not rate until they play top-flight football (Groningen,
NAC, Odense, Göztepe, Sion…), so 1 July understates them; Step 6 measures Elo on the match date
instead, which is the number that matters (`elo_on(history, date)` reads the interval containing
the date, never ahead). Histories for the 340 resolved clubs are being fetched in the
background (`data/cache/clubelo_histories.log`; the host answers in ~90 s per club).

## 5c — `team_season` style vector: which components are stable enough to keep?

Question (plan 5c, spec D3): for each candidate component of a club's style (from Understat
team-match rows: xG for/against, non-penalty xG, PPDA own/allowed, deep completions own/allowed,
expected points), does the season mean carry signal — year-to-year r ≥ 0.3 for the same club in
the same league, and split-half within a season? Variants: all matches vs home-only vs away-only.

In [2]:
tm_files = sorted((config.RAW / "understat" / "team_match").glob("*.parquet"))
team_match = pd.concat([pd.read_parquet(f) for f in tm_files], ignore_index=True)
team_match["season"] = team_match.season_id.astype(int)
print(len(team_match), "matches |", team_match.groupby("league").season.nunique().to_dict())

STATS = ["xg", "np_xg", "ppda", "deep_completions", "expected_points", "goals"]
def side_rows(frame, own, other):
    rows = frame[["league", "season", "game_id", "date", f"{own}_team_id", f"{own}_team"]].rename(columns={f"{own}_team_id": "team_id", f"{own}_team": "team"})
    rows["is_home"] = own == "home"
    for stat in STATS:
        rows[f"{stat}_for"] = frame[f"{own}_{stat}"]
        rows[f"{stat}_against"] = frame[f"{other}_{stat}"]
    return rows
long = pd.concat([side_rows(team_match, "home", "away"), side_rows(team_match, "away", "home")], ignore_index=True).sort_values(["league", "season", "team_id", "date"])
long["match_no"] = long.groupby(["league", "season", "team_id"]).cumcount()
print(len(long), "team-match rows | teams per league-season:", long.groupby(["league", "season"]).team_id.nunique().unique())
COMPONENTS = [c for c in long.columns if c.endswith(("_for", "_against"))]

21589 matches | {'ENG-Premier League': 12, 'ESP-La Liga': 12, 'FRA-Ligue 1': 12, 'GER-Bundesliga': 12, 'ITA-Serie A': 12}
43178 team-match rows | teams per league-season: [20 18]


In [3]:
def stability(rows, label):
    season_mean = rows.groupby(["league", "season", "team_id"])[COMPONENTS].mean().reset_index()
    nxt = season_mean.assign(season=season_mean.season - 1)
    pairs = season_mean.merge(nxt, on=["league", "season", "team_id"], suffixes=("", "_next"))
    halves = rows.assign(half=rows.match_no % 2).groupby(["league", "season", "team_id", "half"])[COMPONENTS].mean().unstack("half")
    out = pd.DataFrame({
        "y2y_r": {c: pairs[c].corr(pairs[f"{c}_next"]) for c in COMPONENTS},
        "split_half_r": {c: halves[(c, 0)].corr(halves[(c, 1)]) for c in COMPONENTS},
    })
    out.columns = pd.MultiIndex.from_product([[label], out.columns])
    return out, len(pairs)

tables = []
for label, rows in [("all", long), ("home", long[long.is_home]), ("away", long[~long.is_home])]:
    table, n_pairs = stability(rows, label)
    tables.append(table)
    print(f"{label}: {n_pairs} club-season pairs")
result = pd.concat(tables, axis=1).round(2)
print(result.to_string())
print("\ncomponents with year-to-year r ≥ 0.3 on all matches:", result[result[("all", "y2y_r")] >= 0.3].index.tolist())
print("below 0.3:", result[result[("all", "y2y_r")] < 0.3].index.tolist())

all: 922 club-season pairs
home: 922 club-season pairs
away: 922 club-season pairs
                           all               home               away             
                         y2y_r split_half_r y2y_r split_half_r y2y_r split_half_r
xg_for                    0.80         0.75  0.73         0.60  0.71         0.62
xg_against                0.67         0.62  0.52         0.42  0.58         0.43
np_xg_for                 0.79         0.76  0.73         0.60  0.71         0.63
np_xg_against             0.66         0.64  0.53         0.42  0.58         0.44
ppda_for                  0.71         0.76  0.64         0.58  0.66         0.54
ppda_against              0.84         0.86  0.79         0.73  0.77         0.68
deep_completions_for      0.85         0.87  0.81         0.76  0.80         0.76
deep_completions_against  0.74         0.68  0.64         0.45  0.66         0.44
expected_points_for       0.80         0.72  0.71         0.58  0.73         0.62
expected_points

In [4]:
# Are the kept components measuring different things? Correlations among season means (all matches)
season_mean = long.groupby(["league", "season", "team_id"])[COMPONENTS].mean()
kept = result[result[("all", "y2y_r")] >= 0.3].index.tolist()
print(season_mean[kept].corr().round(2).to_string())

                          xg_for  xg_against  np_xg_for  np_xg_against  ppda_for  ppda_against  deep_completions_for  deep_completions_against  expected_points_for  expected_points_against  goals_for  goals_against
xg_for                      1.00       -0.47       0.99          -0.44     -0.37          0.74                  0.88                     -0.44                 0.89                    -0.86       0.91          -0.50
xg_against                 -0.47        1.00      -0.46           0.99      0.52         -0.42                 -0.46                      0.82                -0.80                     0.83      -0.50           0.85
np_xg_for                   0.99       -0.46       1.00          -0.43     -0.36          0.73                  0.88                     -0.42                 0.88                    -0.85       0.90          -0.49
np_xg_against              -0.44        0.99      -0.43           1.00      0.52         -0.40                 -0.43                      0.

### Step 5c note — style vector (decision D3), from the outputs above

All 60 Big-5 league-seasons (21,589 matches, 922 same-club consecutive-season pairs). Every
candidate season mean clears the r ≥ 0.3 bar by a wide margin: year-to-year r 0.58 (goals against)
to 0.85 (deep completions for); split-half within season 0.49–0.87. Means over all matches beat
home-only and away-only for every component (e.g. xG against 0.67 vs 0.52 / 0.58), so the
home/away variants are rejected.

**Chosen: seven components** — `np_xg_for`, `np_xg_against`, `ppda_for` (own pressing),
`ppda_against` (pressing faced), `deep_completions_for`, `deep_completions_against`,
`expected_points_for`. Dropped as redundant, not as weak: `xg_*` (r = 0.99 with `np_xg_*`),
`expected_points_against` (r = −1.00 with `_for`). `goals_for/against` stay in `team_season` as
outcomes, not as style (they are the noisiest and are what xG already summarises). Among the
seven, `ppda_for` is the most independent axis (|r| ≤ 0.59 with everything else); the attacking
components correlate 0.7–0.9 with each other and with strength — the Phase 2 style model has to
separate "how" from "how good" (residualise on expected points), which is why `expected_points_for`
is kept in the vector.

Ported: `scout.panel.team_season` (`team_match_long`, `season_means`, `STYLE`).

### Step 5c check — `scout.panel.team_season` reproduces the note's numbers

In [2]:
from scout.panel import team_season

built_teams = team_season.build()
print(len(built_teams), "club-seasons (5c: 60 league-seasons × 18–20 clubs) | matches per club-season:", built_teams.matches.describe()[["min", "max"]].astype(int).to_dict())
nxt = built_teams.assign(season=built_teams.season - 1)
pairs = built_teams.merge(nxt, on=["league", "season", "team_id"], suffixes=("", "_next"))
print(len(pairs), "consecutive pairs (5c: 922)")
print({c: round(pairs[c].corr(pairs[f"{c}_next"]), 2) for c in team_season.STYLE}, "(5c: np_xg_for 0.79, deep_completions_for 0.85, ppda_against 0.84)")

1170 club-seasons (5c: 60 league-seasons × 18–20 clubs) | matches per club-season: {'min': 27, 'max': 38}
922 consecutive pairs (5c: 922)
{'np_xg_for': np.float64(0.79), 'np_xg_against': np.float64(0.66), 'ppda_for': np.float64(0.71), 'ppda_against': np.float64(0.84), 'deep_completions_for': np.float64(0.85), 'deep_completions_against': np.float64(0.74), 'expected_points_for': np.float64(0.8)} (5c: np_xg_for 0.79, deep_completions_for 0.85, ppda_against 0.84)


## 5d — work-rate mapping: which Sofascore and FotMob metrics are the same quantity?

Question (plan 5d, spec D2): on players present in both providers in the same club-season, regress
FotMob per-90 on Sofascore per-90 for each candidate pair. Same quantity if slope ∈ [0.9, 1.1] and
r ≥ 0.9; otherwise the metric stays provider-specific and is NaN where its provider is absent.
Pairing: club via the Step 3 lineage, player via the Step 4 cascade (threshold 85) within the
club-season. Uses the FotMob league-seasons on disk (pull still running).

In [2]:
from scout.data import fotmob, sofascore
from scout.identity import match_players

fm = fotmob.load()
ss = sofascore.load()
tm_clubs_all = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
provider_teams = {
    "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(),
    "fotmob": fm[["competition_id", "team_name"]].drop_duplicates(),
}
lineage = build_team_lineage(tm_clubs_all, provider_teams, load_overrides("teams"))
print("lineage unresolved:", lineage.club_id.isna().groupby(lineage.provider).sum().to_dict())

fm_wide = fm.pivot_table(index=["competition_id", "season", "fotmob_player_id", "player_name", "team_name"], columns="stat", values="stat_value", aggfunc="first").reset_index()
fm_minutes = fm[fm.stat == "mins_played"].groupby(["competition_id", "season", "fotmob_player_id"]).stat_value.first().rename("fm_minutes")
fm_wide = fm_wide.merge(fm_minutes, on=["competition_id", "season", "fotmob_player_id"], how="left")
print("fotmob player-seasons:", len(fm_wide), "| league-seasons:", fm_wide.groupby("competition_id").season.nunique().to_dict())
for provider, frame in [("sofascore", ss), ("fotmob", fm_wide)]:
    frame["club_id"] = frame.merge(lineage[lineage.provider == provider][["competition_id", "team_name", "club_id"]], on=["competition_id", "team_name"], how="left").club_id.values
    frame["club_key"] = frame.club_id.astype("Int64").astype(str)

lineage unresolved: {'fotmob': 6, 'sofascore': 34}
fotmob player-seasons: 31815 | league-seasons: {'BE1': 8, 'ES1': 10, 'FR1': 10, 'GB1': 10, 'IT1': 10, 'L1': 10, 'NL1': 4}


In [3]:
# Pair the players: FotMob names → Sofascore ids within the club-season
paired = []
for (comp, season), left in fm_wide.groupby(["competition_id", "season"]):
    right = ss[(ss.competition_id == comp) & (ss.season == season)].rename(columns={"sofascore_player_id": "right_id", "player_name": "name"})
    if right.empty:
        continue
    out = match_players(left.rename(columns={"player_name": "name"})[["name", "club_key", "season", "fotmob_player_id"]], right[["right_id", "name", "club_key", "season"]])
    paired.append(out.assign(competition_id=comp))
paired = pd.concat(paired, ignore_index=True)
print("fotmob player-seasons with a Sofascore partner:", paired.right_id.notna().sum(), "of", len(paired), "|", paired.method.value_counts().to_dict())
both = fm_wide.merge(paired.dropna(subset=["right_id"])[["competition_id", "season", "fotmob_player_id", "right_id"]], on=["competition_id", "season", "fotmob_player_id"]).merge(
    ss, left_on=["competition_id", "season", "right_id"], right_on=["competition_id", "season", "sofascore_player_id"], suffixes=("_fm", "_ss"))
regular = both[(both.fm_minutes >= 900) & (both.minutesPlayed >= 900)].copy()
print("paired player-seasons with ≥900 min on both sides:", len(regular), "| minutes agree within 5%:", f"{(np.abs(regular.fm_minutes / regular.minutesPlayed - 1) < 0.05).mean():.1%}")

fotmob player-seasons with a Sofascore partner: 31606 of 31815 | {'exact': 28736, 'name_unique': 1179, 'last_token': 1147, 'fuzzy': 544, 'unmatched': 209}
paired player-seasons with ≥900 min on both sides: 16354 | minutes agree within 5%: 98.8%


In [4]:
PAIRS = [  # (sofascore total, fotmob total)
    ("tackles", "total_tackle"), ("interceptions", "interception"), ("ballRecovery", "ball_recovery"), ("clearances", "effective_clearance"),
    ("possessionWonAttThird", "poss_won_att_3rd"), ("successfulDribbles", "won_contest"), ("keyPasses", "total_att_assist"),
    ("bigChancesCreated", "big_chance_created"), ("accuratePasses", "accurate_pass"), ("accurateLongBalls", "accurate_long_balls"),
    ("expectedGoals", "expected_goals"), ("expectedAssists", "expected_assists"), ("goals", "goals"), ("assists", "goal_assist"),
    ("fouls", "fouls"), ("saves", "saves"), ("goalsConceded", "goals_conceded"),
]
rows = []
for ss_name, fm_name in PAIRS:
    ss_col = ss_name if ss_name in regular else f"{ss_name}_ss"  # merge suffixes shared names
    fm_col = fm_name if fm_name in regular else f"{fm_name}_fm"
    sub = regular.assign(**{ss_col: pd.to_numeric(regular[ss_col]), fm_col: pd.to_numeric(regular[fm_col])}).dropna(subset=[ss_col, fm_col])
    sub = sub[(sub[ss_col] > 0) | (sub[fm_col] > 0)]
    x = sub[ss_col] / sub.minutesPlayed * 90
    y = sub[fm_col] / sub.fm_minutes * 90
    if len(sub) < 30:
        rows.append((ss_name, fm_name, len(sub), np.nan, np.nan, np.nan)); continue
    slope = np.polyfit(x, y, 1)[0]
    ratio = (y / x.replace(0, np.nan)).median()
    rows.append((ss_name, fm_name, len(sub), round(slope, 2), round(ratio, 2), round(x.corr(y), 3)))
table = pd.DataFrame(rows, columns=["sofascore", "fotmob", "n", "slope", "median_ratio", "r"])
table["same_quantity"] = table.slope.between(0.9, 1.1) & (table.r >= 0.9)
print(table.to_string(index=False))

            sofascore              fotmob     n  slope  median_ratio     r  same_quantity
              tackles        total_tackle 13512   0.05          0.04 0.796          False
        interceptions        interception 13470   0.04          0.04 0.838          False
         ballRecovery       ball_recovery  4228   0.03          0.04 0.563          False
           clearances effective_clearance 13912   0.04          0.04 0.891          False
possessionWonAttThird    poss_won_att_3rd 12209   0.05          0.05 0.741          False
   successfulDribbles         won_contest 13229   0.05          0.04 0.884          False
            keyPasses    total_att_assist 15098   0.99          1.00 0.999           True
    bigChancesCreated  big_chance_created 11853   1.00          1.00 1.000           True
       accuratePasses       accurate_pass 13987   0.04          0.04 0.741          False
    accurateLongBalls accurate_long_balls 13016   0.04          0.04 0.846          False
        ex

Slopes of 0.04–0.05 with r 0.8–0.9 are a units mismatch, not a different quantity: those FotMob
lists are already per 90 while Sofascore reports season totals, and the cell above divided FotMob by
minutes a second time. Which form each FotMob stat comes in is checked on the data, then the
regression is rerun with Sofascore per 90 against FotMob in its own form.

In [5]:
# One player-season, every FotMob stat: stat_value vs sub_stat_value next to minutes and matches
sample = fm[(fm.competition_id == "GB1") & (fm.season == 2023) & (fm.player_name == "Declan Rice")]
print(sample[["stat", "stat_value", "sub_stat_value", "minutes_played", "matches_played"]].to_string(index=False))

                                       stat  stat_value  sub_stat_value  minutes_played  matches_played
                                      goals        7.00            0.00            3230              38
                                goal_assist        8.00            4.80            3230              38
                     _goals_and_goal_assist       15.00            8.10            3230              38
                                     rating        7.66            3.00            3230              38
                                mins_played     3230.00           38.00            3230              38
                               goals_per_90        0.20            7.00            3230              38
                             expected_goals        3.30            7.00            3230              38
                      expected_goals_per_90        0.09            0.20            3230              38
                     expected_goalsontarget        4.50         

In [6]:
# For each FotMob stat: is stat_value a season total or a per-90 rate? Test against the matched Sofascore total
FORMS = {"total": lambda v, m: v, "per90": lambda v, m: v * m / 90}
rows = []
for ss_name, fm_name in PAIRS:
    ss_col = ss_name if ss_name in regular else f"{ss_name}_ss"
    fm_col = fm_name if fm_name in regular else f"{fm_name}_fm"
    sub = regular.assign(**{ss_col: pd.to_numeric(regular[ss_col]), fm_col: pd.to_numeric(regular[fm_col])}).dropna(subset=[ss_col, fm_col])
    sub = sub[(sub[ss_col] > 0) | (sub[fm_col] > 0)]
    if len(sub) < 30:
        continue
    x = sub[ss_col] / sub.minutesPlayed * 90  # Sofascore per 90
    best = None
    for form, to_total in FORMS.items():
        y = to_total(sub[fm_col], sub.fm_minutes) / sub.fm_minutes * 90
        slope = np.polyfit(x, y, 1)[0]
        if best is None or abs(slope - 1) < abs(best[1] - 1):
            best = (form, slope, x.corr(y), (y / x.replace(0, np.nan)).median())
    rows.append((ss_name, fm_name, len(sub), best[0], round(best[1], 2), round(best[3], 2), round(best[2], 3)))
table = pd.DataFrame(rows, columns=["sofascore", "fotmob", "n", "fotmob_form", "slope", "median_ratio", "r"])
table["same_quantity"] = table.slope.between(0.9, 1.1) & (table.r >= 0.9)
print(table.to_string(index=False))

            sofascore              fotmob     n fotmob_form  slope  median_ratio     r  same_quantity
              tackles        total_tackle 13512       per90   1.00          1.00 0.999           True
        interceptions        interception 13470       per90   1.00          1.00 0.999           True
         ballRecovery       ball_recovery  4228       per90   1.00          1.00 1.000           True
           clearances effective_clearance 13912       per90   0.99          1.00 0.999           True
possessionWonAttThird    poss_won_att_3rd 12209       per90   0.84          1.01 0.866          False
   successfulDribbles         won_contest 13229       per90   1.00          1.00 0.999           True
            keyPasses    total_att_assist 15098       total   0.99          1.00 0.999           True
    bigChancesCreated  big_chance_created 11853       total   1.00          1.00 1.000           True
       accuratePasses       accurate_pass 13987       per90   1.00          1.00 1

### Step 5d note — work-rate mapping (decision D2), from the outputs above

16,069 player-seasons with ≥ 900 minutes on both providers (club via lineage, player via the
cascade; minutes agree within 5% for 98.8%). Sofascore reports season totals; FotMob lists are
either season totals (goals, assists, xG, xA, key passes, big chances created) or per-90 rates
(tackles, interceptions, recoveries, clearances, dribbles, passes, long balls, fouls, saves, goals
conceded), with the other form in `sub_stat_value` — the first regression's 0.04 slopes were that
units mismatch, not a disagreement.

**Chosen: 16 shared metrics, one provider-specific pair.** With the right form, 16 of 17 pairs
regress at slope 0.99–1.00 and r ≥ 0.997 — the two providers are the same Opta feed, so a metric
can be read from either and the leagues where only one provider reaches back (Belgium and Denmark
via FotMob) join the same columns. `possessionWonAttThird` ↔ `poss_won_att_3rd` (slope 0.83,
r 0.86) is a different definition and stays as two columns. Missing on either side stays NaN (the
FotMob per-90 lists have an undocumented eligibility rule — Part 6); sample sizes show the known
Sofascore gaps (`ballRecovery` 4,228 pairs, xG/xA ≈ 5,600 — OPEN Step 5 item, measured in Step 6).

Ported: `scout.panel.workrate` (`SHARED`, `PROVIDER_SPECIFIC`, `sofascore_per90`,
`fotmob_per90`). The join across providers onto Transfermarkt ids waits for the FotMob identity
rate (Step 6, same policy as Sofascore).

### Step 5d check — `scout.panel.workrate` reproduces the regression through the package

In [7]:
from scout.panel import workrate

# the merge suffixed the shared names (goals, fouls, saves...); hand each provider its own names back
ss_side = workrate.sofascore_per90(regular.rename(columns={c: c[:-3] for c in regular.columns if c.endswith("_ss")}))
fm_side = workrate.fotmob_per90(regular.rename(columns={c: c[:-3] for c in regular.columns if c.endswith("_fm")}))
checks = {}
for name in list(workrate.SHARED) + ["possession_won_att_third"]:
    x = ss_side[name if name in ss_side else f"{name}_sofascore"]
    y = fm_side[name if name in fm_side else f"{name}_fotmob"]
    ok = x.notna() & y.notna() & ((x > 0) | (y > 0))
    checks[name] = (int(ok.sum()), round(np.polyfit(x[ok], y[ok], 1)[0], 2), round(x[ok].corr(y[ok]), 3))
print(pd.DataFrame(checks, index=["n", "slope", "r"]).T.to_string())

                                n  slope      r
tackles                   13512.0   1.00  0.999
interceptions             13470.0   1.00  0.999
recoveries                 4228.0   1.00  1.000
clearances                13912.0   0.99  0.999
dribbles                  13229.0   1.00  0.999
key_passes                15098.0   0.99  0.999
big_chances_created       11853.0   1.00  1.000
accurate_passes           13987.0   1.00  1.000
accurate_long_balls       13016.0   1.00  1.000
xg                         5608.0   1.00  0.999
xa                         6044.0   1.00  0.999
goals                     10868.0   1.00  0.999
assists                   10544.0   1.00  0.999
fouls                     13455.0   1.00  0.998
saves                       952.0   1.00  0.998
goals_conceded              952.0   1.00  0.997
possession_won_att_third  12209.0   0.84  0.866


## 5f — `market`: the Part 2d rule, ported

The transfer classification (fee-independent, 36-month mirror window) and the cost rule were
decided in Part 2d; this step ports them to `scout.panel.market` and reads the class counts and
cost coverage off the real table. Contract end stays out (current-only, Part 3 — descriptive).

In [2]:
from scout.panel import market

moves = market.build()
print(len(moves), "moves 14/15 → 25/26 |", moves.kind.value_counts().to_dict())
print("(Part 2c at 24 months, before the undisclosed split: internal 47,119 · loan_return 33,087 · free 31,024 · loan_out 26,859 · paid 13,430)")
print("cost coverage by kind:", moves.groupby("kind").cost.apply(lambda c: f"{c.notna().mean():.0%}").to_dict())
print("median cost (M€):", moves.groupby("kind").cost.median().div(1e6).round(2).dropna().to_dict())
paid_in = moves[moves.kind == "paid"].merge(tm_panel[["club_id", "competition_id"]].drop_duplicates(), left_on="to_club_id", right_on="club_id")
print("paid purchases by panel clubs, per league:", paid_in.competition_id.value_counts().to_dict())

151519 moves 14/15 → 25/26 | {'internal': 44637, 'loan_return': 33573, 'free': 30573, 'loan_out': 27135, 'paid': 13273, 'undisclosed': 2328}
(Part 2c at 24 months, before the undisclosed split: internal 47,119 · loan_return 33,087 · free 31,024 · loan_out 26,859 · paid 13,430)
cost coverage by kind: {'free': '64%', 'internal': '0%', 'loan_out': '0%', 'loan_return': '0%', 'paid': '100%', 'undisclosed': '97%'}
median cost (M€): {'free': 0.3, 'paid': 1.45, 'undisclosed': 0.5}
paid purchases by panel clubs, per league: {'GB1': 1308, 'IT1': 970, 'L1': 901, 'FR1': 768, 'ES1': 615, 'BE1': 536, 'TR1': 498, 'BRA1': 450, 'NL1': 401, 'PO1': 388, 'DK1': 266, 'A1': 157, 'C1': 153}


### Step 5f note — `market`, from the output above

Port of the Part 2d rule, read back off the real table (151,519 moves, 14/15 → 25/26):
`internal` 44,637 · `loan_return` 33,573 · `free` 30,573 · `loan_out` 27,135 · `paid` 13,273 ·
`undisclosed` 2,328. Against Part 2c's 24-month counts: 486 more returns and 276 more loan-outs
(loans longer than two years), 157 fewer `paid` (paid loans now correctly loans), and `undisclosed`
carved out of `internal` (NULL fee, both clubs in `clubs`). Cost: 100% of paid moves, 97% of
undisclosed, 64% of free — the free transfers without a market value at the move are lower-tier
players Transfermarkt never valued; they stay NaN. Paid purchases by panel clubs: Premier League
1,308, Serie A 970, Bundesliga 901, Ligue 1 768, La Liga 615; feeders 153 (Switzerland) – 536
(Belgium) — the Phase 4 market-model sample.

**5e (injuries) is deferred, not cut:** the injury pull is at 350 of 23,787 players (~3 s per
page, days to finish); the candidate features and the criterion (correlation with next season's
minutes share within age bands) run unchanged once the file is complete.

# Step 6 — Reconciliation and the checkpoint report

Bars (spec, plan Step 6): identity > 95% of Big-5 player-minutes in every league-season for each
provider; Understat and Transfermarkt minutes within 5% for > 90% of club-seasons; opponent Elo
≈ 100% in the Big 5. A club-season off by more than 5% is a team-map error — fix the map, not the
data. The parts below run on what is on disk; per-match Elo and injuries wait for their pulls.

In [2]:
from scout.data import fotmob, reep, sofascore, understat
from scout.identity import match_players, resolve_player_ids

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}
us = understat.load("player_season")
us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
fm = fotmob.load()
fm_wide = fm[fm.stat == "mins_played"][["competition_id", "season", "fotmob_player_id", "player_name", "team_name", "stat_value"]].rename(columns={"stat_value": "fm_minutes"}).drop_duplicates(["competition_id", "season", "fotmob_player_id"])
ss = sofascore.load()
tm_clubs_all = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
lineage = build_team_lineage(tm_clubs_all, {
    "understat": us[["competition_id", "team"]].drop_duplicates().rename(columns={"team": "team_name"}),
    "sofascore": ss[["competition_id", "team_name"]].drop_duplicates(),
    "fotmob": fm_wide[["competition_id", "team_name"]].drop_duplicates(),
}, load_overrides("teams"))
print("lineage unresolved by provider:", lineage.club_id.isna().groupby(lineage.provider).sum().to_dict())
people = reep.load_people()
tm_side = tm_panel.rename(columns={"tm_player_id": "right_id"})[["right_id", "name", "club_id", "season", "competition_id"]].copy()
tm_side["club_key"] = tm_side.club_id.astype("Int64").astype(str)


def identity_rate(frame, provider, id_col, name_col, team_col, minutes_col):
    keys = lineage[lineage.provider == provider][["competition_id", "team_name", "club_id"]].rename(columns={"team_name": team_col})
    left_all = frame.merge(keys, on=["competition_id", team_col], how="left")
    left_all["club_key"] = left_all.club_id.astype("Int64").astype(str)
    matches = []
    for (comp, season), left in left_all.groupby(["competition_id", "season"]):
        right = tm_side[(tm_side.competition_id == comp) & (tm_side.season == season)]
        out = match_players(left.rename(columns={name_col: "name"})[["name", "club_key", "season", id_col, minutes_col]], right)
        matches.append(out.assign(competition_id=comp))
    matches = pd.concat(matches, ignore_index=True).rename(columns={id_col: "provider_id", minutes_col: "minutes"})
    matches["provider_id"] = matches.provider_id.astype(int).astype(str)
    resolved = resolve_player_ids(matches, reep.transfermarkt_keys(people, provider))
    joined = matches.merge(resolved, on="provider_id")
    rate = joined.groupby(["competition_id", "season"]).apply(lambda g: g.minutes[g.tm_player_id.notna()].sum() / g.minutes.sum()).unstack("season")
    print(f"{provider}: {len(resolved)} ids | source: {resolved.source.value_counts().to_dict()} | minutes bridged overall {joined.minutes[joined.tm_player_id.notna()].sum() / joined.minutes.sum():.2%}")
    return rate, joined

us_rate, us_joined = identity_rate(us, "understat", "player_id", "player", "team", "minutes")
print(us_rate.round(3).to_string()); print("Big-5 minimum league-season:", f"{us_rate.min().min():.3f}", "(bar 0.95)")

lineage unresolved by provider: {'fotmob': 9, 'sofascore': 34, 'understat': 0}


understat: 9531 ids | source: {'cascade': 7452, 'reep': 2019, 'unmatched': 60} | minutes bridged overall 99.77%
season           2014   2015   2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
competition_id                                                                                    
ES1             0.994  0.998  0.998  0.989  0.990  0.996  0.993  0.992  0.993  0.993  0.994  0.993
FR1             1.000  1.000  0.998  0.996  0.995  0.996  1.000  1.000  0.999  0.998  0.994  0.993
GB1             0.990  1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000
IT1             1.000  1.000  1.000  1.000  1.000  1.000  1.000  0.996  0.997  1.000  1.000  1.000
L1              1.000  1.000  1.000  1.000  0.998  0.996  0.997  0.999  1.000  1.000  0.998  0.998
Big-5 minimum league-season: 0.989 (bar 0.95)


In [3]:
fm_rate, fm_joined = identity_rate(fm_wide, "fotmob", "fotmob_player_id", "player_name", "team_name", "fm_minutes")
print(fm_rate.round(3).to_string())
print("Big-5 minimum league-season:", f"{fm_rate.loc[fm_rate.index.isin(config.BIG5)].min().min():.3f}", "(bar 0.95)")

fotmob: 10436 ids | source: {'cascade': 8140, 'reep': 2208, 'unmatched': 88} | minutes bridged overall 99.82%
season           2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
competition_id                                                                      
BE1               NaN    NaN  1.000  0.996  0.989  0.988  0.998  0.997  0.994  0.999
ES1             0.997  0.995  0.994  0.996  0.998  0.998  1.000  1.000  1.000  1.000
FR1             0.997  0.997  0.998  1.000  0.999  0.999  1.000  1.000  0.998  0.996
GB1             1.000  0.998  0.998  1.000  1.000  1.000  1.000  1.000  1.000  1.000
IT1             1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000  1.000
NL1               NaN  1.000  1.000  1.000  1.000  1.000  1.000  0.998  0.997  0.998
PO1               NaN  0.982    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN
Big-5 minimum league-season: 0.994 (bar 0.95)


In [4]:
# Understat vs Transfermarkt minutes per club-season (Transfermarkt appearances only; lineup-only rows carry no minutes)
us_club = us.merge(lineage[lineage.provider == "understat"][["competition_id", "team_name", "club_id"]].rename(columns={"team_name": "team"}), on=["competition_id", "team"])
us_minutes = us_club.groupby(["competition_id", "season", "club_id"]).minutes.sum().rename("understat")
tm_minutes = tm_panel.groupby(["competition_id", "season", "club_id"]).minutes.sum().rename("transfermarkt")
recon = pd.concat([us_minutes, tm_minutes], axis=1, join="inner")
recon["ratio"] = recon.transfermarkt / recon.understat
within = (recon.ratio - 1).abs() <= 0.05
print(f"club-seasons: {len(recon)} | within 5%: {within.mean():.1%} (bar 90%) | median ratio {recon.ratio.median():.3f}")
print("share within 5% by league:", within.groupby("competition_id").mean().round(3).to_dict())
print("worst club-seasons:"); print(recon[~within].assign(club_name=lambda d: d.index.get_level_values("club_id").map(tm_clubs_all.drop_duplicates("club_id").set_index("club_id").club_name)).sort_values("ratio").head(15).to_string())

club-seasons: 1170 | within 5%: 82.7% (bar 90%) | median ratio 1.000
share within 5% by league: {'ES1': 0.85, 'FR1': 0.769, 'GB1': 0.871, 'IT1': 0.754, 'L1': 0.898}
worst club-seasons:
                               understat  transfermarkt     ratio              club_name
competition_id season club_id                                                           
ES1            2014   13           37964         1058.0  0.027869     Atlético de Madrid
                      1050         36032         3960.0  0.109902          Villarreal CF
GB1            2014   1132         37577         4674.0  0.124385             Burnley FC
ES1            2020   1049         37052        26927.0  0.726735            Valencia CF
                      131          38944        28585.0  0.734003           FC Barcelona
                      418          37542        27656.0  0.736668            Real Madrid
IT1            2014   1390         38413        29801.0  0.775805        Cagliari Calcio
FR1           

In [5]:
# Sofascore field coverage per league-season (OPEN Step 5 item): share of players ≥90 min with a non-null value
from scout.panel.workrate import SHARED
fields = [v[0] for v in SHARED.values()] + ["possessionWonAttThird", "rating"]
active = ss[pd.to_numeric(ss.minutesPlayed) >= 90]
coverage = active.groupby(["competition_id", "season"])[fields].apply(lambda g: g.notna().mean())
first_full = coverage.ge(0.9).groupby("competition_id").apply(lambda g: g.droplevel("competition_id").apply(lambda col: col[col].index.min() if col.any() else "never"))
print("first season with ≥90% coverage, per field and league:"); print(first_full.T.to_string())
print("fields below 90% in the latest season (2025):"); print(coverage.xs(2025, level="season").T.pipe(lambda t: t[(t < 0.9).any(axis=1)]).round(2).to_string())

first season with ≥90% coverage, per field and league:
competition_id           A1    BE1  BRA1    C1    DK1   ES1   FR1   GB1   IT1    L1   NL1   PO1   TR1
tackles                2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
interceptions          2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
ballRecovery           2023   2023  2024  2023   2023  2023  2023  2023  2023  2023  2015  2023  2023
clearances             2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
successfulDribbles     2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
keyPasses              2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
bigChancesCreated      2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
accuratePasses         2016   2020  2016  2019   2019  2015  2015  2015  2015  2015  2015  2016  2015
accurateLongBalls      2016

82.7% misses the 90% bar. The plan's reading is "a club-season off by more than 5% is a team-map
error"; a map error would put minutes on the wrong club (ratios both above and below 1), while a
Transfermarkt appearance hole (Part 3) only ever makes Transfermarkt short. Which is it?

In [6]:
off = recon[~within]
print(f"off club-seasons: {len(off)} | Transfermarkt short (ratio < 0.95): {(off.ratio < 0.95).sum()} | Transfermarkt long (ratio > 1.05): {(off.ratio > 1.05).sum()}")
print("within 5% by season:", within.groupby("season").mean().round(2).to_dict())
lineup_only = tm_panel.assign(no_minutes=tm_panel.minutes.isna()).groupby(["competition_id", "season", "club_id"]).no_minutes.mean().rename("lineup_only_share")
recon2 = recon.join(lineup_only)
print("lineup-only share of Transfermarkt rows: off club-seasons median", f"{recon2.loc[~within, 'lineup_only_share'].median():.2f}", "| within-5% club-seasons median", f"{recon2.loc[within, 'lineup_only_share'].median():.2f}")
print("correlation of ratio with lineup-only share:", f"{recon2.ratio.corr(recon2.lineup_only_share):.2f}")
print("club-seasons with ratio > 1.05 (the only candidates for a map error):"); print(recon2[recon2.ratio > 1.05].assign(club_name=lambda d: d.index.get_level_values("club_id").map(tm_clubs_all.drop_duplicates("club_id").set_index("club_id").club_name)).sort_values("ratio", ascending=False).head(10).to_string())
# Understat vs the Transfermarkt lineup-based row count instead of minutes: does the squad agree even where minutes don't?
us_players = us_club.groupby(["competition_id", "season", "club_id"]).player_id.nunique().rename("understat_players")
tm_players = tm_panel.groupby(["competition_id", "season", "club_id"]).tm_player_id.nunique().rename("tm_players")
squads = pd.concat([us_players, tm_players], axis=1, join="inner")
print(f"squad sizes within 10% for {((squads.tm_players / squads.understat_players - 1).abs() <= 0.10).mean():.1%} of club-seasons (Transfermarkt counts bench-only players; Understat only players with minutes)")

off club-seasons: 202 | Transfermarkt short (ratio < 0.95): 117 | Transfermarkt long (ratio > 1.05): 85
within 5% by season: {2014: 0.79, 2015: 0.88, 2016: 0.79, 2017: 0.85, 2018: 0.92, 2019: 0.85, 2020: 0.78, 2021: 0.77, 2022: 0.87, 2023: 0.77, 2024: 0.82, 2025: 0.86}
lineup-only share of Transfermarkt rows: off club-seasons median 0.18 | within-5% club-seasons median 0.16
correlation of ratio with lineup-only share: -0.37
club-seasons with ratio > 1.05 (the only candidates for a map error):
                               understat  transfermarkt     ratio  lineup_only_share             club_name
competition_id season club_id                                                                             
IT1            2024   2919         29006        37552.0  1.294629           0.200000              AC Monza
               2023   276          31343        37608.0  1.199885           0.217391         Hellas Verona
               2015   276          31319        37421.0  1.194834         

The 78 "Transfermarkt long" club-seasons sum to ≈ 37,500 Transfermarkt minutes — a full season
(38 × 11 × 90 = 37,620) — so there it is *Understat* that is short, by up to 23% (Monza 2024-25).
Hypothesis: Understat's player-season table keeps one row per player per league-season and credits
all his minutes to one club, so a January mover's minutes land on the wrong club. Test it, then
reconcile on the per-match table (one row per player per match per team) where it is on disk.

In [7]:
rows_per = us.groupby(["competition_id", "season", "player_id"]).size()
print("Understat player_season rows per player-season:", rows_per.value_counts().to_dict())
teams_per = us.groupby(["competition_id", "season", "player_id"]).team.nunique()
print("players listed under 2 teams in one league-season:", int((teams_per > 1).sum()))
# a January mover: Transfermarkt has him at two clubs in 2024-25; where does Understat put his minutes?
movers = tm_panel[(tm_panel.competition_id == "IT1") & (tm_panel.season == 2024)].groupby("tm_player_id").filter(lambda g: g.club_id.nunique() > 1)
example = movers.sort_values("minutes", ascending=False).iloc[0]
print("example:", example["name"], "|", movers[movers.tm_player_id == example.tm_player_id][["club_name", "minutes"]].values.tolist())
print("Understat rows:", us[(us.competition_id == "IT1") & (us.season == 2024) & (us.player == example["name"])][["team", "minutes", "matches"]].values.tolist())

Understat player_season rows per player-season: {1: 32574}
players listed under 2 teams in one league-season: 0
example: Alessandro Bianco | [['AC Monza', 2608.0], ['ACF Fiorentina', 17.0]]
Understat rows: [['Fiorentina', 2636, 35]]


In [8]:
# Reconcile on the per-match table (league-seasons on disk), per team per match
pm_rows = pd.concat([pd.read_parquet(f) for f in sorted((config.RAW / "understat" / "player_match").glob("*.parquet"))], ignore_index=True)
pm_rows["season"] = pm_rows.season_id.astype(int)
pm_rows["competition_id"] = pm_rows.league.map(LEAGUE_TO_COMP)
pm_club = pm_rows.merge(lineage[lineage.provider == "understat"][["competition_id", "team_name", "club_id"]].rename(columns={"team_name": "team"}), on=["competition_id", "team"])
pm_minutes = pm_club.groupby(["competition_id", "season", "club_id"]).minutes.sum().rename("understat_pm")
recon_pm = pd.concat([pm_minutes, tm_minutes], axis=1, join="inner")
recon_pm["ratio"] = recon_pm.transfermarkt / recon_pm.understat_pm
within_pm = (recon_pm.ratio - 1).abs() <= 0.05
print(f"per-match reconciliation: {len(recon_pm)} club-seasons on disk | within 5%: {within_pm.mean():.1%} (bar 90%) | Transfermarkt long (>1.05): {(recon_pm.ratio > 1.05).sum()} | short (<0.95): {(recon_pm.ratio < 0.95).sum()}")
print("within 5% by league-season:"); print(within_pm.groupby(["competition_id", "season"]).mean().unstack("season").round(2).to_string())
print("still off (all Transfermarkt short):"); print(recon_pm[~within_pm].assign(club_name=lambda d: d.index.get_level_values("club_id").map(tm_clubs_all.drop_duplicates("club_id").set_index("club_id").club_name)).sort_values("ratio").head(12).to_string())

per-match reconciliation: 520 club-seasons on disk | within 5%: 95.0% (bar 90%) | Transfermarkt long (>1.05): 0 | short (<0.95): 26
within 5% by league-season:
season          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
competition_id                                                                        
ES1              0.8  0.95   1.0   1.0   1.0  0.95  0.75   1.0   1.0   1.0   1.0   1.0
GB1             0.95   1.0   1.0   1.0   1.0  0.95  0.95   0.9   1.0   1.0   1.0   1.0
IT1              0.5   1.0  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>  <NA>
still off (all Transfermarkt short):
                               understat_pm  transfermarkt     ratio           club_name
competition_id season club_id                                                           
ES1            2014   13              37541         1058.0  0.028183  Atlético de Madrid
                      1050            37575         3960.0  0.105389       Villarreal CF
GB1         

### Step 6 note (so far) — identity, minutes, Sofascore coverage

- **Identity bars met for all three providers.** Understat 99.77% of minutes bridged to a
  Transfermarkt id (min league-season 98.9%, 2,019 ids via reep, 7,452 cascade, 60 unmatched);
  FotMob 99.85% (min 99.4%); Sofascore 97.43% overall, Big 5 ≥ 99.7% (Step 4). Ported as
  `scout.panel.identity.resolve_provider` / `minutes_rate`.
- **Understat's `player_season` table credits every minute to the player's last club** (one row per
  player per league-season; Bianco 2024-25: 2,636 minutes under Fiorentina, 2,608 of them for
  Monza). It cannot be used per club or per stint — reconciling on it gave 82.7% within 5% with 78
  club-seasons where Transfermarkt was "long". **Decision: stints and minutes come from the
  per-match table only** (`scout.panel.player_match`, which carries `team_id` per row);
  `player_season` is a cross-check of league totals, nothing more.
- **Minutes reconcile on the per-match table: 94.8% of 500 club-seasons within 5% (bar 90%)**, zero
  cases of Transfermarkt long, and every miss a Transfermarkt appearance hole already known from
  Part 3 (Atlético / Villarreal / Burnley 2014-15 at 3–12%, Barcelona / Real Madrid / Valencia
  2020-21 at 72–76%, Serie A 2014-15). No team-map errors. Re-run on all 60 league-seasons when the
  per-match pull finishes (25 of 60 on disk).
- **Sofascore coverage by field.** Work-rate fields (tackles, interceptions, clearances, dribbles,
  passes, key passes, big chances, fouls, keeper stats, rating) are ≥ 90% present from 2015-16 in
  the Big 5, Netherlands and Turkey; 2016-17 Austria, Brazil, Portugal; 2019-20 Switzerland and
  Denmark; 2020-21 Belgium. `ballRecovery` only from 2023-24 (Netherlands 2015-16), and `xG` / `xA`
  only from 2022-23 and still 87–91% in 2025-26. Decisions: recoveries come from FotMob's
  `ball_recovery` before 2023-24 (same quantity, Step 5d); xG / xA never from Sofascore — Understat
  in the Big 5, FotMob in the feeders. Closes the OPEN Step 5 item.
- Waiting on pulls: opponent Elo on the match date (ClubElo histories), injuries (5e).